# Corsi / HAR-RV Forecast Model Research

**Purpose.** This notebook evaluates whether an intraday-realized-variance Corsi/HAR model should replace the legacy VRP denominator used in the tenor-selection process.

**Research conclusion.** The notebook should be read as a model-selection and data-audit notebook, not as the final signal-parameter sweep. The final out-of-sample test carries forward two denominators for the next notebook:

1. **Current / legacy denominator** - baseline and still very competitive.
2. **`har_total_simple`** - Corsi/HAR realized-variance challenger.

`har_implied_shrinkage` is rejected as the trading-signal denominator despite better forecast-error statistics; it remains useful as a forecast diagnostic. Signal definitions, thresholds, RSI filters, Core/Secondary rules, and other parameters should be tested in a separate notebook.

**Output discipline.** The code cells save processed panels and audit CSVs. Outputs have been stripped from this cleaned copy so the notebook is easier to review, version, and rerun.


## 1. Setup and ThetaData stock endpoint audit

Creates project paths, configures ThetaData SPY 5-minute OHLC and EOD endpoints, tests available venues, and defines shared helper functions for later cells.

In [ ]:
# Cell 1: Corsi / HAR-RV v1 setup and ThetaData SPY stock endpoint audit
#
# Purpose:
# 1. Create a clean new Corsi research folder.
# 2. Test ThetaData stock OHLC access for SPY.
# 3. Test ThetaData stock EOD access for SPY.
# 4. Confirm we can compute one sample daily 5-minute realized variance.
# 5. Save audit/sample files only inside the Corsi research folder.
#
# This cell does NOT modify production files.

import json
import math
import os
from datetime import datetime
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests


# -----------------------------
# 1. Project paths
# -----------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks forecast model corsi v1"

CORSI_PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "forecast_model_corsi_v1"
)

CORSI_AUDIT_DIR = (
    PROJECT_ROOT
    / "data"
    / "audit"
    / "forecast_model_corsi_v1"
)

CORSI_RAW_SAMPLE_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "thetadata_spy_stock_samples"
)

for p in [
    NOTEBOOK_DIR,
    CORSI_PROCESSED_DIR,
    CORSI_AUDIT_DIR,
    CORSI_RAW_SAMPLE_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("Corsi research setup")
print(f"Project root:        {PROJECT_ROOT}")
print(f"Notebook dir:        {NOTEBOOK_DIR}")
print(f"Processed dir:       {CORSI_PROCESSED_DIR}")
print(f"Audit dir:           {CORSI_AUDIT_DIR}")
print(f"Raw sample dir:      {CORSI_RAW_SAMPLE_DIR}")
print(f"Run timestamp:       {RUN_TIMESTAMP}")


# -----------------------------
# 2. ThetaData endpoints / config
# -----------------------------

THETA_BASE_URL = "http://127.0.0.1:25503"

THETA_STOCK_OHLC_ENDPOINT = f"{THETA_BASE_URL}/v3/stock/history/ohlc"
THETA_STOCK_EOD_ENDPOINT = f"{THETA_BASE_URL}/v3/stock/history/eod"

SYMBOL = "SPY"

# Start with a normal liquid date.
# We are not downloading history yet.
TEST_DATE = "20240103"

# Short one-month sample.
# ThetaData OHLC multi-day requests are limited to one month, so this tests the intended monthly batching pattern.
TEST_START_DATE = "20240102"
TEST_END_DATE = "20240131"

# Expected pre-2020 no-data / limited-history test for SPY.
# This confirms the boundary, but we do not rely on this succeeding.
PRE_2020_TEST_DATE = "20191231"

INTERVAL = "5m"

# For SPY research, prefer consolidated UTP+CTA rather than Nasdaq Basic only.
# We test both to see what the subscription returns.
VENUES_TO_TEST = ["utp_cta", "nqb"]

REGULAR_SESSION_START = "09:30:00.000"
REGULAR_SESSION_END = "16:00:00.000"

print()
print("ThetaData config")
print(f"Base URL:             {THETA_BASE_URL}")
print(f"OHLC endpoint:        {THETA_STOCK_OHLC_ENDPOINT}")
print(f"EOD endpoint:         {THETA_STOCK_EOD_ENDPOINT}")
print(f"Symbol:               {SYMBOL}")
print(f"Interval:             {INTERVAL}")
print(f"Venues to test:       {VENUES_TO_TEST}")
print(f"Regular session:      {REGULAR_SESSION_START} to {REGULAR_SESSION_END}")


# -----------------------------
# 3. Helpers
# -----------------------------

def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")


def choose_col(df, candidates, required=True, label=None):
    existing = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        key = str(c).lower()
        if key in existing:
            return existing[key]

    if required:
        raise ValueError(
            f"Could not find required column {label or candidates}. "
            f"Available columns: {list(df.columns)}"
        )

    return None


def theta_csv_request(endpoint, params, timeout=60):
    """
    Make a ThetaData request and parse CSV response into a DataFrame.

    Returns:
        df, audit_record
    """
    record = {
        "endpoint": endpoint,
        "params_json": json.dumps(params, sort_keys=True),
        "status_code": None,
        "ok": False,
        "rows": 0,
        "columns": "",
        "url": None,
        "error": "",
        "response_preview": "",
    }

    try:
        response = requests.get(endpoint, params=params, timeout=timeout)

        record["status_code"] = response.status_code
        record["url"] = response.url
        record["response_preview"] = response.text[:500].replace("\n", " | ")

        if response.status_code != 200:
            record["error"] = f"HTTP {response.status_code}"
            return pd.DataFrame(), record

        text = response.text.strip()

        if not text:
            record["error"] = "Empty response body"
            return pd.DataFrame(), record

        try:
            df = pd.read_csv(StringIO(text))
        except Exception as parse_error:
            record["error"] = f"CSV parse error: {repr(parse_error)}"
            return pd.DataFrame(), record

        df.columns = [str(c).strip() for c in df.columns]

        record["rows"] = len(df)
        record["columns"] = ", ".join(df.columns)
        record["ok"] = len(df) > 0

        return df, record

    except requests.exceptions.ConnectionError as e:
        record["error"] = (
            "ConnectionError. ThetaData Terminal may not be running, "
            "or the local v3 API is unavailable."
        )
        record["response_preview"] = repr(e)[:500]
        return pd.DataFrame(), record

    except requests.exceptions.Timeout as e:
        record["error"] = "Timeout"
        record["response_preview"] = repr(e)[:500]
        return pd.DataFrame(), record

    except Exception as e:
        record["error"] = repr(e)
        return pd.DataFrame(), record


def normalize_ohlc_df(df):
    """
    Normalize ThetaData stock OHLC response into a standard format.
    """
    if df.empty:
        return df.copy()

    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]

    timestamp_col = choose_col(out, ["timestamp", "datetime", "time"], label="timestamp")
    open_col = choose_col(out, ["open"], label="open")
    high_col = choose_col(out, ["high"], label="high")
    low_col = choose_col(out, ["low"], label="low")
    close_col = choose_col(out, ["close"], label="close")

    volume_col = choose_col(out, ["volume"], required=False, label="volume")
    count_col = choose_col(out, ["count"], required=False, label="count")
    vwap_col = choose_col(out, ["vwap"], required=False, label="vwap")

    out["timestamp"] = pd.to_datetime(out[timestamp_col], errors="coerce")
    out["open"] = safe_numeric(out[open_col])
    out["high"] = safe_numeric(out[high_col])
    out["low"] = safe_numeric(out[low_col])
    out["close"] = safe_numeric(out[close_col])

    if volume_col is not None:
        out["volume"] = safe_numeric(out[volume_col])
    else:
        out["volume"] = np.nan

    if count_col is not None:
        out["count"] = safe_numeric(out[count_col])
    else:
        out["count"] = np.nan

    if vwap_col is not None:
        out["vwap"] = safe_numeric(out[vwap_col])
    else:
        out["vwap"] = np.nan

    out = out.dropna(subset=["timestamp", "open", "high", "low", "close"]).copy()
    out = out.sort_values("timestamp").reset_index(drop=True)

    out["date"] = out["timestamp"].dt.normalize()
    out["trade_date"] = out["timestamp"].dt.strftime("%Y%m%d").astype(int)

    return out[
        [
            "trade_date",
            "date",
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "count",
            "vwap",
        ]
    ].copy()


def normalize_eod_df(df):
    """
    Normalize ThetaData stock EOD response into a standard format.
    """
    if df.empty:
        return df.copy()

    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]

    created_col = choose_col(out, ["created"], required=False, label="created")
    last_trade_col = choose_col(out, ["last_trade"], required=False, label="last_trade")

    open_col = choose_col(out, ["open"], label="open")
    high_col = choose_col(out, ["high"], label="high")
    low_col = choose_col(out, ["low"], label="low")
    close_col = choose_col(out, ["close"], label="close")

    volume_col = choose_col(out, ["volume"], required=False, label="volume")
    count_col = choose_col(out, ["count"], required=False, label="count")

    if last_trade_col is not None:
        out["timestamp_for_date"] = pd.to_datetime(out[last_trade_col], errors="coerce")
    elif created_col is not None:
        out["timestamp_for_date"] = pd.to_datetime(out[created_col], errors="coerce")
    else:
        raise ValueError("EOD response has neither last_trade nor created timestamp.")

    if created_col is not None:
        out["created"] = pd.to_datetime(out[created_col], errors="coerce")
    else:
        out["created"] = pd.NaT

    if last_trade_col is not None:
        out["last_trade"] = pd.to_datetime(out[last_trade_col], errors="coerce")
    else:
        out["last_trade"] = pd.NaT

    out["date"] = out["timestamp_for_date"].dt.normalize()
    out["trade_date"] = out["timestamp_for_date"].dt.strftime("%Y%m%d").astype(int)

    out["open"] = safe_numeric(out[open_col])
    out["high"] = safe_numeric(out[high_col])
    out["low"] = safe_numeric(out[low_col])
    out["close"] = safe_numeric(out[close_col])

    if volume_col is not None:
        out["volume"] = safe_numeric(out[volume_col])
    else:
        out["volume"] = np.nan

    if count_col is not None:
        out["count"] = safe_numeric(out[count_col])
    else:
        out["count"] = np.nan

    out = out.dropna(subset=["date", "open", "high", "low", "close"]).copy()
    out = out.sort_values("date").reset_index(drop=True)

    return out[
        [
            "trade_date",
            "date",
            "created",
            "last_trade",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "count",
        ]
    ].copy()


def compute_daily_intraday_rv(ohlc):
    """
    Compute daily realized variance from intraday OHLC bars.

    For each day:
        first bar return = log(first_bar_close / first_bar_open)
        subsequent returns = log(close_t / close_{t-1})
        intraday_rv = sum of squared intraday bar returns

    This is a research convention for v1.
    """
    if ohlc.empty:
        return pd.DataFrame()

    rows = []

    for trade_date, g in ohlc.groupby("trade_date"):
        g = g.sort_values("timestamp").copy()

        first_open = g["open"].iloc[0]
        first_close = g["close"].iloc[0]
        last_close = g["close"].iloc[-1]

        close_to_close_bar_returns = np.log(g["close"] / g["close"].shift(1))
        first_bar_return = np.log(first_close / first_open)

        all_returns = pd.concat([
            pd.Series([first_bar_return], index=[g.index[0]]),
            close_to_close_bar_returns.iloc[1:],
        ])

        intraday_rv = float(np.nansum(all_returns ** 2))
        intraday_vol_annualized = float(np.sqrt(intraday_rv * 252) * 100)

        intraday_open_to_close_return = float(np.log(last_close / first_open))

        rows.append({
            "trade_date": int(trade_date),
            "date": g["date"].iloc[0],
            "bar_count": int(len(g)),
            "first_timestamp": g["timestamp"].iloc[0],
            "last_timestamp": g["timestamp"].iloc[-1],
            "first_open": float(first_open),
            "first_close": float(first_close),
            "last_close": float(last_close),
            "intraday_open_to_close_return": intraday_open_to_close_return,
            "intraday_realized_variance_raw": intraday_rv,
            "intraday_realized_variance_annualized": intraday_rv * 252,
            "intraday_realized_vol_annualized": intraday_vol_annualized,
            "total_volume": float(g["volume"].sum()) if g["volume"].notna().any() else np.nan,
            "total_trade_count": float(g["count"].sum()) if g["count"].notna().any() else np.nan,
        })

    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


# -----------------------------
# 4. Test one-day OHLC access by venue
# -----------------------------

audit_records = []
ohlc_test_results = {}

for venue in VENUES_TO_TEST:
    params = {
        "symbol": SYMBOL,
        "date": TEST_DATE,
        "interval": INTERVAL,
        "start_time": REGULAR_SESSION_START,
        "end_time": REGULAR_SESSION_END,
        "venue": venue,
        "format": "csv",
    }

    df_raw, rec = theta_csv_request(
        THETA_STOCK_OHLC_ENDPOINT,
        params=params,
        timeout=60,
    )

    rec["request_name"] = f"one_day_ohlc_{venue}"
    rec["symbol"] = SYMBOL
    rec["test_date"] = TEST_DATE
    rec["venue"] = venue
    audit_records.append(rec)

    try:
        df_norm = normalize_ohlc_df(df_raw)
    except Exception as e:
        df_norm = pd.DataFrame()
        audit_records[-1]["error"] = f"{audit_records[-1]['error']} | normalize error: {repr(e)}"

    ohlc_test_results[venue] = {
        "raw": df_raw,
        "normalized": df_norm,
    }

print()
print("One-day OHLC endpoint test summary:")
one_day_ohlc_audit_df = pd.DataFrame(audit_records)
display(one_day_ohlc_audit_df[
    [
        "request_name",
        "status_code",
        "ok",
        "rows",
        "columns",
        "error",
        "response_preview",
    ]
])


# -----------------------------
# 5. Select venue for research sample
# -----------------------------

selected_venue = None

for venue in VENUES_TO_TEST:
    if not ohlc_test_results[venue]["normalized"].empty:
        selected_venue = venue
        break

print()
print(f"Selected venue for this notebook run: {selected_venue}")

if selected_venue is None:
    print()
    print("No usable SPY OHLC data returned.")
    print("Most likely causes:")
    print("1. Theta Terminal is not running.")
    print("2. The stock subscription has not activated yet.")
    print("3. The endpoint is blocked by permission or entitlement.")
    print("4. The requested venue/date has no data.")
else:
    print()
    print(f"Normalized one-day OHLC sample for {SYMBOL} on {TEST_DATE}, venue={selected_venue}:")
    display(ohlc_test_results[selected_venue]["normalized"].head())
    display(ohlc_test_results[selected_venue]["normalized"].tail())


# -----------------------------
# 6. Test one-month OHLC pull
# -----------------------------

if selected_venue is not None:
    params = {
        "symbol": SYMBOL,
        "start_date": TEST_START_DATE,
        "end_date": TEST_END_DATE,
        "interval": INTERVAL,
        "start_time": REGULAR_SESSION_START,
        "end_time": REGULAR_SESSION_END,
        "venue": selected_venue,
        "format": "csv",
    }

    month_raw, month_rec = theta_csv_request(
        THETA_STOCK_OHLC_ENDPOINT,
        params=params,
        timeout=120,
    )

    month_rec["request_name"] = "one_month_ohlc"
    month_rec["symbol"] = SYMBOL
    month_rec["test_start_date"] = TEST_START_DATE
    month_rec["test_end_date"] = TEST_END_DATE
    month_rec["venue"] = selected_venue
    audit_records.append(month_rec)

    try:
        month_ohlc = normalize_ohlc_df(month_raw)
    except Exception as e:
        month_ohlc = pd.DataFrame()
        audit_records[-1]["error"] = f"{audit_records[-1]['error']} | normalize error: {repr(e)}"

else:
    month_raw = pd.DataFrame()
    month_ohlc = pd.DataFrame()

print()
print("One-month OHLC sample summary:")
if month_ohlc.empty:
    print("No normalized one-month OHLC data available.")
else:
    month_summary = pd.DataFrame([{
        "rows": len(month_ohlc),
        "first_timestamp": month_ohlc["timestamp"].min(),
        "last_timestamp": month_ohlc["timestamp"].max(),
        "trading_days": month_ohlc["trade_date"].nunique(),
        "min_bars_per_day": month_ohlc.groupby("trade_date").size().min(),
        "max_bars_per_day": month_ohlc.groupby("trade_date").size().max(),
        "selected_venue": selected_venue,
    }])
    display(month_summary)
    display(month_ohlc.head())
    display(month_ohlc.tail())


# -----------------------------
# 7. Test EOD pull
# -----------------------------

params = {
    "symbol": SYMBOL,
    "start_date": TEST_START_DATE,
    "end_date": TEST_END_DATE,
    "format": "csv",
}

eod_raw, eod_rec = theta_csv_request(
    THETA_STOCK_EOD_ENDPOINT,
    params=params,
    timeout=60,
)

eod_rec["request_name"] = "one_month_eod"
eod_rec["symbol"] = SYMBOL
eod_rec["test_start_date"] = TEST_START_DATE
eod_rec["test_end_date"] = TEST_END_DATE
audit_records.append(eod_rec)

try:
    eod_df = normalize_eod_df(eod_raw)
except Exception as e:
    eod_df = pd.DataFrame()
    audit_records[-1]["error"] = f"{audit_records[-1]['error']} | normalize error: {repr(e)}"

print()
print("EOD endpoint sample summary:")
if eod_df.empty:
    print("No normalized EOD data available.")
else:
    eod_summary = pd.DataFrame([{
        "rows": len(eod_df),
        "first_date": eod_df["date"].min(),
        "last_date": eod_df["date"].max(),
        "trading_days": eod_df["trade_date"].nunique(),
        "missing_open": int(eod_df["open"].isna().sum()),
        "missing_close": int(eod_df["close"].isna().sum()),
    }])
    display(eod_summary)
    display(eod_df.head())
    display(eod_df.tail())


# -----------------------------
# 8. Compute sample daily intraday realized variance
# -----------------------------

daily_intraday_rv_sample = compute_daily_intraday_rv(month_ohlc)

print()
print("Sample daily intraday realized variance from 5-minute SPY bars:")
if daily_intraday_rv_sample.empty:
    print("No daily RV sample available.")
else:
    display(daily_intraday_rv_sample.head())
    display(daily_intraday_rv_sample.tail())


# -----------------------------
# 9. Compare OHLC-derived open/close to EOD open/close
# -----------------------------

if not daily_intraday_rv_sample.empty and not eod_df.empty:
    eod_compare = eod_df[
        [
            "trade_date",
            "open",
            "close",
            "volume",
            "count",
        ]
    ].rename(columns={
        "open": "eod_open",
        "close": "eod_close",
        "volume": "eod_volume",
        "count": "eod_count",
    })

    ohlc_eod_compare = daily_intraday_rv_sample.merge(
        eod_compare,
        on="trade_date",
        how="left",
        validate="one_to_one",
    )

    ohlc_eod_compare["open_diff"] = (
        ohlc_eod_compare["first_open"]
        - ohlc_eod_compare["eod_open"]
    )

    ohlc_eod_compare["close_diff"] = (
        ohlc_eod_compare["last_close"]
        - ohlc_eod_compare["eod_close"]
    )

    ohlc_eod_compare["abs_open_diff"] = ohlc_eod_compare["open_diff"].abs()
    ohlc_eod_compare["abs_close_diff"] = ohlc_eod_compare["close_diff"].abs()

    print()
    print("OHLC sample vs EOD open/close comparison:")
    display(ohlc_eod_compare[
        [
            "trade_date",
            "date",
            "bar_count",
            "first_open",
            "eod_open",
            "open_diff",
            "last_close",
            "eod_close",
            "close_diff",
            "intraday_realized_vol_annualized",
        ]
    ].head(20))

    compare_summary = pd.DataFrame([{
        "rows": len(ohlc_eod_compare),
        "missing_eod_rows": int(ohlc_eod_compare["eod_open"].isna().sum()),
        "max_abs_open_diff": ohlc_eod_compare["abs_open_diff"].max(),
        "max_abs_close_diff": ohlc_eod_compare["abs_close_diff"].max(),
        "mean_abs_open_diff": ohlc_eod_compare["abs_open_diff"].mean(),
        "mean_abs_close_diff": ohlc_eod_compare["abs_close_diff"].mean(),
    }])

    print()
    print("OHLC vs EOD comparison summary:")
    display(compare_summary)

else:
    ohlc_eod_compare = pd.DataFrame()
    print()
    print("Skipped OHLC vs EOD comparison because OHLC or EOD sample is missing.")


# -----------------------------
# 10. Optional pre-2020 SPY availability test
# -----------------------------

params = {
    "symbol": SYMBOL,
    "date": PRE_2020_TEST_DATE,
    "interval": INTERVAL,
    "start_time": REGULAR_SESSION_START,
    "end_time": REGULAR_SESSION_END,
    "venue": "utp_cta",
    "format": "csv",
}

pre2020_raw, pre2020_rec = theta_csv_request(
    THETA_STOCK_OHLC_ENDPOINT,
    params=params,
    timeout=60,
)

pre2020_rec["request_name"] = "pre_2020_spy_ohlc_availability_check"
pre2020_rec["symbol"] = SYMBOL
pre2020_rec["test_date"] = PRE_2020_TEST_DATE
pre2020_rec["venue"] = "utp_cta"
audit_records.append(pre2020_rec)

print()
print("Pre-2020 SPY OHLC availability check:")
display(pd.DataFrame([pre2020_rec])[
    [
        "request_name",
        "status_code",
        "ok",
        "rows",
        "columns",
        "error",
        "response_preview",
    ]
])


# -----------------------------
# 11. Final audit summary and save outputs
# -----------------------------

request_audit_df = pd.DataFrame(audit_records)

request_audit_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell1_thetadata_request_audit_{RUN_TIMESTAMP}.csv"
)

one_day_sample_output = (
    CORSI_RAW_SAMPLE_DIR
    / f"spy_5m_ohlc_one_day_sample_{TEST_DATE}_{selected_venue or 'none'}_{RUN_TIMESTAMP}.csv"
)

month_sample_output = (
    CORSI_RAW_SAMPLE_DIR
    / f"spy_5m_ohlc_month_sample_{TEST_START_DATE}_{TEST_END_DATE}_{selected_venue or 'none'}_{RUN_TIMESTAMP}.csv"
)

eod_sample_output = (
    CORSI_RAW_SAMPLE_DIR
    / f"spy_eod_sample_{TEST_START_DATE}_{TEST_END_DATE}_{RUN_TIMESTAMP}.csv"
)

rv_sample_output = (
    CORSI_AUDIT_DIR
    / f"spy_daily_intraday_rv_sample_{TEST_START_DATE}_{TEST_END_DATE}_{selected_venue or 'none'}_{RUN_TIMESTAMP}.csv"
)

ohlc_eod_compare_output = (
    CORSI_AUDIT_DIR
    / f"spy_ohlc_vs_eod_compare_{TEST_START_DATE}_{TEST_END_DATE}_{selected_venue or 'none'}_{RUN_TIMESTAMP}.csv"
)

request_audit_df.to_csv(request_audit_output, index=False)

if selected_venue is not None and not ohlc_test_results[selected_venue]["normalized"].empty:
    ohlc_test_results[selected_venue]["normalized"].to_csv(one_day_sample_output, index=False)

if not month_ohlc.empty:
    month_ohlc.to_csv(month_sample_output, index=False)

if not eod_df.empty:
    eod_df.to_csv(eod_sample_output, index=False)

if not daily_intraday_rv_sample.empty:
    daily_intraday_rv_sample.to_csv(rv_sample_output, index=False)

if not ohlc_eod_compare.empty:
    ohlc_eod_compare.to_csv(ohlc_eod_compare_output, index=False)

print()
print("Final ThetaData request audit:")
display(request_audit_df[
    [
        "request_name",
        "status_code",
        "ok",
        "rows",
        "columns",
        "error",
        "response_preview",
    ]
])

print()
print("Cell 1 outputs saved:")
print(f"Request audit:           {request_audit_output}")

if selected_venue is not None and not ohlc_test_results[selected_venue]["normalized"].empty:
    print(f"One-day OHLC sample:     {one_day_sample_output}")

if not month_ohlc.empty:
    print(f"One-month OHLC sample:   {month_sample_output}")

if not eod_df.empty:
    print(f"EOD sample:              {eod_sample_output}")

if not daily_intraday_rv_sample.empty:
    print(f"Daily RV sample:         {rv_sample_output}")

if not ohlc_eod_compare.empty:
    print(f"OHLC vs EOD comparison:  {ohlc_eod_compare_output}")

print()
print("Cell 1 complete.")
print("Next step will be Cell 2: download SPY 5-minute OHLC history month-by-month and build the full daily intraday RV table.")

## 2. Clean SPY 5-minute sample and construct daily realized variance

Removes weekends/holidays, non-positive rows, and the problematic 16:00 bar; computes intraday realized variance using 5-minute bars and joins overnight variance from EOD open/close data.

In [ ]:
# Cell 2: Clean SPY 5-minute sample and fix realized-variance construction
#
# Purpose:
# Cell 1 proved the ThetaData stock endpoint works, but it also revealed two problems:
#
# 1. Multi-day OHLC requests return rows for weekends/holidays with zero prices.
# 2. The 16:00 bar can be a tiny/duplicate/bad bar and sometimes has zero close.
#
# Therefore, before downloading full history, we need a robust cleaner:
# - Use 09:30 through 15:55 bars only.
# - Drop any bar with non-positive OHLC.
# - Keep only dates that exist in the EOD file.
# - Require normal full-day bar count of 78 for now.
# - Compute intraday RV from cleaned bars only.
#
# This cell uses the Cell 1 sample data only.
# It does NOT download full history yet.
# It does NOT modify production files.

# -----------------------------
# 1. Cleaning configuration
# -----------------------------

CLEAN_SESSION_START = "09:30:00"
CLEAN_SESSION_END = "15:55:00"

EXPECTED_FULL_DAY_5M_BARS = 78

OPEN_CLOSE_DIFF_WARN_THRESHOLD = 0.15
OPEN_CLOSE_DIFF_FAIL_THRESHOLD = 0.50

print("SPY 5-minute sample cleaning configuration:")
print(f"Clean session start:       {CLEAN_SESSION_START}")
print(f"Clean session end:         {CLEAN_SESSION_END}")
print(f"Expected full-day bars:    {EXPECTED_FULL_DAY_5M_BARS}")
print(f"Warn open/close diff:      {OPEN_CLOSE_DIFF_WARN_THRESHOLD}")
print(f"Fail open/close diff:      {OPEN_CLOSE_DIFF_FAIL_THRESHOLD}")


# -----------------------------
# 2. Helpers
# -----------------------------

def clean_spy_5m_ohlc_sample(ohlc_df, eod_df):
    """
    Clean normalized ThetaData SPY 5-minute OHLC bars.

    Rules:
    - Keep only dates appearing in EOD file.
    - Keep only 09:30 through 15:55 bars.
    - Drop non-positive OHLC bars.
    - Preserve excluded rows with reasons for audit.
    """
    if ohlc_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    out = ohlc_df.copy()

    out["timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce")
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["trade_date"] = pd.to_numeric(out["trade_date"], errors="coerce").astype("Int64")

    for c in ["open", "high", "low", "close", "volume", "count", "vwap"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    eod_trade_dates = set(
        pd.to_numeric(eod_df["trade_date"], errors="coerce")
        .dropna()
        .astype(int)
        .tolist()
    )

    start_t = pd.to_datetime(CLEAN_SESSION_START).time()
    end_t = pd.to_datetime(CLEAN_SESSION_END).time()

    out["bar_time"] = out["timestamp"].dt.time

    out["is_eod_trading_date"] = out["trade_date"].astype(float).isin(eod_trade_dates)

    out["is_clean_time"] = (
        (out["bar_time"] >= start_t)
        & (out["bar_time"] <= end_t)
    )

    out["positive_ohlc"] = (
        (out["open"] > 0)
        & (out["high"] > 0)
        & (out["low"] > 0)
        & (out["close"] > 0)
    )

    out["valid_clean_bar"] = (
        out["is_eod_trading_date"]
        & out["is_clean_time"]
        & out["positive_ohlc"]
    )

    def exclusion_reason(row):
        reasons = []

        if not row["is_eod_trading_date"]:
            reasons.append("not_eod_trading_date")

        if not row["is_clean_time"]:
            reasons.append("outside_clean_time_window")

        if not row["positive_ohlc"]:
            reasons.append("non_positive_ohlc")

        if not reasons:
            return ""

        return "|".join(reasons)

    out["exclusion_reason"] = out.apply(exclusion_reason, axis=1)

    clean = (
        out
        .loc[out["valid_clean_bar"]]
        .sort_values(["trade_date", "timestamp"])
        .reset_index(drop=True)
        .copy()
    )

    excluded = (
        out
        .loc[~out["valid_clean_bar"]]
        .sort_values(["trade_date", "timestamp"])
        .reset_index(drop=True)
        .copy()
    )

    return clean, excluded


def compute_clean_daily_intraday_rv(clean_ohlc_df):
    """
    Compute daily intraday realized variance from cleaned 5-minute bars.

    For each day:
        first return = log(first 5m close / first 5m open)
        later returns = log(close_t / close_t-1)

    This uses only valid positive bars from 09:30 through 15:55.
    """
    if clean_ohlc_df.empty:
        return pd.DataFrame()

    rows = []

    for trade_date, g in clean_ohlc_df.groupby("trade_date"):
        g = g.sort_values("timestamp").copy()

        first_open = g["open"].iloc[0]
        first_close = g["close"].iloc[0]
        last_close = g["close"].iloc[-1]

        close_returns = np.log(g["close"] / g["close"].shift(1))
        first_bar_return = np.log(first_close / first_open)

        rv_returns = close_returns.copy()
        rv_returns.iloc[0] = first_bar_return

        intraday_rv_raw = float(np.nansum(rv_returns ** 2))
        intraday_rv_ann = intraday_rv_raw * 252
        intraday_vol_ann = np.sqrt(intraday_rv_ann) * 100

        open_to_close_return = float(np.log(last_close / first_open))

        rows.append({
            "trade_date": int(trade_date),
            "date": g["date"].iloc[0],
            "bar_count": int(len(g)),
            "first_timestamp": g["timestamp"].iloc[0],
            "last_timestamp": g["timestamp"].iloc[-1],
            "first_open": float(first_open),
            "first_close": float(first_close),
            "last_close": float(last_close),
            "intraday_open_to_close_return": open_to_close_return,
            "intraday_realized_variance_raw": intraday_rv_raw,
            "intraday_realized_variance_annualized": intraday_rv_ann,
            "intraday_realized_vol_annualized": intraday_vol_ann,
            "total_volume": float(g["volume"].sum()) if "volume" in g.columns else np.nan,
            "total_trade_count": float(g["count"].sum()) if "count" in g.columns else np.nan,
        })

    daily = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

    return daily


def add_eod_and_overnight_variance(daily_rv_df, eod_df):
    """
    Join daily intraday RV to EOD open/close and compute overnight variance.
    """
    if daily_rv_df.empty or eod_df.empty:
        return pd.DataFrame()

    eod_clean = eod_df.copy().sort_values("date").reset_index(drop=True)

    eod_clean["prior_eod_close"] = eod_clean["close"].shift(1)

    eod_clean["overnight_return"] = np.log(
        eod_clean["open"] / eod_clean["prior_eod_close"]
    )

    eod_clean["overnight_realized_variance_raw"] = eod_clean["overnight_return"] ** 2

    eod_join = eod_clean[
        [
            "trade_date",
            "date",
            "open",
            "close",
            "prior_eod_close",
            "overnight_return",
            "overnight_realized_variance_raw",
            "volume",
            "count",
        ]
    ].rename(columns={
        "open": "eod_open",
        "close": "eod_close",
        "volume": "eod_volume",
        "count": "eod_count",
    })

    out = daily_rv_df.merge(
        eod_join,
        on=["trade_date", "date"],
        how="left",
        validate="one_to_one",
    )

    out["open_diff_vs_eod"] = out["first_open"] - out["eod_open"]
    out["close_diff_vs_eod"] = out["last_close"] - out["eod_close"]

    out["abs_open_diff_vs_eod"] = out["open_diff_vs_eod"].abs()
    out["abs_close_diff_vs_eod"] = out["close_diff_vs_eod"].abs()

    out["total_realized_variance_raw"] = (
        out["overnight_realized_variance_raw"].fillna(0)
        + out["intraday_realized_variance_raw"]
    )

    out["total_realized_variance_annualized"] = (
        out["total_realized_variance_raw"] * 252
    )

    out["total_realized_vol_annualized"] = (
        np.sqrt(out["total_realized_variance_annualized"]) * 100
    )

    out["bar_count_ok"] = out["bar_count"] == EXPECTED_FULL_DAY_5M_BARS

    out["open_diff_status"] = np.select(
        [
            out["abs_open_diff_vs_eod"] <= OPEN_CLOSE_DIFF_WARN_THRESHOLD,
            out["abs_open_diff_vs_eod"] <= OPEN_CLOSE_DIFF_FAIL_THRESHOLD,
        ],
        [
            "OK",
            "WARN",
        ],
        default="FAIL",
    )

    out["close_diff_status"] = np.select(
        [
            out["abs_close_diff_vs_eod"] <= OPEN_CLOSE_DIFF_WARN_THRESHOLD,
            out["abs_close_diff_vs_eod"] <= OPEN_CLOSE_DIFF_FAIL_THRESHOLD,
        ],
        [
            "OK",
            "WARN",
        ],
        default="FAIL",
    )

    out["finite_rv_ok"] = (
        np.isfinite(out["intraday_realized_variance_raw"])
        & np.isfinite(out["total_realized_variance_raw"])
        & (out["intraday_realized_variance_raw"] >= 0)
        & (out["total_realized_variance_raw"] >= 0)
    )

    out["daily_rv_quality_status"] = np.where(
        out["bar_count_ok"]
        & (out["open_diff_status"] != "FAIL")
        & (out["close_diff_status"] != "FAIL")
        & out["finite_rv_ok"],
        "OK",
        "REVIEW",
    )

    return out.sort_values("date").reset_index(drop=True)


# -----------------------------
# 3. Clean sample bars
# -----------------------------

clean_month_ohlc, excluded_month_ohlc = clean_spy_5m_ohlc_sample(
    month_ohlc,
    eod_df,
)

print()
print("Raw vs clean OHLC sample row counts:")
raw_clean_summary = pd.DataFrame([{
    "raw_rows": len(month_ohlc),
    "clean_rows": len(clean_month_ohlc),
    "excluded_rows": len(excluded_month_ohlc),
    "raw_calendar_dates": month_ohlc["trade_date"].nunique() if not month_ohlc.empty else 0,
    "clean_trading_dates": clean_month_ohlc["trade_date"].nunique() if not clean_month_ohlc.empty else 0,
    "eod_trading_dates": eod_df["trade_date"].nunique() if not eod_df.empty else 0,
}])
display(raw_clean_summary)

print()
print("Excluded row reason counts:")
if excluded_month_ohlc.empty:
    print("No excluded rows.")
else:
    display(
        excluded_month_ohlc["exclusion_reason"]
        .value_counts()
        .reset_index()
        .rename(columns={"index": "exclusion_reason", "exclusion_reason": "rows"})
    )

print()
print("Clean OHLC sample head/tail:")
display(clean_month_ohlc.head())
display(clean_month_ohlc.tail())


# -----------------------------
# 4. Compute cleaned daily intraday RV
# -----------------------------

clean_daily_intraday_rv = compute_clean_daily_intraday_rv(clean_month_ohlc)

clean_daily_rv_with_eod = add_eod_and_overnight_variance(
    clean_daily_intraday_rv,
    eod_df,
)

print()
print("Clean daily SPY intraday + overnight RV sample:")
display(clean_daily_rv_with_eod)


# -----------------------------
# 5. QA summary
# -----------------------------

cell2_quality_summary = pd.DataFrame([{
    "sample_start_date": clean_daily_rv_with_eod["date"].min() if not clean_daily_rv_with_eod.empty else pd.NaT,
    "sample_end_date": clean_daily_rv_with_eod["date"].max() if not clean_daily_rv_with_eod.empty else pd.NaT,
    "daily_rows": len(clean_daily_rv_with_eod),
    "ok_days": int((clean_daily_rv_with_eod["daily_rv_quality_status"] == "OK").sum()) if not clean_daily_rv_with_eod.empty else 0,
    "review_days": int((clean_daily_rv_with_eod["daily_rv_quality_status"] == "REVIEW").sum()) if not clean_daily_rv_with_eod.empty else 0,
    "min_bar_count": clean_daily_rv_with_eod["bar_count"].min() if not clean_daily_rv_with_eod.empty else np.nan,
    "max_bar_count": clean_daily_rv_with_eod["bar_count"].max() if not clean_daily_rv_with_eod.empty else np.nan,
    "max_abs_open_diff_vs_eod": clean_daily_rv_with_eod["abs_open_diff_vs_eod"].max() if not clean_daily_rv_with_eod.empty else np.nan,
    "max_abs_close_diff_vs_eod": clean_daily_rv_with_eod["abs_close_diff_vs_eod"].max() if not clean_daily_rv_with_eod.empty else np.nan,
    "mean_intraday_realized_vol_annualized": clean_daily_rv_with_eod["intraday_realized_vol_annualized"].mean() if not clean_daily_rv_with_eod.empty else np.nan,
    "mean_total_realized_vol_annualized": clean_daily_rv_with_eod["total_realized_vol_annualized"].mean() if not clean_daily_rv_with_eod.empty else np.nan,
}])

print()
print("Cell 2 quality summary:")
display(cell2_quality_summary)

print()
print("Daily quality status counts:")
if clean_daily_rv_with_eod.empty:
    print("No daily RV rows.")
else:
    display(
        clean_daily_rv_with_eod["daily_rv_quality_status"]
        .value_counts()
        .reset_index()
        .rename(columns={"index": "daily_rv_quality_status", "daily_rv_quality_status": "days"})
    )


# -----------------------------
# 6. Pre-2020 availability check with positive-price rule
# -----------------------------

try:
    pre2020_norm = normalize_ohlc_df(pre2020_raw)
    pre2020_positive_rows = (
        (pre2020_norm["open"] > 0)
        & (pre2020_norm["high"] > 0)
        & (pre2020_norm["low"] > 0)
        & (pre2020_norm["close"] > 0)
    ).sum()

    pre2020_availability_summary = pd.DataFrame([{
        "test_date": PRE_2020_TEST_DATE,
        "raw_rows": len(pre2020_norm),
        "positive_ohlc_rows": int(pre2020_positive_rows),
        "has_real_positive_price_data": bool(pre2020_positive_rows > 0),
    }])
except Exception as e:
    pre2020_availability_summary = pd.DataFrame([{
        "test_date": PRE_2020_TEST_DATE,
        "raw_rows": np.nan,
        "positive_ohlc_rows": np.nan,
        "has_real_positive_price_data": False,
        "error": repr(e),
    }])

print()
print("Pre-2020 availability check using positive-price rule:")
display(pre2020_availability_summary)


# -----------------------------
# 7. Save Cell 2 outputs
# -----------------------------

cell2_clean_bars_output = (
    CORSI_RAW_SAMPLE_DIR
    / f"spy_5m_ohlc_clean_sample_{TEST_START_DATE}_{TEST_END_DATE}_{selected_venue}_{RUN_TIMESTAMP}.csv"
)

cell2_excluded_bars_output = (
    CORSI_AUDIT_DIR
    / f"spy_5m_ohlc_excluded_rows_sample_{TEST_START_DATE}_{TEST_END_DATE}_{selected_venue}_{RUN_TIMESTAMP}.csv"
)

cell2_daily_rv_output = (
    CORSI_AUDIT_DIR
    / f"spy_clean_daily_rv_sample_{TEST_START_DATE}_{TEST_END_DATE}_{selected_venue}_{RUN_TIMESTAMP}.csv"
)

cell2_quality_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell2_quality_summary_{TEST_START_DATE}_{TEST_END_DATE}_{selected_venue}_{RUN_TIMESTAMP}.csv"
)

cell2_pre2020_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell2_pre2020_availability_check_{PRE_2020_TEST_DATE}_{RUN_TIMESTAMP}.csv"
)

clean_month_ohlc.to_csv(cell2_clean_bars_output, index=False)
excluded_month_ohlc.to_csv(cell2_excluded_bars_output, index=False)
clean_daily_rv_with_eod.to_csv(cell2_daily_rv_output, index=False)
cell2_quality_summary.to_csv(cell2_quality_summary_output, index=False)
pre2020_availability_summary.to_csv(cell2_pre2020_output, index=False)

print()
print("Cell 2 outputs saved:")
print(f"Clean OHLC sample:          {cell2_clean_bars_output}")
print(f"Excluded row sample:        {cell2_excluded_bars_output}")
print(f"Clean daily RV sample:      {cell2_daily_rv_output}")
print(f"Quality summary:            {cell2_quality_summary_output}")
print(f"Pre-2020 availability:      {cell2_pre2020_output}")

print()
print("Cell 2 complete.")
print("Next step will be Cell 3: if this sample is clean, download SPY 5-minute history month-by-month.")

## 3. Build full SPY intraday realized-variance history

Downloads or loads cached SPY 5-minute monthly OHLC/EOD data from 2018-06-25 through the current production feature-panel end date. Builds the full daily realized-variance table.

In [ ]:
# Cell 3: Download SPY 5-minute history month-by-month and build full daily RV table
#
# Purpose:
# 1. Pull SPY 5-minute OHLC history from ThetaData month-by-month.
# 2. Pull SPY EOD history month-by-month.
# 3. Clean raw 5-minute bars using the Cell 2 rules.
# 4. Build a daily SPY realized-variance table:
#       intraday RV
#       overnight RV
#       total RV
# 5. Compare coverage to the existing implied-variance / feature-panel dates.
#
# This cell saves research outputs only under forecast_model_corsi_v1.
# It does NOT modify production files.

import time
from pandas.tseries.offsets import MonthEnd


# -----------------------------
# 1. Download configuration
# -----------------------------

# Use existing production feature panel if available to define the target date range.
PRODUCTION_FEATURE_PANEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "production_feature_panel_v0_1.parquet"
)

IMPLIED_TERM_STRUCTURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vix_term_structure_history_v0_7_1_repaired_total_variance.parquet"
)

# Fallback if source panels are unavailable.
FALLBACK_START_DATE = "20180625"
FALLBACK_END_DATE = datetime.now().strftime("%Y%m%d")

# For initial full pull, leave as None.
# For testing, set something like 3.
MAX_MONTHS_TO_DOWNLOAD = None

FORCE_REFRESH_THETA = False
REQUEST_SLEEP_SECONDS = 0.15

SPY_STOCK_VENUE = selected_venue if "selected_venue" in globals() and selected_venue is not None else "utp_cta"

FULL_HISTORY_INTERVAL = "5m"

# Request through 16:00 because ThetaData returns it; cleaner excludes 16:00.
REQUEST_START_TIME = "09:30:00.000"
REQUEST_END_TIME = "16:00:00.000"

print("Cell 3 download configuration:")
print(f"Production feature panel:       {PRODUCTION_FEATURE_PANEL_PATH}")
print(f"Implied term structure path:    {IMPLIED_TERM_STRUCTURE_PATH}")
print(f"Fallback start date:            {FALLBACK_START_DATE}")
print(f"Fallback end date:              {FALLBACK_END_DATE}")
print(f"Venue:                          {SPY_STOCK_VENUE}")
print(f"Interval:                       {FULL_HISTORY_INTERVAL}")
print(f"Force refresh ThetaData:         {FORCE_REFRESH_THETA}")
print(f"Max months to download:          {MAX_MONTHS_TO_DOWNLOAD}")


# -----------------------------
# 2. Determine target date range
# -----------------------------

source_date_panel = None
source_date_label = None

if PRODUCTION_FEATURE_PANEL_PATH.exists():
    source_date_panel = pd.read_parquet(PRODUCTION_FEATURE_PANEL_PATH)
    source_date_label = "production_feature_panel_v0_1"
elif IMPLIED_TERM_STRUCTURE_PATH.exists():
    source_date_panel = pd.read_parquet(IMPLIED_TERM_STRUCTURE_PATH)
    source_date_label = "implied_term_structure_repaired"
else:
    source_date_panel = None
    source_date_label = "fallback_dates"

if source_date_panel is not None:
    source_date_panel = source_date_panel.copy()

    if "date" in source_date_panel.columns:
        source_date_panel["date"] = pd.to_datetime(source_date_panel["date"])
    elif "trade_date" in source_date_panel.columns:
        source_date_panel["date"] = pd.to_datetime(
            source_date_panel["trade_date"].astype(str),
            format="%Y%m%d",
            errors="coerce",
        )
    else:
        raise ValueError(
            f"Could not infer date from source panel columns: {list(source_date_panel.columns)}"
        )

    target_start_date = source_date_panel["date"].dropna().min()
    target_end_date = source_date_panel["date"].dropna().max()

    target_trade_dates = (
        source_date_panel[["date"]]
        .dropna()
        .drop_duplicates()
        .sort_values("date")
        .copy()
    )

    target_trade_dates["trade_date"] = (
        target_trade_dates["date"].dt.strftime("%Y%m%d").astype(int)
    )

else:
    target_start_date = pd.to_datetime(FALLBACK_START_DATE, format="%Y%m%d")
    target_end_date = pd.to_datetime(FALLBACK_END_DATE, format="%Y%m%d")

    target_trade_dates = pd.DataFrame({
        "date": pd.date_range(target_start_date, target_end_date, freq="B")
    })
    target_trade_dates["trade_date"] = (
        target_trade_dates["date"].dt.strftime("%Y%m%d").astype(int)
    )

print()
print("Target date range:")
print(f"Source date panel:      {source_date_label}")
print(f"Start date:             {target_start_date.date()}")
print(f"End date:               {target_end_date.date()}")
print(f"Target trading dates:   {len(target_trade_dates):,}")


# -----------------------------
# 3. Monthly batching helper
# -----------------------------

def yyyymmdd(ts):
    return pd.to_datetime(ts).strftime("%Y%m%d")


def build_monthly_batches(start_date, end_date):
    """
    Build month-by-month request batches.

    ThetaData stock OHLC multi-day requests are limited to one month.
    """
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    batches = []

    cur_month_start = start_date.replace(day=1)

    while cur_month_start <= end_date:
        month_start = cur_month_start
        month_end = cur_month_start + MonthEnd(0)

        batch_start = max(start_date, month_start)
        batch_end = min(end_date, month_end)

        if batch_start <= batch_end:
            batches.append({
                "batch_start_date": batch_start,
                "batch_end_date": batch_end,
                "start_yyyymmdd": yyyymmdd(batch_start),
                "end_yyyymmdd": yyyymmdd(batch_end),
                "batch_month": batch_start.strftime("%Y-%m"),
            })

        cur_month_start = (cur_month_start + MonthEnd(1) + pd.Timedelta(days=1)).replace(day=1)

    return pd.DataFrame(batches)


monthly_batches = build_monthly_batches(target_start_date, target_end_date)

if MAX_MONTHS_TO_DOWNLOAD is not None:
    monthly_batches = monthly_batches.head(MAX_MONTHS_TO_DOWNLOAD).copy()

print()
print("Monthly batches:")
display(monthly_batches.head())
display(monthly_batches.tail())
print(f"Batch count: {len(monthly_batches):,}")


# -----------------------------
# 4. Cache paths
# -----------------------------

SPY_OHLC_MONTHLY_RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "thetadata_spy_stock_5m_ohlc_monthly"
)

SPY_EOD_MONTHLY_RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "thetadata_spy_stock_eod_monthly"
)

SPY_OHLC_MONTHLY_CLEAN_DIR = (
    CORSI_PROCESSED_DIR
    / "spy_5m_clean_monthly"
)

for p in [
    SPY_OHLC_MONTHLY_RAW_DIR,
    SPY_EOD_MONTHLY_RAW_DIR,
    SPY_OHLC_MONTHLY_CLEAN_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print()
print("Monthly cache directories:")
print(f"Raw 5m OHLC:       {SPY_OHLC_MONTHLY_RAW_DIR}")
print(f"Raw EOD:           {SPY_EOD_MONTHLY_RAW_DIR}")
print(f"Clean 5m OHLC:     {SPY_OHLC_MONTHLY_CLEAN_DIR}")


def monthly_ohlc_cache_path(start_yyyymmdd, end_yyyymmdd, venue):
    return (
        SPY_OHLC_MONTHLY_RAW_DIR
        / f"SPY_5m_ohlc_{start_yyyymmdd}_{end_yyyymmdd}_{venue}.parquet"
    )


def monthly_eod_cache_path(start_yyyymmdd, end_yyyymmdd):
    return (
        SPY_EOD_MONTHLY_RAW_DIR
        / f"SPY_eod_{start_yyyymmdd}_{end_yyyymmdd}.parquet"
    )


def monthly_clean_cache_path(start_yyyymmdd, end_yyyymmdd, venue):
    return (
        SPY_OHLC_MONTHLY_CLEAN_DIR
        / f"SPY_5m_clean_ohlc_{start_yyyymmdd}_{end_yyyymmdd}_{venue}.parquet"
    )


# -----------------------------
# 5. Download monthly OHLC and EOD
# -----------------------------

download_audit_rows = []
monthly_clean_ohlc_list = []
monthly_eod_list = []
monthly_intraday_rv_list = []

for i, batch in monthly_batches.iterrows():
    start_yyyymmdd = batch["start_yyyymmdd"]
    end_yyyymmdd = batch["end_yyyymmdd"]
    batch_month = batch["batch_month"]

    print()
    print("-" * 90)
    print(f"Batch {i + 1:,} / {len(monthly_batches):,}: {batch_month} {start_yyyymmdd} to {end_yyyymmdd}")
    print("-" * 90)

    ohlc_cache = monthly_ohlc_cache_path(
        start_yyyymmdd,
        end_yyyymmdd,
        SPY_STOCK_VENUE,
    )

    eod_cache = monthly_eod_cache_path(
        start_yyyymmdd,
        end_yyyymmdd,
    )

    clean_cache = monthly_clean_cache_path(
        start_yyyymmdd,
        end_yyyymmdd,
        SPY_STOCK_VENUE,
    )

    batch_record = {
        "batch_month": batch_month,
        "start_date": start_yyyymmdd,
        "end_date": end_yyyymmdd,
        "venue": SPY_STOCK_VENUE,
        "ohlc_cache_exists_before": ohlc_cache.exists(),
        "eod_cache_exists_before": eod_cache.exists(),
        "clean_cache_exists_before": clean_cache.exists(),
        "ohlc_source": "",
        "eod_source": "",
        "raw_ohlc_rows": 0,
        "normalized_ohlc_rows": 0,
        "clean_ohlc_rows": 0,
        "excluded_ohlc_rows": 0,
        "eod_rows": 0,
        "intraday_rv_rows": 0,
        "ok": False,
        "error": "",
    }

    # ---- OHLC ----
    try:
        if ohlc_cache.exists() and not FORCE_REFRESH_THETA:
            month_ohlc_norm = pd.read_parquet(ohlc_cache)
            batch_record["ohlc_source"] = "cache"
        else:
            ohlc_params = {
                "symbol": SYMBOL,
                "start_date": start_yyyymmdd,
                "end_date": end_yyyymmdd,
                "interval": FULL_HISTORY_INTERVAL,
                "start_time": REQUEST_START_TIME,
                "end_time": REQUEST_END_TIME,
                "venue": SPY_STOCK_VENUE,
                "format": "csv",
            }

            ohlc_raw, ohlc_rec = theta_csv_request(
                THETA_STOCK_OHLC_ENDPOINT,
                params=ohlc_params,
                timeout=180,
            )

            batch_record["raw_ohlc_rows"] = len(ohlc_raw)

            month_ohlc_norm = normalize_ohlc_df(ohlc_raw)

            month_ohlc_norm.to_parquet(ohlc_cache, index=False)
            batch_record["ohlc_source"] = "thetadata"

            time.sleep(REQUEST_SLEEP_SECONDS)

        batch_record["normalized_ohlc_rows"] = len(month_ohlc_norm)

    except Exception as e:
        batch_record["error"] += f"OHLC error: {repr(e)} | "
        month_ohlc_norm = pd.DataFrame()

    # ---- EOD ----
    try:
        if eod_cache.exists() and not FORCE_REFRESH_THETA:
            month_eod_norm = pd.read_parquet(eod_cache)
            batch_record["eod_source"] = "cache"
        else:
            eod_params = {
                "symbol": SYMBOL,
                "start_date": start_yyyymmdd,
                "end_date": end_yyyymmdd,
                "format": "csv",
            }

            eod_raw_month, eod_rec_month = theta_csv_request(
                THETA_STOCK_EOD_ENDPOINT,
                params=eod_params,
                timeout=90,
            )

            month_eod_norm = normalize_eod_df(eod_raw_month)

            month_eod_norm.to_parquet(eod_cache, index=False)
            batch_record["eod_source"] = "thetadata"

            time.sleep(REQUEST_SLEEP_SECONDS)

        batch_record["eod_rows"] = len(month_eod_norm)

    except Exception as e:
        batch_record["error"] += f"EOD error: {repr(e)} | "
        month_eod_norm = pd.DataFrame()

    # ---- Clean OHLC using monthly EOD dates ----
    try:
        if clean_cache.exists() and not FORCE_REFRESH_THETA:
            clean_ohlc_month = pd.read_parquet(clean_cache)
            excluded_ohlc_month = pd.DataFrame()
        else:
            clean_ohlc_month, excluded_ohlc_month = clean_spy_5m_ohlc_sample(
                month_ohlc_norm,
                month_eod_norm,
            )

            clean_ohlc_month.to_parquet(clean_cache, index=False)

        batch_record["clean_ohlc_rows"] = len(clean_ohlc_month)
        batch_record["excluded_ohlc_rows"] = (
            len(month_ohlc_norm) - len(clean_ohlc_month)
            if len(month_ohlc_norm) >= len(clean_ohlc_month)
            else np.nan
        )

        intraday_rv_month = compute_clean_daily_intraday_rv(clean_ohlc_month)
        batch_record["intraday_rv_rows"] = len(intraday_rv_month)

    except Exception as e:
        batch_record["error"] += f"Clean/RV error: {repr(e)} | "
        clean_ohlc_month = pd.DataFrame()
        intraday_rv_month = pd.DataFrame()

    batch_record["ok"] = (
        len(month_eod_norm) > 0
        and len(clean_ohlc_month) > 0
        and len(intraday_rv_month) > 0
        and batch_record["error"] == ""
    )

    download_audit_rows.append(batch_record)

    if not clean_ohlc_month.empty:
        monthly_clean_ohlc_list.append(clean_ohlc_month)

    if not month_eod_norm.empty:
        monthly_eod_list.append(month_eod_norm)

    if not intraday_rv_month.empty:
        monthly_intraday_rv_list.append(intraday_rv_month)

    print(
        f"OHLC rows norm/clean: {batch_record['normalized_ohlc_rows']:,} / "
        f"{batch_record['clean_ohlc_rows']:,}; "
        f"EOD rows: {batch_record['eod_rows']:,}; "
        f"Daily RV rows: {batch_record['intraday_rv_rows']:,}; "
        f"OK: {batch_record['ok']}"
    )

    if batch_record["error"]:
        print(f"ERROR: {batch_record['error']}")


download_audit_df = pd.DataFrame(download_audit_rows)

print()
print("Download audit summary:")
display(download_audit_df.tail(20))


# -----------------------------
# 6. Combine full history
# -----------------------------

if len(monthly_clean_ohlc_list) == 0:
    raise RuntimeError("No clean OHLC data was built. Stop before continuing.")

if len(monthly_eod_list) == 0:
    raise RuntimeError("No EOD data was built. Stop before continuing.")

if len(monthly_intraday_rv_list) == 0:
    raise RuntimeError("No intraday RV rows were built. Stop before continuing.")

spy_5m_clean_ohlc_full = (
    pd.concat(monthly_clean_ohlc_list, ignore_index=True)
    .drop_duplicates(subset=["trade_date", "timestamp"])
    .sort_values(["trade_date", "timestamp"])
    .reset_index(drop=True)
)

spy_eod_full = (
    pd.concat(monthly_eod_list, ignore_index=True)
    .drop_duplicates(subset=["trade_date"])
    .sort_values("date")
    .reset_index(drop=True)
)

spy_intraday_rv_full = (
    pd.concat(monthly_intraday_rv_list, ignore_index=True)
    .drop_duplicates(subset=["trade_date"])
    .sort_values("date")
    .reset_index(drop=True)
)

# Important: compute overnight variance on the full combined EOD panel,
# not month-by-month, so month boundaries are handled correctly.
spy_daily_rv_full = add_eod_and_overnight_variance(
    spy_intraday_rv_full,
    spy_eod_full,
)

spy_daily_rv_full["overnight_missing_flag"] = spy_daily_rv_full["prior_eod_close"].isna()

print()
print("Combined full-history row counts:")
combined_row_summary = pd.DataFrame([{
    "clean_ohlc_rows": len(spy_5m_clean_ohlc_full),
    "clean_ohlc_dates": spy_5m_clean_ohlc_full["trade_date"].nunique(),
    "eod_rows": len(spy_eod_full),
    "intraday_rv_rows": len(spy_intraday_rv_full),
    "daily_rv_rows": len(spy_daily_rv_full),
    "first_daily_rv_date": spy_daily_rv_full["date"].min(),
    "last_daily_rv_date": spy_daily_rv_full["date"].max(),
}])
display(combined_row_summary)


# -----------------------------
# 7. Full-history QA
# -----------------------------

bar_count_by_day = (
    spy_5m_clean_ohlc_full
    .groupby(["trade_date", "date"])
    .size()
    .reset_index(name="bar_count")
)

bar_count_dist = (
    bar_count_by_day["bar_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

bar_count_dist.columns = ["bar_count", "days"]

quality_status_counts = (
    spy_daily_rv_full["daily_rv_quality_status"]
    .value_counts()
    .reset_index()
)

quality_status_counts.columns = ["daily_rv_quality_status", "days"]

yearly_coverage = (
    spy_daily_rv_full
    .assign(year=spy_daily_rv_full["date"].dt.year)
    .groupby("year")
    .agg(
        daily_rows=("trade_date", "size"),
        ok_days=("daily_rv_quality_status", lambda x: int((x == "OK").sum())),
        review_days=("daily_rv_quality_status", lambda x: int((x == "REVIEW").sum())),
        mean_total_realized_vol=("total_realized_vol_annualized", "mean"),
        p95_total_realized_vol=("total_realized_vol_annualized", lambda x: x.quantile(0.95)),
        max_total_realized_vol=("total_realized_vol_annualized", "max"),
    )
    .reset_index()
)

extreme_overnight_moves = (
    spy_daily_rv_full
    .loc[spy_daily_rv_full["overnight_return"].notna()]
    .assign(abs_overnight_return=lambda x: x["overnight_return"].abs())
    .sort_values("abs_overnight_return", ascending=False)
    .head(25)
    .copy()
)

extreme_total_rv_days = (
    spy_daily_rv_full
    .sort_values("total_realized_vol_annualized", ascending=False)
    .head(25)
    .copy()
)

print()
print("Bar count distribution:")
display(bar_count_dist)

print()
print("Daily RV quality status counts:")
display(quality_status_counts)

print()
print("Yearly coverage / realized-vol summary:")
display(yearly_coverage)

print()
print("Largest absolute overnight returns:")
display(extreme_overnight_moves[
    [
        "trade_date",
        "date",
        "prior_eod_close",
        "eod_open",
        "eod_close",
        "overnight_return",
        "total_realized_vol_annualized",
        "daily_rv_quality_status",
    ]
])

print()
print("Largest total realized-vol days:")
display(extreme_total_rv_days[
    [
        "trade_date",
        "date",
        "intraday_realized_vol_annualized",
        "total_realized_vol_annualized",
        "overnight_return",
        "bar_count",
        "daily_rv_quality_status",
    ]
])


# -----------------------------
# 8. Coverage versus existing target dates
# -----------------------------

rv_trade_dates = set(spy_daily_rv_full["trade_date"].astype(int))

target_trade_date_set = set(target_trade_dates["trade_date"].astype(int))

missing_rv_target_dates = sorted(list(target_trade_date_set - rv_trade_dates))
extra_rv_dates = sorted(list(rv_trade_dates - target_trade_date_set))

coverage_summary = pd.DataFrame([{
    "source_date_panel": source_date_label,
    "target_start_date": target_start_date,
    "target_end_date": target_end_date,
    "target_trade_dates": len(target_trade_date_set),
    "spy_daily_rv_dates": len(rv_trade_dates),
    "missing_rv_target_dates": len(missing_rv_target_dates),
    "extra_rv_dates": len(extra_rv_dates),
    "coverage_pct": (
        100.0 * (len(target_trade_date_set) - len(missing_rv_target_dates))
        / len(target_trade_date_set)
        if len(target_trade_date_set) > 0
        else np.nan
    ),
}])

missing_rv_target_dates_df = pd.DataFrame({
    "trade_date": missing_rv_target_dates
})

if not missing_rv_target_dates_df.empty:
    missing_rv_target_dates_df["date"] = pd.to_datetime(
        missing_rv_target_dates_df["trade_date"].astype(str),
        format="%Y%m%d",
        errors="coerce",
    )

print()
print("Coverage versus existing feature/implied dates:")
display(coverage_summary)

print()
print("First missing RV target dates, if any:")
display(missing_rv_target_dates_df.head(50))


# -----------------------------
# 9. Save Cell 3 outputs
# -----------------------------

safe_start = yyyymmdd(target_start_date)
safe_end = yyyymmdd(target_end_date)

cell3_clean_ohlc_output = (
    CORSI_PROCESSED_DIR
    / f"spy_5m_clean_ohlc_full_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell3_eod_output = (
    CORSI_PROCESSED_DIR
    / f"spy_eod_full_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

cell3_daily_rv_output = (
    CORSI_PROCESSED_DIR
    / f"spy_daily_realized_variance_corsi_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell3_download_audit_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3_monthly_download_audit_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3_quality_status_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3_quality_status_counts_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3_bar_count_dist_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3_bar_count_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3_yearly_coverage_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3_yearly_coverage_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3_coverage_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3_coverage_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3_missing_dates_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3_missing_rv_target_dates_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3_extreme_overnight_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3_extreme_overnight_moves_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3_extreme_total_rv_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3_extreme_total_rv_days_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

spy_5m_clean_ohlc_full.to_parquet(cell3_clean_ohlc_output, index=False)
spy_eod_full.to_parquet(cell3_eod_output, index=False)
spy_daily_rv_full.to_parquet(cell3_daily_rv_output, index=False)

download_audit_df.to_csv(cell3_download_audit_output, index=False)
quality_status_counts.to_csv(cell3_quality_status_output, index=False)
bar_count_dist.to_csv(cell3_bar_count_dist_output, index=False)
yearly_coverage.to_csv(cell3_yearly_coverage_output, index=False)
coverage_summary.to_csv(cell3_coverage_summary_output, index=False)
missing_rv_target_dates_df.to_csv(cell3_missing_dates_output, index=False)
extreme_overnight_moves.to_csv(cell3_extreme_overnight_output, index=False)
extreme_total_rv_days.to_csv(cell3_extreme_total_rv_output, index=False)

print()
print("Cell 3 outputs saved:")
print(f"Clean 5m OHLC full:        {cell3_clean_ohlc_output}")
print(f"SPY EOD full:              {cell3_eod_output}")
print(f"SPY daily RV full:         {cell3_daily_rv_output}")
print(f"Monthly download audit:    {cell3_download_audit_output}")
print(f"Quality status counts:     {cell3_quality_status_output}")
print(f"Bar count distribution:    {cell3_bar_count_dist_output}")
print(f"Yearly coverage:           {cell3_yearly_coverage_output}")
print(f"Coverage summary:          {cell3_coverage_summary_output}")
print(f"Missing target dates:      {cell3_missing_dates_output}")
print(f"Extreme overnight moves:   {cell3_extreme_overnight_output}")
print(f"Extreme total RV days:     {cell3_extreme_total_rv_output}")

print()
print("Cell 3 complete.")
print("Next step will be Cell 4: build forward realized-variance targets from the SPY daily RV table.")

## 3B. Sanitize daily realized variance

Recomputes overnight variance using the prior valid positive close, removes infinities, classifies session quality, and writes the model-ready daily realized-variance table.

In [ ]:
# Cell 3B: Sanitize SPY daily realized-variance table before target construction
#
# Purpose:
# Cell 3 successfully built full SPY daily RV history, but found one major issue:
# - 2019-06-10 has prior_eod_close = 0, causing infinite overnight return / total RV.
#
# Also, some non-78-bar days are legitimate shortened / irregular sessions.
# For Corsi target construction, the key requirement is:
# - finite daily variance
# - positive EOD open/close
# - positive prior valid close
# - traceable quality flags
#
# This cell:
# 1. Recomputes overnight variance using the previous valid positive EOD close.
# 2. Removes infinities.
# 3. Separates full-session from partial-session quality.
# 4. Builds a clean modeling-ready SPY daily RV table.
#
# This does NOT modify production files.

# -----------------------------
# 1. Load from Cell 3 outputs if needed
# -----------------------------

try:
    spy_daily_rv_full
    spy_eod_full
except NameError:
    print("Cell 3 objects not in memory. Loading latest Corsi v1 files from processed folder.")

    latest_daily_rv_files = sorted(
        CORSI_PROCESSED_DIR.glob("spy_daily_realized_variance_corsi_v1_*.parquet")
    )
    latest_eod_files = sorted(
        CORSI_PROCESSED_DIR.glob("spy_eod_full_*.parquet")
    )

    if not latest_daily_rv_files:
        raise FileNotFoundError("No spy_daily_realized_variance_corsi_v1 parquet files found.")

    if not latest_eod_files:
        raise FileNotFoundError("No spy_eod_full parquet files found.")

    latest_daily_rv_file = latest_daily_rv_files[-1]
    latest_eod_file = latest_eod_files[-1]

    print(f"Loading daily RV: {latest_daily_rv_file}")
    print(f"Loading EOD:      {latest_eod_file}")

    spy_daily_rv_full = pd.read_parquet(latest_daily_rv_file)
    spy_eod_full = pd.read_parquet(latest_eod_file)


# -----------------------------
# 2. Build valid EOD close chain
# -----------------------------

spy_eod_sanitized = spy_eod_full.copy()
spy_eod_sanitized["date"] = pd.to_datetime(spy_eod_sanitized["date"])
spy_eod_sanitized = spy_eod_sanitized.sort_values("date").reset_index(drop=True)

for c in ["open", "high", "low", "close"]:
    spy_eod_sanitized[c] = pd.to_numeric(spy_eod_sanitized[c], errors="coerce")

spy_eod_sanitized["eod_open_positive"] = spy_eod_sanitized["open"] > 0
spy_eod_sanitized["eod_close_positive"] = spy_eod_sanitized["close"] > 0

# Use previous valid positive close, not blindly previous row close.
spy_eod_sanitized["valid_positive_close_for_lag"] = (
    spy_eod_sanitized["close"]
    .where(spy_eod_sanitized["close"] > 0)
)

spy_eod_sanitized["prior_valid_eod_close"] = (
    spy_eod_sanitized["valid_positive_close_for_lag"]
    .ffill()
    .shift(1)
)

spy_eod_sanitized["overnight_return_sanitized"] = np.where(
    (spy_eod_sanitized["open"] > 0)
    & (spy_eod_sanitized["prior_valid_eod_close"] > 0),
    np.log(spy_eod_sanitized["open"] / spy_eod_sanitized["prior_valid_eod_close"]),
    np.nan,
)

spy_eod_sanitized["overnight_realized_variance_raw_sanitized"] = (
    spy_eod_sanitized["overnight_return_sanitized"] ** 2
)

eod_sanitized_join = spy_eod_sanitized[
    [
        "trade_date",
        "date",
        "open",
        "close",
        "prior_valid_eod_close",
        "overnight_return_sanitized",
        "overnight_realized_variance_raw_sanitized",
        "eod_open_positive",
        "eod_close_positive",
    ]
].rename(columns={
    "open": "eod_open_sanitized",
    "close": "eod_close_sanitized",
})


# -----------------------------
# 3. Join sanitized overnight values back to daily RV
# -----------------------------

spy_daily_rv_sanitized = spy_daily_rv_full.copy()
spy_daily_rv_sanitized["date"] = pd.to_datetime(spy_daily_rv_sanitized["date"])

# Drop old sanitized columns if rerunning.
drop_cols = [
    "eod_open_sanitized",
    "eod_close_sanitized",
    "prior_valid_eod_close",
    "overnight_return_sanitized",
    "overnight_realized_variance_raw_sanitized",
    "eod_open_positive",
    "eod_close_positive",
]

spy_daily_rv_sanitized = spy_daily_rv_sanitized.drop(
    columns=[c for c in drop_cols if c in spy_daily_rv_sanitized.columns],
    errors="ignore",
)

spy_daily_rv_sanitized = spy_daily_rv_sanitized.merge(
    eod_sanitized_join,
    on=["trade_date", "date"],
    how="left",
    validate="one_to_one",
)

# Replace infinite values from original calculation.
spy_daily_rv_sanitized["intraday_realized_variance_raw"] = pd.to_numeric(
    spy_daily_rv_sanitized["intraday_realized_variance_raw"],
    errors="coerce",
)

spy_daily_rv_sanitized["intraday_realized_variance_raw"] = (
    spy_daily_rv_sanitized["intraday_realized_variance_raw"]
    .replace([np.inf, -np.inf], np.nan)
)

spy_daily_rv_sanitized["total_realized_variance_raw_sanitized"] = (
    spy_daily_rv_sanitized["intraday_realized_variance_raw"]
    + spy_daily_rv_sanitized["overnight_realized_variance_raw_sanitized"].fillna(0.0)
)

spy_daily_rv_sanitized["total_realized_variance_raw_sanitized"] = (
    spy_daily_rv_sanitized["total_realized_variance_raw_sanitized"]
    .replace([np.inf, -np.inf], np.nan)
)

spy_daily_rv_sanitized["intraday_realized_variance_annualized_sanitized"] = (
    spy_daily_rv_sanitized["intraday_realized_variance_raw"] * 252
)

spy_daily_rv_sanitized["intraday_realized_vol_annualized_sanitized"] = (
    np.sqrt(spy_daily_rv_sanitized["intraday_realized_variance_annualized_sanitized"])
    * 100
)

spy_daily_rv_sanitized["total_realized_variance_annualized_sanitized"] = (
    spy_daily_rv_sanitized["total_realized_variance_raw_sanitized"] * 252
)

spy_daily_rv_sanitized["total_realized_vol_annualized_sanitized"] = (
    np.sqrt(spy_daily_rv_sanitized["total_realized_variance_annualized_sanitized"])
    * 100
)


# -----------------------------
# 4. Quality flags
# -----------------------------

spy_daily_rv_sanitized["finite_intraday_rv"] = np.isfinite(
    spy_daily_rv_sanitized["intraday_realized_variance_raw"]
)

spy_daily_rv_sanitized["finite_total_rv_sanitized"] = np.isfinite(
    spy_daily_rv_sanitized["total_realized_variance_raw_sanitized"]
)

spy_daily_rv_sanitized["positive_total_rv_sanitized"] = (
    spy_daily_rv_sanitized["total_realized_variance_raw_sanitized"] >= 0
)

spy_daily_rv_sanitized["has_valid_overnight"] = (
    spy_daily_rv_sanitized["overnight_return_sanitized"].notna()
)

# Bar-count session classification.
# 78 = normal full session under our 09:30 to 15:55 convention.
# 43 is common for early closes.
spy_daily_rv_sanitized["session_length_class"] = np.select(
    [
        spy_daily_rv_sanitized["bar_count"] == 78,
        spy_daily_rv_sanitized["bar_count"].between(40, 50),
        spy_daily_rv_sanitized["bar_count"].between(70, 77),
        spy_daily_rv_sanitized["bar_count"] < 40,
    ],
    [
        "FULL_78_BARS",
        "SHORT_SESSION_LIKELY_EARLY_CLOSE",
        "NEAR_FULL_SESSION_MISSING_BARS",
        "VERY_SHORT_OR_BAD_SESSION",
    ],
    default="OTHER",
)

spy_daily_rv_sanitized["corsi_model_usable"] = (
    spy_daily_rv_sanitized["finite_intraday_rv"]
    & spy_daily_rv_sanitized["finite_total_rv_sanitized"]
    & spy_daily_rv_sanitized["positive_total_rv_sanitized"]
    & spy_daily_rv_sanitized["eod_open_positive"].fillna(False)
    & spy_daily_rv_sanitized["eod_close_positive"].fillna(False)
)

spy_daily_rv_sanitized["corsi_quality_status"] = np.select(
    [
        spy_daily_rv_sanitized["corsi_model_usable"]
        & (spy_daily_rv_sanitized["session_length_class"] == "FULL_78_BARS"),

        spy_daily_rv_sanitized["corsi_model_usable"]
        & (spy_daily_rv_sanitized["session_length_class"] == "SHORT_SESSION_LIKELY_EARLY_CLOSE"),

        spy_daily_rv_sanitized["corsi_model_usable"]
        & (spy_daily_rv_sanitized["session_length_class"] == "NEAR_FULL_SESSION_MISSING_BARS"),

        spy_daily_rv_sanitized["corsi_model_usable"],
    ],
    [
        "OK_FULL_SESSION",
        "OK_SHORT_SESSION",
        "OK_NEAR_FULL_REVIEW_BARS",
        "OK_OTHER_REVIEW_BARS",
    ],
    default="EXCLUDE_BAD_DATA",
)

spy_daily_rv_sanitized = (
    spy_daily_rv_sanitized
    .sort_values("date")
    .reset_index(drop=True)
)


# -----------------------------
# 5. Diagnostics
# -----------------------------

sanitize_summary = pd.DataFrame([{
    "rows": len(spy_daily_rv_sanitized),
    "first_date": spy_daily_rv_sanitized["date"].min(),
    "last_date": spy_daily_rv_sanitized["date"].max(),
    "finite_total_rv_rows": int(spy_daily_rv_sanitized["finite_total_rv_sanitized"].sum()),
    "nonfinite_total_rv_rows": int((~spy_daily_rv_sanitized["finite_total_rv_sanitized"]).sum()),
    "corsi_model_usable_rows": int(spy_daily_rv_sanitized["corsi_model_usable"].sum()),
    "excluded_bad_data_rows": int((spy_daily_rv_sanitized["corsi_quality_status"] == "EXCLUDE_BAD_DATA").sum()),
    "max_total_realized_vol_sanitized": spy_daily_rv_sanitized["total_realized_vol_annualized_sanitized"].max(),
    "mean_total_realized_vol_sanitized": spy_daily_rv_sanitized["total_realized_vol_annualized_sanitized"].mean(),
}])

quality_counts_3b = (
    spy_daily_rv_sanitized["corsi_quality_status"]
    .value_counts()
    .reset_index()
)

quality_counts_3b.columns = ["corsi_quality_status", "days"]

session_length_counts_3b = (
    spy_daily_rv_sanitized["session_length_class"]
    .value_counts()
    .reset_index()
)

session_length_counts_3b.columns = ["session_length_class", "days"]

bad_rows_3b = (
    spy_daily_rv_sanitized
    .loc[spy_daily_rv_sanitized["corsi_quality_status"] == "EXCLUDE_BAD_DATA"]
    .copy()
)

largest_total_rv_3b = (
    spy_daily_rv_sanitized
    .sort_values("total_realized_vol_annualized_sanitized", ascending=False)
    .head(25)
    .copy()
)

largest_overnight_3b = (
    spy_daily_rv_sanitized
    .loc[spy_daily_rv_sanitized["overnight_return_sanitized"].notna()]
    .assign(abs_overnight_return_sanitized=lambda x: x["overnight_return_sanitized"].abs())
    .sort_values("abs_overnight_return_sanitized", ascending=False)
    .head(25)
    .copy()
)

yearly_sanitized_summary = (
    spy_daily_rv_sanitized
    .assign(year=spy_daily_rv_sanitized["date"].dt.year)
    .groupby("year")
    .agg(
        rows=("trade_date", "size"),
        usable_rows=("corsi_model_usable", "sum"),
        excluded_rows=("corsi_model_usable", lambda x: int((~x).sum())),
        full_session_rows=("session_length_class", lambda x: int((x == "FULL_78_BARS").sum())),
        short_session_rows=("session_length_class", lambda x: int((x == "SHORT_SESSION_LIKELY_EARLY_CLOSE").sum())),
        mean_total_realized_vol=("total_realized_vol_annualized_sanitized", "mean"),
        p95_total_realized_vol=("total_realized_vol_annualized_sanitized", lambda x: x.quantile(0.95)),
        max_total_realized_vol=("total_realized_vol_annualized_sanitized", "max"),
    )
    .reset_index()
)

print("Cell 3B sanitize summary:")
display(sanitize_summary)

print()
print("Corsi quality status counts:")
display(quality_counts_3b)

print()
print("Session length counts:")
display(session_length_counts_3b)

print()
print("Excluded bad-data rows:")
display(bad_rows_3b[
    [
        "trade_date",
        "date",
        "bar_count",
        "first_open",
        "last_close",
        "eod_open_sanitized",
        "eod_close_sanitized",
        "prior_valid_eod_close",
        "intraday_realized_variance_raw",
        "overnight_return_sanitized",
        "total_realized_variance_raw_sanitized",
        "corsi_quality_status",
        "session_length_class",
    ]
])

print()
print("Yearly sanitized realized-vol summary:")
display(yearly_sanitized_summary)

print()
print("Largest total realized-vol days after sanitizing:")
display(largest_total_rv_3b[
    [
        "trade_date",
        "date",
        "intraday_realized_vol_annualized_sanitized",
        "total_realized_vol_annualized_sanitized",
        "overnight_return_sanitized",
        "bar_count",
        "session_length_class",
        "corsi_quality_status",
    ]
])

print()
print("Largest absolute overnight returns after sanitizing:")
display(largest_overnight_3b[
    [
        "trade_date",
        "date",
        "prior_valid_eod_close",
        "eod_open_sanitized",
        "eod_close_sanitized",
        "overnight_return_sanitized",
        "total_realized_vol_annualized_sanitized",
        "session_length_class",
        "corsi_quality_status",
    ]
])


# -----------------------------
# 6. Build modeling-ready table
# -----------------------------

spy_daily_rv_model_ready = (
    spy_daily_rv_sanitized
    .loc[spy_daily_rv_sanitized["corsi_model_usable"]]
    .copy()
)

# Create clean canonical columns for downstream notebooks.
spy_daily_rv_model_ready["spy_intraday_rv_raw"] = (
    spy_daily_rv_model_ready["intraday_realized_variance_raw"]
)

spy_daily_rv_model_ready["spy_overnight_rv_raw"] = (
    spy_daily_rv_model_ready["overnight_realized_variance_raw_sanitized"]
)

spy_daily_rv_model_ready["spy_total_rv_raw"] = (
    spy_daily_rv_model_ready["total_realized_variance_raw_sanitized"]
)

spy_daily_rv_model_ready["spy_intraday_vol_ann"] = (
    spy_daily_rv_model_ready["intraday_realized_vol_annualized_sanitized"]
)

spy_daily_rv_model_ready["spy_total_vol_ann"] = (
    spy_daily_rv_model_ready["total_realized_vol_annualized_sanitized"]
)

spy_daily_rv_model_ready["spy_overnight_return"] = (
    spy_daily_rv_model_ready["overnight_return_sanitized"]
)

spy_daily_rv_model_ready = (
    spy_daily_rv_model_ready
    .sort_values("date")
    .reset_index(drop=True)
)

model_ready_cols = [
    "trade_date",
    "date",
    "bar_count",
    "session_length_class",
    "corsi_quality_status",
    "first_open",
    "last_close",
    "eod_open_sanitized",
    "eod_close_sanitized",
    "prior_valid_eod_close",
    "intraday_open_to_close_return",
    "spy_overnight_return",
    "spy_intraday_rv_raw",
    "spy_overnight_rv_raw",
    "spy_total_rv_raw",
    "spy_intraday_vol_ann",
    "spy_total_vol_ann",
    "total_volume",
    "total_trade_count",
]

print()
print("Model-ready SPY daily RV table:")
display(spy_daily_rv_model_ready[model_ready_cols].head())
display(spy_daily_rv_model_ready[model_ready_cols].tail())


# -----------------------------
# 7. Save Cell 3B outputs
# -----------------------------

safe_start = spy_daily_rv_sanitized["date"].min().strftime("%Y%m%d")
safe_end = spy_daily_rv_sanitized["date"].max().strftime("%Y%m%d")

cell3b_sanitized_output = (
    CORSI_PROCESSED_DIR
    / f"spy_daily_realized_variance_corsi_v1_sanitized_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell3b_model_ready_output = (
    CORSI_PROCESSED_DIR
    / f"spy_daily_realized_variance_corsi_v1_model_ready_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell3b_model_ready_csv_output = (
    CORSI_AUDIT_DIR
    / f"spy_daily_realized_variance_corsi_v1_model_ready_sample_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3b_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3b_sanitize_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3b_quality_counts_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3b_quality_counts_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3b_session_counts_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3b_session_length_counts_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3b_bad_rows_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3b_excluded_bad_rows_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3b_yearly_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3b_yearly_sanitized_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3b_largest_total_rv_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3b_largest_total_rv_days_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell3b_largest_overnight_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell3b_largest_overnight_moves_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

spy_daily_rv_sanitized.to_parquet(cell3b_sanitized_output, index=False)
spy_daily_rv_model_ready.to_parquet(cell3b_model_ready_output, index=False)

spy_daily_rv_model_ready[model_ready_cols].to_csv(cell3b_model_ready_csv_output, index=False)
sanitize_summary.to_csv(cell3b_summary_output, index=False)
quality_counts_3b.to_csv(cell3b_quality_counts_output, index=False)
session_length_counts_3b.to_csv(cell3b_session_counts_output, index=False)
bad_rows_3b.to_csv(cell3b_bad_rows_output, index=False)
yearly_sanitized_summary.to_csv(cell3b_yearly_output, index=False)
largest_total_rv_3b.to_csv(cell3b_largest_total_rv_output, index=False)
largest_overnight_3b.to_csv(cell3b_largest_overnight_output, index=False)

print()
print("Cell 3B outputs saved:")
print(f"Sanitized daily RV table:       {cell3b_sanitized_output}")
print(f"Model-ready daily RV table:     {cell3b_model_ready_output}")
print(f"Model-ready sample CSV:         {cell3b_model_ready_csv_output}")
print(f"Sanitize summary:               {cell3b_summary_output}")
print(f"Quality counts:                 {cell3b_quality_counts_output}")
print(f"Session counts:                 {cell3b_session_counts_output}")
print(f"Excluded bad rows:              {cell3b_bad_rows_output}")
print(f"Yearly sanitized summary:       {cell3b_yearly_output}")
print(f"Largest total RV days:          {cell3b_largest_total_rv_output}")
print(f"Largest overnight moves:        {cell3b_largest_overnight_output}")

print()
print("Cell 3B complete.")
print("Next step will be Cell 4: build forward realized-variance targets from the model-ready SPY daily RV table.")

## 4. Build Corsi/HAR forward realized-variance targets

Creates the forward realized-variance target for each fixed tenor. The target window is **date > signal date** and **date <= signal date + tenor calendar days**. Variance is annualized using **365 / tenor calendar days**.

In [ ]:
# Cell 4: Build Corsi / HAR-RV forward realized-variance targets
#
# Purpose:
# Build the supervised-learning target for the Corsi model.
#
# For each production feature-panel date and target tenor:
#
#   target = realized SPY total variance over the next T calendar days
#
# using the model-ready SPY daily RV table from Cell 3B.
#
# Convention:
#   Forward window = dates > signal_date and <= signal_date + tenor calendar days
#   Annualization = 365 / tenor_calendar_days
#
# This matches the calendar-day nature of the SPX implied-variance term structure.
#
# This cell does NOT fit the model yet.
# It creates the target table that later cells will use.

# -----------------------------
# 1. Load required data if needed
# -----------------------------

try:
    spy_daily_rv_model_ready
except NameError:
    print("spy_daily_rv_model_ready not in memory. Loading latest model-ready Corsi RV file.")

    latest_model_ready_files = sorted(
        CORSI_PROCESSED_DIR.glob("spy_daily_realized_variance_corsi_v1_model_ready_*.parquet")
    )

    if not latest_model_ready_files:
        raise FileNotFoundError("No model-ready SPY daily RV parquet files found.")

    latest_model_ready_file = latest_model_ready_files[-1]
    print(f"Loading: {latest_model_ready_file}")

    spy_daily_rv_model_ready = pd.read_parquet(latest_model_ready_file)

try:
    feature
except NameError:
    if "PRODUCTION_FEATURE_PANEL_PATH" not in globals():
        PRODUCTION_FEATURE_PANEL_PATH = (
            PROJECT_ROOT
            / "data"
            / "processed"
            / "production_feature_panel_v0_1.parquet"
        )

    if not PRODUCTION_FEATURE_PANEL_PATH.exists():
        raise FileNotFoundError(f"Production feature panel not found: {PRODUCTION_FEATURE_PANEL_PATH}")

    print(f"Loading production feature panel: {PRODUCTION_FEATURE_PANEL_PATH}")
    feature = pd.read_parquet(PRODUCTION_FEATURE_PANEL_PATH)


# -----------------------------
# 2. Configuration
# -----------------------------

CORSI_TARGET_ANNUALIZATION_DAYS = 365.0

DEFAULT_TARGET_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "tenor" in feature.columns:
    TARGET_TENORS = sorted(
        pd.to_numeric(feature["tenor"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
else:
    TARGET_TENORS = DEFAULT_TARGET_TENORS

TARGET_TENORS = [t for t in TARGET_TENORS if t in DEFAULT_TARGET_TENORS]

print("Cell 4 Corsi target configuration:")
print(f"Target tenors:                  {TARGET_TENORS}")
print(f"Annualization days:             {CORSI_TARGET_ANNUALIZATION_DAYS}")
print(f"Forward window convention:       date > signal_date and date <= signal_date + tenor calendar days")


# -----------------------------
# 3. Normalize source tables
# -----------------------------

spy_rv = spy_daily_rv_model_ready.copy()
spy_rv["date"] = pd.to_datetime(spy_rv["date"])
spy_rv["trade_date"] = pd.to_numeric(spy_rv["trade_date"], errors="coerce").astype(int)

required_spy_cols = [
    "trade_date",
    "date",
    "spy_intraday_rv_raw",
    "spy_overnight_rv_raw",
    "spy_total_rv_raw",
    "spy_intraday_vol_ann",
    "spy_total_vol_ann",
    "spy_overnight_return",
    "session_length_class",
    "corsi_quality_status",
]

missing_spy_cols = [c for c in required_spy_cols if c not in spy_rv.columns]

if missing_spy_cols:
    raise ValueError(f"Missing required SPY RV columns: {missing_spy_cols}")

spy_rv = (
    spy_rv[required_spy_cols]
    .sort_values("date")
    .reset_index(drop=True)
)

feature_dates = feature.copy()

if "date" in feature_dates.columns:
    feature_dates["date"] = pd.to_datetime(feature_dates["date"])
elif "trade_date" in feature_dates.columns:
    feature_dates["date"] = pd.to_datetime(
        feature_dates["trade_date"].astype(str),
        format="%Y%m%d",
        errors="coerce",
    )
else:
    raise ValueError("Feature panel has neither date nor trade_date column.")

feature_dates["trade_date"] = (
    feature_dates["date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

signal_dates = (
    feature_dates[["trade_date", "date"]]
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

print()
print("Source table summaries:")
source_summary = pd.DataFrame([
    {
        "table": "spy_rv_model_ready",
        "rows": len(spy_rv),
        "first_date": spy_rv["date"].min(),
        "last_date": spy_rv["date"].max(),
        "unique_trade_dates": spy_rv["trade_date"].nunique(),
    },
    {
        "table": "production_signal_dates",
        "rows": len(signal_dates),
        "first_date": signal_dates["date"].min(),
        "last_date": signal_dates["date"].max(),
        "unique_trade_dates": signal_dates["trade_date"].nunique(),
    },
])
display(source_summary)


# -----------------------------
# 4. Build forward target table
# -----------------------------

max_spy_rv_date = spy_rv["date"].max()
min_spy_rv_date = spy_rv["date"].min()

spy_rv_dates = spy_rv["date"].to_numpy()

target_rows = []

for _, signal_row in signal_dates.iterrows():
    signal_date = pd.Timestamp(signal_row["date"])
    signal_trade_date = int(signal_row["trade_date"])

    for tenor in TARGET_TENORS:
        tenor = int(tenor)

        forward_end_date = signal_date + pd.Timedelta(days=tenor)

        window = spy_rv[
            (spy_rv["date"] > signal_date)
            & (spy_rv["date"] <= forward_end_date)
        ].copy()

        forward_window_complete = (
            forward_end_date <= max_spy_rv_date
            and signal_date >= min_spy_rv_date
        )

        forward_num_sessions = len(window)

        if forward_num_sessions > 0:
            intraday_sum_raw = float(window["spy_intraday_rv_raw"].sum())
            overnight_sum_raw = float(window["spy_overnight_rv_raw"].fillna(0.0).sum())
            total_sum_raw = float(window["spy_total_rv_raw"].sum())

            annualized_total_var = total_sum_raw * CORSI_TARGET_ANNUALIZATION_DAYS / tenor
            annualized_intraday_var = intraday_sum_raw * CORSI_TARGET_ANNUALIZATION_DAYS / tenor
            annualized_overnight_var = overnight_sum_raw * CORSI_TARGET_ANNUALIZATION_DAYS / tenor

            annualized_total_vol = np.sqrt(annualized_total_var) * 100
            annualized_intraday_vol = np.sqrt(annualized_intraday_var) * 100
            annualized_overnight_vol_component = np.sqrt(annualized_overnight_var) * 100

            first_forward_date = window["date"].min()
            last_forward_date = window["date"].max()

            short_session_count = int(
                (window["session_length_class"] == "SHORT_SESSION_LIKELY_EARLY_CLOSE").sum()
            )

            near_full_review_count = int(
                (window["session_length_class"] == "NEAR_FULL_SESSION_MISSING_BARS").sum()
            )

            other_review_count = int(
            (
                ~window["corsi_quality_status"].isin([
                "OK_FULL_SESSION",
                "OK_SHORT_SESSION",
                ])
            ).sum()
          )

            review_session_count = int(
                window["corsi_quality_status"].str.contains("REVIEW", na=False).sum()
            )

            overnight_share_of_variance = (
                overnight_sum_raw / total_sum_raw
                if total_sum_raw > 0
                else np.nan
            )

        else:
            intraday_sum_raw = np.nan
            overnight_sum_raw = np.nan
            total_sum_raw = np.nan

            annualized_total_var = np.nan
            annualized_intraday_var = np.nan
            annualized_overnight_var = np.nan

            annualized_total_vol = np.nan
            annualized_intraday_vol = np.nan
            annualized_overnight_vol_component = np.nan

            first_forward_date = pd.NaT
            last_forward_date = pd.NaT

            short_session_count = 0
            near_full_review_count = 0
            other_review_count = 0
            review_session_count = 0
            overnight_share_of_variance = np.nan

        target_rows.append({
            "trade_date": signal_trade_date,
            "date": signal_date,
            "tenor": tenor,
            "target_days": tenor,
            "forward_start_exclusive_date": signal_date,
            "forward_end_inclusive_calendar_date": forward_end_date,
            "first_forward_rv_date": first_forward_date,
            "last_forward_rv_date": last_forward_date,
            "forward_window_complete_corsi": bool(forward_window_complete),
            "forward_num_sessions_corsi": int(forward_num_sessions),
            "forward_intraday_rv_raw_sum_corsi": intraday_sum_raw,
            "forward_overnight_rv_raw_sum_corsi": overnight_sum_raw,
            "forward_total_rv_raw_sum_corsi": total_sum_raw,
            "forward_intraday_variance_corsi": annualized_intraday_var,
            "forward_overnight_variance_corsi": annualized_overnight_var,
            "forward_realized_variance_corsi": annualized_total_var,
            "forward_intraday_vol_corsi": annualized_intraday_vol,
            "forward_overnight_vol_component_corsi": annualized_overnight_vol_component,
            "forward_realized_vol_corsi": annualized_total_vol,
            "forward_overnight_share_of_variance_corsi": overnight_share_of_variance,
            "forward_short_session_count_corsi": short_session_count,
            "forward_near_full_review_count_corsi": near_full_review_count,
            "forward_other_review_count_corsi": other_review_count,
            "forward_review_session_count_corsi": review_session_count,
            "target_source": "SPY_5m_intraday_plus_overnight",
            "target_methodology": "corsi_v1_forward_total_rv_calendar_annualized_365",
        })

corsi_forward_targets = (
    pd.DataFrame(target_rows)
    .sort_values(["date", "tenor"])
    .reset_index(drop=True)
)

# Log targets for modeling.
EPSILON_VARIANCE = 1e-12

corsi_forward_targets["log_forward_realized_variance_corsi"] = np.log(
    corsi_forward_targets["forward_realized_variance_corsi"].clip(lower=EPSILON_VARIANCE)
)

corsi_forward_targets["log_forward_intraday_variance_corsi"] = np.log(
    corsi_forward_targets["forward_intraday_variance_corsi"].clip(lower=EPSILON_VARIANCE)
)

corsi_forward_targets["log_forward_overnight_variance_corsi"] = np.log(
    corsi_forward_targets["forward_overnight_variance_corsi"].clip(lower=EPSILON_VARIANCE)
)

print()
print("Corsi forward target table built:")
target_summary = pd.DataFrame([{
    "rows": len(corsi_forward_targets),
    "first_date": corsi_forward_targets["date"].min(),
    "last_date": corsi_forward_targets["date"].max(),
    "tenor_count": corsi_forward_targets["tenor"].nunique(),
    "complete_rows": int(corsi_forward_targets["forward_window_complete_corsi"].sum()),
    "incomplete_rows": int((~corsi_forward_targets["forward_window_complete_corsi"]).sum()),
    "missing_target_variance_rows": int(corsi_forward_targets["forward_realized_variance_corsi"].isna().sum()),
}])
display(target_summary)

display(corsi_forward_targets.head(15))
display(corsi_forward_targets.tail(15))


# -----------------------------
# 5. Coverage and completeness by tenor
# -----------------------------

corsi_target_coverage_by_tenor = (
    corsi_forward_targets
    .groupby("tenor")
    .agg(
        rows=("date", "size"),
        complete_rows=("forward_window_complete_corsi", "sum"),
        incomplete_rows=("forward_window_complete_corsi", lambda x: int((~x).sum())),
        missing_variance_rows=("forward_realized_variance_corsi", lambda x: int(x.isna().sum())),
        mean_forward_sessions=("forward_num_sessions_corsi", "mean"),
        min_forward_sessions=("forward_num_sessions_corsi", "min"),
        max_forward_sessions=("forward_num_sessions_corsi", "max"),
        mean_forward_realized_vol=("forward_realized_vol_corsi", "mean"),
        p95_forward_realized_vol=("forward_realized_vol_corsi", lambda x: x.quantile(0.95)),
        max_forward_realized_vol=("forward_realized_vol_corsi", "max"),
        mean_overnight_share=("forward_overnight_share_of_variance_corsi", "mean"),
        max_review_sessions=("forward_review_session_count_corsi", "max"),
    )
    .reset_index()
)

print()
print("Corsi target coverage by tenor:")
display(corsi_target_coverage_by_tenor)


# -----------------------------
# 6. Merge with production feature panel for comparison
# -----------------------------

feature_for_compare = feature_dates.copy()

# Keep only useful comparison columns if available.
preferred_compare_cols = [
    "date",
    "trade_date",
    "tenor",
    "target_days",
    "tenor_group",
    "spx_close",
    "spx_log_return",
    "implied_variance",
    "vix_style_vol",
    "trailing_realized_variance",
    "trailing_realized_vol",
    "forecast_variance",
    "forecast_vol",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "forward_realized_variance",
    "forward_realized_vol",
    "forward_window_complete",
]

existing_compare_cols = [c for c in preferred_compare_cols if c in feature_for_compare.columns]

feature_for_compare = feature_for_compare[existing_compare_cols].copy()

# Ensure tenor is numeric.
feature_for_compare["tenor"] = pd.to_numeric(feature_for_compare["tenor"], errors="coerce").astype(int)
feature_for_compare["trade_date"] = pd.to_numeric(feature_for_compare["trade_date"], errors="coerce").astype(int)

corsi_feature_target_panel = feature_for_compare.merge(
    corsi_forward_targets,
    on=["date", "trade_date", "tenor"],
    how="left",
    validate="one_to_one",
)

print()
print("Merged feature + Corsi target panel summary:")
merged_summary = pd.DataFrame([{
    "rows": len(corsi_feature_target_panel),
    "target_rows": len(corsi_forward_targets),
    "missing_corsi_target_rows": int(corsi_feature_target_panel["forward_realized_variance_corsi"].isna().sum()),
    "complete_corsi_target_rows": int(corsi_feature_target_panel["forward_window_complete_corsi"].fillna(False).sum()),
}])
display(merged_summary)


# -----------------------------
# 7. Compare old SPX close-to-close forward RV vs new SPY intraday+overnight forward RV
# -----------------------------

compare_rows = []

if "forward_realized_variance" in corsi_feature_target_panel.columns:
    compare_df = corsi_feature_target_panel.copy()

    compare_df["old_forward_complete"] = (
        compare_df["forward_window_complete"].fillna(False).astype(bool)
        if "forward_window_complete" in compare_df.columns
        else compare_df["forward_realized_variance"].notna()
    )

    compare_df["new_forward_complete"] = (
        compare_df["forward_window_complete_corsi"].fillna(False).astype(bool)
    )

    compare_df["both_complete"] = (
        compare_df["old_forward_complete"]
        & compare_df["new_forward_complete"]
        & compare_df["forward_realized_variance"].notna()
        & compare_df["forward_realized_variance_corsi"].notna()
        & (compare_df["forward_realized_variance"] > 0)
        & (compare_df["forward_realized_variance_corsi"] > 0)
    )

    compare_df["log_old_forward_rv"] = np.log(compare_df["forward_realized_variance"])
    compare_df["log_corsi_forward_rv"] = np.log(compare_df["forward_realized_variance_corsi"])

    compare_df["corsi_minus_old_forward_variance"] = (
        compare_df["forward_realized_variance_corsi"]
        - compare_df["forward_realized_variance"]
    )

    compare_df["corsi_over_old_forward_variance_ratio"] = (
        compare_df["forward_realized_variance_corsi"]
        / compare_df["forward_realized_variance"]
    )

    compare_df["log_corsi_minus_log_old_forward_rv"] = (
        compare_df["log_corsi_forward_rv"]
        - compare_df["log_old_forward_rv"]
    )

    for tenor, g in compare_df.loc[compare_df["both_complete"]].groupby("tenor"):
        if len(g) >= 10:
            corr = g["forward_realized_variance"].corr(g["forward_realized_variance_corsi"])
            log_corr = g["log_old_forward_rv"].corr(g["log_corsi_forward_rv"])
        else:
            corr = np.nan
            log_corr = np.nan

        compare_rows.append({
            "tenor": int(tenor),
            "rows": int(len(g)),
            "old_mean_forward_vol": g["forward_realized_vol"].mean()
                if "forward_realized_vol" in g.columns else np.nan,
            "corsi_mean_forward_vol": g["forward_realized_vol_corsi"].mean(),
            "old_median_forward_vol": g["forward_realized_vol"].median()
                if "forward_realized_vol" in g.columns else np.nan,
            "corsi_median_forward_vol": g["forward_realized_vol_corsi"].median(),
            "mean_corsi_over_old_variance_ratio": g["corsi_over_old_forward_variance_ratio"].mean(),
            "median_corsi_over_old_variance_ratio": g["corsi_over_old_forward_variance_ratio"].median(),
            "mean_log_corsi_minus_log_old": g["log_corsi_minus_log_old_forward_rv"].mean(),
            "corr_variance": corr,
            "corr_log_variance": log_corr,
        })

    old_vs_corsi_forward_compare_by_tenor = pd.DataFrame(compare_rows)

    print()
    print("Old SPX close-to-close forward RV vs new SPY intraday+overnight forward RV:")
    display(old_vs_corsi_forward_compare_by_tenor)

    print()
    print("Largest absolute old-vs-Corsi forward target differences:")
    largest_old_new_diffs = (
        compare_df
        .loc[compare_df["both_complete"]]
        .assign(abs_log_diff=lambda x: x["log_corsi_minus_log_old_forward_rv"].abs())
        .sort_values("abs_log_diff", ascending=False)
        .head(25)
        .copy()
    )

    display(largest_old_new_diffs[
        [
            "trade_date",
            "date",
            "tenor",
            "forward_realized_vol",
            "forward_realized_vol_corsi",
            "forward_realized_variance",
            "forward_realized_variance_corsi",
            "corsi_over_old_forward_variance_ratio",
            "log_corsi_minus_log_old_forward_rv",
            "forward_num_sessions_corsi",
            "forward_review_session_count_corsi",
        ]
    ])

else:
    old_vs_corsi_forward_compare_by_tenor = pd.DataFrame()
    largest_old_new_diffs = pd.DataFrame()
    print()
    print("Production panel does not contain old forward_realized_variance; skipping old-vs-Corsi comparison.")


# -----------------------------
# 8. Save Cell 4 outputs
# -----------------------------

safe_start = corsi_forward_targets["date"].min().strftime("%Y%m%d")
safe_end = corsi_forward_targets["date"].max().strftime("%Y%m%d")

cell4_targets_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_forward_realized_variance_targets_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell4_targets_csv_sample_output = (
    CORSI_AUDIT_DIR
    / f"corsi_forward_realized_variance_targets_v1_sample_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell4_feature_target_panel_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_feature_target_panel_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell4_target_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell4_target_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell4_coverage_by_tenor_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell4_target_coverage_by_tenor_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell4_merged_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell4_merged_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell4_old_vs_corsi_compare_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell4_old_vs_corsi_forward_compare_by_tenor_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell4_largest_old_new_diffs_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell4_largest_old_new_forward_target_diffs_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

corsi_forward_targets.to_parquet(cell4_targets_output, index=False)
corsi_forward_targets.head(250).to_csv(cell4_targets_csv_sample_output, index=False)
corsi_feature_target_panel.to_parquet(cell4_feature_target_panel_output, index=False)

target_summary.to_csv(cell4_target_summary_output, index=False)
corsi_target_coverage_by_tenor.to_csv(cell4_coverage_by_tenor_output, index=False)
merged_summary.to_csv(cell4_merged_summary_output, index=False)

if not old_vs_corsi_forward_compare_by_tenor.empty:
    old_vs_corsi_forward_compare_by_tenor.to_csv(cell4_old_vs_corsi_compare_output, index=False)

if not largest_old_new_diffs.empty:
    largest_old_new_diffs.to_csv(cell4_largest_old_new_diffs_output, index=False)

print()
print("Cell 4 outputs saved:")
print(f"Corsi forward targets:             {cell4_targets_output}")
print(f"Corsi target CSV sample:           {cell4_targets_csv_sample_output}")
print(f"Feature + target panel:            {cell4_feature_target_panel_output}")
print(f"Target summary:                    {cell4_target_summary_output}")
print(f"Coverage by tenor:                 {cell4_coverage_by_tenor_output}")
print(f"Merged summary:                    {cell4_merged_summary_output}")

if not old_vs_corsi_forward_compare_by_tenor.empty:
    print(f"Old-vs-Corsi compare:              {cell4_old_vs_corsi_compare_output}")

if not largest_old_new_diffs.empty:
    print(f"Largest old-new target diffs:      {cell4_largest_old_new_diffs_output}")

print()
print("Cell 4 complete.")
print("Next step will be Cell 5: build HAR/Corsi predictor features from SPY daily RV and the production feature panel.")

## 5. Build HAR/Corsi predictor features

Constructs realized-variance state variables, rolling means, EWMA measures, return/drawdown state, tenor features, and implied-variance features. These become the candidate model inputs.

In [ ]:
# Cell 5: Build HAR/Corsi predictor features
#
# Purpose:
# Build the modeling dataset for the Corsi / HAR-RV forecast.
#
# We now have:
#   1. SPY daily realized variance table from intraday + overnight data.
#   2. Corsi forward realized-variance targets by date x tenor.
#   3. Existing production feature panel with implied variance, tenor, RSI, etc.
#
# This cell creates predictor features:
#   - HAR realized-variance features: 1d, 5d, 21d, 63d, 126d
#   - EWMA realized-variance states
#   - overnight variance features
#   - downside/upside realized-variance features
#   - return and drawdown features
#   - implied-variance curve features
#   - tenor features
#
# This cell does NOT fit the model yet.

import json


# -----------------------------
# 1. Load objects if needed
# -----------------------------

try:
    spy_daily_rv_model_ready
except NameError:
    latest_model_ready_files = sorted(
        CORSI_PROCESSED_DIR.glob("spy_daily_realized_variance_corsi_v1_model_ready_*.parquet")
    )
    if not latest_model_ready_files:
        raise FileNotFoundError("No model-ready SPY daily RV file found.")
    print(f"Loading SPY model-ready RV: {latest_model_ready_files[-1]}")
    spy_daily_rv_model_ready = pd.read_parquet(latest_model_ready_files[-1])

try:
    corsi_feature_target_panel
except NameError:
    latest_feature_target_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_feature_target_panel_v1_*.parquet")
    )
    if not latest_feature_target_files:
        raise FileNotFoundError("No Corsi feature target panel file found.")
    print(f"Loading Corsi feature target panel: {latest_feature_target_files[-1]}")
    corsi_feature_target_panel = pd.read_parquet(latest_feature_target_files[-1])

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"


# -----------------------------
# 2. Configuration
# -----------------------------

EPSILON_VARIANCE = 1e-12

ROLLING_WINDOWS = [3, 5, 10, 21, 42, 63, 126, 252]
EWMA_HALFLIVES = [3, 5, 10, 21, 42, 63]

ANCHOR_TENORS = [9, 21, 33]

print("Cell 5 predictor feature configuration:")
print(f"Rolling windows:     {ROLLING_WINDOWS}")
print(f"EWMA halflives:      {EWMA_HALFLIVES}")
print(f"Anchor tenors:       {ANCHOR_TENORS}")
print(f"SPY venue:           {SPY_STOCK_VENUE}")


# -----------------------------
# 3. Prepare SPY daily feature table
# -----------------------------

spy_daily = spy_daily_rv_model_ready.copy()
spy_daily["date"] = pd.to_datetime(spy_daily["date"])
spy_daily["trade_date"] = pd.to_numeric(spy_daily["trade_date"], errors="coerce").astype(int)

spy_daily = spy_daily.sort_values("date").reset_index(drop=True)

# Ensure required columns exist.
required_daily_cols = [
    "trade_date",
    "date",
    "spy_intraday_rv_raw",
    "spy_overnight_rv_raw",
    "spy_total_rv_raw",
    "spy_intraday_vol_ann",
    "spy_total_vol_ann",
    "spy_overnight_return",
    "intraday_open_to_close_return",
    "eod_close_sanitized",
    "session_length_class",
    "corsi_quality_status",
]

missing_daily_cols = [c for c in required_daily_cols if c not in spy_daily.columns]
if missing_daily_cols:
    raise ValueError(f"Missing required SPY daily columns: {missing_daily_cols}")

# Clean numeric columns.
for c in [
    "spy_intraday_rv_raw",
    "spy_overnight_rv_raw",
    "spy_total_rv_raw",
    "spy_intraday_vol_ann",
    "spy_total_vol_ann",
    "spy_overnight_return",
    "intraday_open_to_close_return",
    "eod_close_sanitized",
]:
    spy_daily[c] = pd.to_numeric(spy_daily[c], errors="coerce")

spy_daily["spy_overnight_return"] = spy_daily["spy_overnight_return"].fillna(0.0)
spy_daily["spy_overnight_rv_raw"] = spy_daily["spy_overnight_rv_raw"].fillna(0.0)

spy_daily["spy_total_return"] = (
    spy_daily["spy_overnight_return"]
    + spy_daily["intraday_open_to_close_return"].fillna(0.0)
)

spy_daily["spy_abs_total_return"] = spy_daily["spy_total_return"].abs()
spy_daily["spy_negative_return"] = spy_daily["spy_total_return"].clip(upper=0.0)
spy_daily["spy_positive_return"] = spy_daily["spy_total_return"].clip(lower=0.0)

spy_daily["spy_downside_rv_raw"] = np.where(
    spy_daily["spy_total_return"] < 0,
    spy_daily["spy_total_rv_raw"],
    0.0,
)

spy_daily["spy_upside_rv_raw"] = np.where(
    spy_daily["spy_total_return"] > 0,
    spy_daily["spy_total_rv_raw"],
    0.0,
)

spy_daily["spy_total_rv_1d_ann"] = spy_daily["spy_total_rv_raw"] * 252
spy_daily["spy_intraday_rv_1d_ann"] = spy_daily["spy_intraday_rv_raw"] * 252
spy_daily["spy_overnight_rv_1d_ann"] = spy_daily["spy_overnight_rv_raw"] * 252
spy_daily["spy_downside_rv_1d_ann"] = spy_daily["spy_downside_rv_raw"] * 252
spy_daily["spy_upside_rv_1d_ann"] = spy_daily["spy_upside_rv_raw"] * 252

spy_daily["log_spy_total_rv_1d_ann"] = np.log(
    spy_daily["spy_total_rv_1d_ann"].clip(lower=EPSILON_VARIANCE)
)

spy_daily["log_spy_intraday_rv_1d_ann"] = np.log(
    spy_daily["spy_intraday_rv_1d_ann"].clip(lower=EPSILON_VARIANCE)
)

spy_daily["log_spy_overnight_rv_1d_ann"] = np.log(
    spy_daily["spy_overnight_rv_1d_ann"].clip(lower=EPSILON_VARIANCE)
)

spy_daily["spy_overnight_share_1d"] = np.where(
    spy_daily["spy_total_rv_raw"] > 0,
    spy_daily["spy_overnight_rv_raw"] / spy_daily["spy_total_rv_raw"],
    np.nan,
)


# -----------------------------
# 4. Rolling HAR features
# -----------------------------

def rolling_min_periods(window):
    return max(2, int(np.ceil(window / 3)))


def add_rolling_variance_features(df, raw_col, prefix, windows):
    out = df.copy()

    for w in windows:
        min_p = rolling_min_periods(w)

        mean_raw = out[raw_col].rolling(w, min_periods=min_p).mean()
        sum_raw = out[raw_col].rolling(w, min_periods=min_p).sum()

        ann_mean_col = f"{prefix}_rv_mean_{w}d_ann"
        ann_sum_col = f"{prefix}_rv_sum_{w}d_raw"
        log_col = f"log_{prefix}_rv_mean_{w}d_ann"

        out[ann_mean_col] = mean_raw * 252
        out[ann_sum_col] = sum_raw
        out[log_col] = np.log(out[ann_mean_col].clip(lower=EPSILON_VARIANCE))

    return out


spy_daily = add_rolling_variance_features(
    spy_daily,
    "spy_total_rv_raw",
    "spy_total",
    ROLLING_WINDOWS,
)

spy_daily = add_rolling_variance_features(
    spy_daily,
    "spy_intraday_rv_raw",
    "spy_intraday",
    ROLLING_WINDOWS,
)

spy_daily = add_rolling_variance_features(
    spy_daily,
    "spy_overnight_rv_raw",
    "spy_overnight",
    ROLLING_WINDOWS,
)

spy_daily = add_rolling_variance_features(
    spy_daily,
    "spy_downside_rv_raw",
    "spy_downside",
    ROLLING_WINDOWS,
)

spy_daily = add_rolling_variance_features(
    spy_daily,
    "spy_upside_rv_raw",
    "spy_upside",
    ROLLING_WINDOWS,
)

# Rolling returns and overnight share.
for w in ROLLING_WINDOWS:
    min_p = rolling_min_periods(w)

    spy_daily[f"spy_return_sum_{w}d"] = (
        spy_daily["spy_total_return"]
        .rolling(w, min_periods=min_p)
        .sum()
    )

    spy_daily[f"spy_abs_return_mean_{w}d"] = (
        spy_daily["spy_abs_total_return"]
        .rolling(w, min_periods=min_p)
        .mean()
    )

    spy_daily[f"spy_negative_return_sum_{w}d"] = (
        spy_daily["spy_negative_return"]
        .rolling(w, min_periods=min_p)
        .sum()
    )

    spy_daily[f"spy_positive_return_sum_{w}d"] = (
        spy_daily["spy_positive_return"]
        .rolling(w, min_periods=min_p)
        .sum()
    )

    total_rv_sum = (
        spy_daily["spy_total_rv_raw"]
        .rolling(w, min_periods=min_p)
        .sum()
    )

    overnight_rv_sum = (
        spy_daily["spy_overnight_rv_raw"]
        .rolling(w, min_periods=min_p)
        .sum()
    )

    downside_rv_sum = (
        spy_daily["spy_downside_rv_raw"]
        .rolling(w, min_periods=min_p)
        .sum()
    )

    spy_daily[f"spy_overnight_share_{w}d"] = np.where(
        total_rv_sum > 0,
        overnight_rv_sum / total_rv_sum,
        np.nan,
    )

    spy_daily[f"spy_downside_share_{w}d"] = np.where(
        total_rv_sum > 0,
        downside_rv_sum / total_rv_sum,
        np.nan,
    )

    spy_daily[f"spy_log_total_rv_vol_of_vol_{w}d"] = (
        spy_daily["log_spy_total_rv_1d_ann"]
        .rolling(w, min_periods=min_p)
        .std()
    )


# -----------------------------
# 5. EWMA variance-state features
# -----------------------------

for hl in EWMA_HALFLIVES:
    min_p = rolling_min_periods(hl)

    spy_daily[f"spy_total_rv_ewma_hl{hl}_ann"] = (
        spy_daily["spy_total_rv_raw"]
        .ewm(halflife=hl, min_periods=min_p, adjust=False)
        .mean()
        * 252
    )

    spy_daily[f"spy_intraday_rv_ewma_hl{hl}_ann"] = (
        spy_daily["spy_intraday_rv_raw"]
        .ewm(halflife=hl, min_periods=min_p, adjust=False)
        .mean()
        * 252
    )

    spy_daily[f"spy_overnight_rv_ewma_hl{hl}_ann"] = (
        spy_daily["spy_overnight_rv_raw"]
        .ewm(halflife=hl, min_periods=min_p, adjust=False)
        .mean()
        * 252
    )

    spy_daily[f"log_spy_total_rv_ewma_hl{hl}_ann"] = np.log(
        spy_daily[f"spy_total_rv_ewma_hl{hl}_ann"].clip(lower=EPSILON_VARIANCE)
    )

    spy_daily[f"log_spy_intraday_rv_ewma_hl{hl}_ann"] = np.log(
        spy_daily[f"spy_intraday_rv_ewma_hl{hl}_ann"].clip(lower=EPSILON_VARIANCE)
    )

    spy_daily[f"log_spy_overnight_rv_ewma_hl{hl}_ann"] = np.log(
        spy_daily[f"spy_overnight_rv_ewma_hl{hl}_ann"].clip(lower=EPSILON_VARIANCE)
    )


# -----------------------------
# 6. Drawdown / trend features
# -----------------------------

spy_daily["spy_eod_close"] = spy_daily["eod_close_sanitized"]

for w in [21, 42, 63, 126, 252]:
    min_p = rolling_min_periods(w)

    rolling_max_close = (
        spy_daily["spy_eod_close"]
        .rolling(w, min_periods=min_p)
        .max()
    )

    spy_daily[f"spy_drawdown_{w}d"] = (
        spy_daily["spy_eod_close"] / rolling_max_close
        - 1.0
    )

    spy_daily[f"spy_return_{w}d"] = (
        np.log(spy_daily["spy_eod_close"] / spy_daily["spy_eod_close"].shift(w))
    )


# -----------------------------
# 7. Session quality features
# -----------------------------

spy_daily["is_full_session"] = (
    spy_daily["session_length_class"] == "FULL_78_BARS"
).astype(int)

spy_daily["is_short_session"] = (
    spy_daily["session_length_class"] == "SHORT_SESSION_LIKELY_EARLY_CLOSE"
).astype(int)

spy_daily["is_review_session"] = (
    ~spy_daily["corsi_quality_status"].isin([
        "OK_FULL_SESSION",
        "OK_SHORT_SESSION",
    ])
).astype(int)

spy_daily_feature_cols_base = [
    "trade_date",
    "date",
    "spy_total_return",
    "spy_abs_total_return",
    "spy_negative_return",
    "spy_positive_return",
    "spy_total_rv_1d_ann",
    "spy_intraday_rv_1d_ann",
    "spy_overnight_rv_1d_ann",
    "log_spy_total_rv_1d_ann",
    "log_spy_intraday_rv_1d_ann",
    "log_spy_overnight_rv_1d_ann",
    "spy_overnight_share_1d",
    "is_full_session",
    "is_short_session",
    "is_review_session",
]

spy_daily_predictor_features = spy_daily.copy()


# -----------------------------
# 8. Build implied-curve features from production panel
# -----------------------------

panel = corsi_feature_target_panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

panel = panel.sort_values(["date", "tenor"]).reset_index(drop=True)

for c in ["implied_variance", "vix_style_vol", "forecast_variance", "trailing_realized_variance"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

if "implied_variance" not in panel.columns:
    raise ValueError("corsi_feature_target_panel must contain implied_variance.")

iv_pivot = (
    panel
    .pivot_table(
        index=["date", "trade_date"],
        columns="tenor",
        values="implied_variance",
        aggfunc="first",
    )
    .sort_index()
)

vol_pivot = (
    panel
    .pivot_table(
        index=["date", "trade_date"],
        columns="tenor",
        values="vix_style_vol",
        aggfunc="first",
    )
    .sort_index()
    if "vix_style_vol" in panel.columns
    else None
)

curve_features = iv_pivot.copy()
curve_features.columns = [f"iv_{int(c)}d" for c in curve_features.columns]
curve_features = curve_features.reset_index()

# Anchor features.
for t in ANCHOR_TENORS:
    col = f"iv_{t}d"
    if col in curve_features.columns:
        curve_features[f"log_{col}"] = np.log(
            curve_features[col].clip(lower=EPSILON_VARIANCE)
        )

available_iv_cols = [c for c in curve_features.columns if c.startswith("iv_") and c.endswith("d")]

curve_features["iv_curve_mean"] = curve_features[available_iv_cols].mean(axis=1)
curve_features["iv_curve_min"] = curve_features[available_iv_cols].min(axis=1)
curve_features["iv_curve_max"] = curve_features[available_iv_cols].max(axis=1)
curve_features["iv_curve_range"] = curve_features["iv_curve_max"] - curve_features["iv_curve_min"]
curve_features["log_iv_curve_mean"] = np.log(
    curve_features["iv_curve_mean"].clip(lower=EPSILON_VARIANCE)
)

if "iv_9d" in curve_features.columns and "iv_33d" in curve_features.columns:
    curve_features["iv_slope_33d_9d"] = curve_features["iv_33d"] - curve_features["iv_9d"]
    curve_features["iv_ratio_33d_9d"] = curve_features["iv_33d"] / curve_features["iv_9d"]
    curve_features["log_iv_ratio_33d_9d"] = np.log(
        curve_features["iv_ratio_33d_9d"].clip(lower=EPSILON_VARIANCE)
    )

if "iv_9d" in curve_features.columns and "iv_21d" in curve_features.columns:
    curve_features["iv_slope_21d_9d"] = curve_features["iv_21d"] - curve_features["iv_9d"]
    curve_features["iv_ratio_21d_9d"] = curve_features["iv_21d"] / curve_features["iv_9d"]
    curve_features["log_iv_ratio_21d_9d"] = np.log(
        curve_features["iv_ratio_21d_9d"].clip(lower=EPSILON_VARIANCE)
    )

if all(c in curve_features.columns for c in ["iv_9d", "iv_21d", "iv_33d"]):
    curve_features["iv_curvature_9_21_33"] = (
        curve_features["iv_9d"]
        - 2.0 * curve_features["iv_21d"]
        + curve_features["iv_33d"]
    )

if vol_pivot is not None:
    vol_features = vol_pivot.copy()
    vol_features.columns = [f"ivol_{int(c)}d" for c in vol_features.columns]
    vol_features = vol_features.reset_index()

    curve_features = curve_features.merge(
        vol_features,
        on=["date", "trade_date"],
        how="left",
        validate="one_to_one",
    )

    if "ivol_9d" in curve_features.columns and "ivol_33d" in curve_features.columns:
        curve_features["ivol_slope_33d_9d"] = (
            curve_features["ivol_33d"] - curve_features["ivol_9d"]
        )

    if all(c in curve_features.columns for c in ["ivol_9d", "ivol_21d", "ivol_33d"]):
        curve_features["ivol_curvature_9_21_33"] = (
            curve_features["ivol_9d"]
            - 2.0 * curve_features["ivol_21d"]
            + curve_features["ivol_33d"]
        )


# -----------------------------
# 9. Merge SPY daily features and curve features into date x tenor panel
# -----------------------------

# Keep all SPY columns except duplicate source columns that are unwieldy.
spy_feature_drop_cols = [
    "first_timestamp",
    "last_timestamp",
    "created",
    "last_trade",
]

spy_feature_for_merge = spy_daily_predictor_features.drop(
    columns=[c for c in spy_feature_drop_cols if c in spy_daily_predictor_features.columns],
    errors="ignore",
).copy()

model_panel = panel.merge(
    spy_feature_for_merge,
    on=["date", "trade_date"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_spy_daily"),
)

model_panel = model_panel.merge(
    curve_features,
    on=["date", "trade_date"],
    how="left",
    validate="many_to_one",
)

# Tenor features.
model_panel["tenor"] = pd.to_numeric(model_panel["tenor"], errors="coerce").astype(int)
model_panel["tenor_scaled_30d"] = model_panel["tenor"] / 30.0
model_panel["log_tenor"] = np.log(model_panel["tenor"])

if "tenor_group" in model_panel.columns:
    tenor_dummies = pd.get_dummies(
        model_panel["tenor_group"].astype(str),
        prefix="tenor_group",
        dtype=int,
    )

    model_panel = pd.concat([model_panel, tenor_dummies], axis=1)

# Row-level implied feature.
model_panel["log_implied_variance"] = np.log(
    model_panel["implied_variance"].clip(lower=EPSILON_VARIANCE)
)

if "vix_style_vol" in model_panel.columns:
    model_panel["log_vix_style_vol"] = np.log(
        pd.to_numeric(model_panel["vix_style_vol"], errors="coerce").clip(lower=EPSILON_VARIANCE)
    )

# Calendar features.
model_panel["month"] = model_panel["date"].dt.month
model_panel["year"] = model_panel["date"].dt.year
model_panel["quarter"] = model_panel["date"].dt.quarter


# -----------------------------
# 10. Define feature sets
# -----------------------------

realized_only_features = [
    "log_spy_total_rv_1d_ann",
    "log_spy_intraday_rv_1d_ann",
    "log_spy_overnight_rv_1d_ann",
    "spy_total_return",
    "spy_abs_total_return",
    "spy_negative_return",
    "spy_positive_return",
    "spy_overnight_share_1d",
    "is_full_session",
    "is_short_session",
    "is_review_session",
]

for w in ROLLING_WINDOWS:
    realized_only_features += [
        f"log_spy_total_rv_mean_{w}d_ann",
        f"log_spy_intraday_rv_mean_{w}d_ann",
        f"log_spy_overnight_rv_mean_{w}d_ann",
        f"log_spy_downside_rv_mean_{w}d_ann",
        f"log_spy_upside_rv_mean_{w}d_ann",
        f"spy_return_sum_{w}d",
        f"spy_abs_return_mean_{w}d",
        f"spy_negative_return_sum_{w}d",
        f"spy_positive_return_sum_{w}d",
        f"spy_overnight_share_{w}d",
        f"spy_downside_share_{w}d",
        f"spy_log_total_rv_vol_of_vol_{w}d",
    ]

for hl in EWMA_HALFLIVES:
    realized_only_features += [
        f"log_spy_total_rv_ewma_hl{hl}_ann",
        f"log_spy_intraday_rv_ewma_hl{hl}_ann",
        f"log_spy_overnight_rv_ewma_hl{hl}_ann",
    ]

for w in [21, 42, 63, 126, 252]:
    realized_only_features += [
        f"spy_drawdown_{w}d",
        f"spy_return_{w}d",
    ]

tenor_features = [
    "tenor_scaled_30d",
    "log_tenor",
]

tenor_features += [
    c for c in model_panel.columns
    if c.startswith("tenor_group_")
]

implied_features = [
    "log_implied_variance",
    "log_iv_curve_mean",
    "iv_curve_range",
]

for t in ANCHOR_TENORS:
    if f"log_iv_{t}d" in model_panel.columns:
        implied_features.append(f"log_iv_{t}d")

for c in [
    "iv_slope_33d_9d",
    "iv_slope_21d_9d",
    "log_iv_ratio_33d_9d",
    "log_iv_ratio_21d_9d",
    "iv_curvature_9_21_33",
    "ivol_slope_33d_9d",
    "ivol_curvature_9_21_33",
]:
    if c in model_panel.columns:
        implied_features.append(c)

production_state_features = []

for c in [
    "spx_log_return",
    "rv21d",
    "rsi14",
    "spx_rsi_14",
    "trailing_realized_variance",
    "trailing_realized_vol",
]:
    if c in model_panel.columns:
        production_state_features.append(c)

# Keep only features that actually exist.
realized_only_features = [c for c in realized_only_features if c in model_panel.columns]
tenor_features = [c for c in tenor_features if c in model_panel.columns]
implied_features = [c for c in implied_features if c in model_panel.columns]
production_state_features = [c for c in production_state_features if c in model_panel.columns]

feature_sets = {
    "realized_only": realized_only_features + tenor_features,
    "realized_plus_implied": realized_only_features + tenor_features + implied_features,
    "realized_plus_implied_plus_state": (
        realized_only_features
        + tenor_features
        + implied_features
        + production_state_features
    ),
}

# Deduplicate while preserving order.
for k, cols in feature_sets.items():
    seen = set()
    deduped = []
    for c in cols:
        if c not in seen:
            deduped.append(c)
            seen.add(c)
    feature_sets[k] = deduped


# -----------------------------
# 11. Modeling-row flags
# -----------------------------

TARGET_COL = "log_forward_realized_variance_corsi"
TARGET_VAR_COL = "forward_realized_variance_corsi"

model_panel["has_complete_corsi_target"] = (
    model_panel["forward_window_complete_corsi"].fillna(False).astype(bool)
    & model_panel[TARGET_COL].notna()
    & model_panel[TARGET_VAR_COL].notna()
    & (model_panel[TARGET_VAR_COL] > 0)
)

for feature_set_name, cols in feature_sets.items():
    model_panel[f"{feature_set_name}_feature_complete"] = (
        model_panel[cols]
        .replace([np.inf, -np.inf], np.nan)
        .notna()
        .all(axis=1)
    )

    model_panel[f"{feature_set_name}_model_row"] = (
        model_panel["has_complete_corsi_target"]
        & model_panel[f"{feature_set_name}_feature_complete"]
    )


# -----------------------------
# 12. Diagnostics
# -----------------------------

model_panel_summary_rows = []

for feature_set_name, cols in feature_sets.items():
    model_panel_summary_rows.append({
        "feature_set": feature_set_name,
        "feature_count": len(cols),
        "rows": len(model_panel),
        "complete_target_rows": int(model_panel["has_complete_corsi_target"].sum()),
        "feature_complete_rows": int(model_panel[f"{feature_set_name}_feature_complete"].sum()),
        "model_rows": int(model_panel[f"{feature_set_name}_model_row"].sum()),
        "first_model_date": model_panel.loc[model_panel[f"{feature_set_name}_model_row"], "date"].min(),
        "last_model_date": model_panel.loc[model_panel[f"{feature_set_name}_model_row"], "date"].max(),
    })

model_panel_summary = pd.DataFrame(model_panel_summary_rows)

print()
print("Corsi model panel summary:")
display(model_panel_summary)

feature_count_rows = []

for feature_set_name, cols in feature_sets.items():
    missing_counts = (
        model_panel[cols]
        .replace([np.inf, -np.inf], np.nan)
        .isna()
        .sum()
        .sort_values(ascending=False)
    )

    for feature_name, missing_count in missing_counts.items():
        if missing_count > 0:
            feature_count_rows.append({
                "feature_set": feature_set_name,
                "feature_name": feature_name,
                "missing_rows": int(missing_count),
                "missing_pct": float(missing_count / len(model_panel)),
            })

feature_missing_summary = pd.DataFrame(feature_count_rows)

print()
print("Feature missing summary, top 40:")
if feature_missing_summary.empty:
    print("No missing feature rows.")
else:
    display(feature_missing_summary.head(40))

model_rows_by_year = []

for feature_set_name in feature_sets.keys():
    tmp = (
        model_panel
        .assign(year=model_panel["date"].dt.year)
        .groupby("year")
        .agg(
            rows=("date", "size"),
            complete_target_rows=("has_complete_corsi_target", "sum"),
            model_rows=(f"{feature_set_name}_model_row", "sum"),
        )
        .reset_index()
    )
    tmp["feature_set"] = feature_set_name
    model_rows_by_year.append(tmp)

model_rows_by_year = pd.concat(model_rows_by_year, ignore_index=True)

print()
print("Model rows by year:")
display(model_rows_by_year)

print()
print("Latest model panel rows:")
latest_display_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_group",
    "implied_variance",
    "vix_style_vol",
    "forward_realized_variance_corsi",
    "forward_realized_vol_corsi",
    "forward_window_complete_corsi",
    "log_spy_total_rv_1d_ann",
    "log_spy_total_rv_mean_21d_ann",
    "log_spy_total_rv_mean_63d_ann",
    "log_implied_variance",
    "realized_only_model_row",
    "realized_plus_implied_model_row",
]

latest_display_cols = [c for c in latest_display_cols if c in model_panel.columns]
display(model_panel[latest_display_cols].tail(20))


# -----------------------------
# 13. Save outputs
# -----------------------------

safe_start = model_panel["date"].min().strftime("%Y%m%d")
safe_end = model_panel["date"].max().strftime("%Y%m%d")

cell5_model_panel_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_model_feature_panel_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell5_model_panel_csv_sample_output = (
    CORSI_AUDIT_DIR
    / f"corsi_model_feature_panel_v1_sample_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell5_feature_sets_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell5_feature_sets_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.json"
)

cell5_feature_list_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell5_feature_list_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell5_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell5_model_panel_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell5_missing_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell5_feature_missing_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell5_model_rows_by_year_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell5_model_rows_by_year_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

model_panel.to_parquet(cell5_model_panel_output, index=False)
model_panel.head(500).to_csv(cell5_model_panel_csv_sample_output, index=False)

with open(cell5_feature_sets_output, "w") as f:
    json.dump(feature_sets, f, indent=2)

feature_list_rows = []
for feature_set_name, cols in feature_sets.items():
    for c in cols:
        feature_list_rows.append({
            "feature_set": feature_set_name,
            "feature_name": c,
        })

feature_list_df = pd.DataFrame(feature_list_rows)

feature_list_df.to_csv(cell5_feature_list_output, index=False)
model_panel_summary.to_csv(cell5_summary_output, index=False)
feature_missing_summary.to_csv(cell5_missing_summary_output, index=False)
model_rows_by_year.to_csv(cell5_model_rows_by_year_output, index=False)

print()
print("Cell 5 outputs saved:")
print(f"Corsi model feature panel:       {cell5_model_panel_output}")
print(f"Corsi model panel CSV sample:    {cell5_model_panel_csv_sample_output}")
print(f"Feature sets JSON:               {cell5_feature_sets_output}")
print(f"Feature list CSV:                {cell5_feature_list_output}")
print(f"Model panel summary:             {cell5_summary_output}")
print(f"Feature missing summary:         {cell5_missing_summary_output}")
print(f"Model rows by year:              {cell5_model_rows_by_year_output}")

print()
print("Cell 5 complete.")
print("Next step will be Cell 6: fit first direct HAR/Corsi forecast models and compare realized-only vs realized+implied.")

## 6A. Environment dependency check

Installs or verifies `scikit-learn` in the active notebook environment.

In [ ]:
# Cell 6A: Install / verify sklearn dependency
#
# Run this once if Cell 6 fails with:
# ModuleNotFoundError: No module named 'sklearn'

import sys
import subprocess

try:
    import sklearn
    print(f"scikit-learn already installed: {sklearn.__version__}")
except ModuleNotFoundError:
    print("Installing scikit-learn into current notebook Python environment...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "scikit-learn",
    ])

    import sklearn
    print(f"scikit-learn installed: {sklearn.__version__}")

## 6. First wide direct HAR/Corsi model attempt - rejected

Fits a wider ridge-regression feature set as an initial diagnostic. This attempt produced unstable forecasts and is retained only as research history. It should not be used for model selection.

In [ ]:
# Cell 6: Fit first direct HAR/Corsi forecast models
#
# Purpose:
# Fit first-pass direct forecast models for:
#
#   y = log(forward realized variance over tenor T)
#
# using the feature panel from Cell 5.
#
# Models:
#   1. realized_only
#   2. realized_plus_implied
#   3. realized_plus_implied_plus_state
#
# Method:
#   - pooled direct model across all tenors
#   - Ridge regression with standardized features
#   - walk-forward yearly test
#   - 33-calendar-day embargo before each test year
#
# Outputs:
#   - prediction panel
#   - model metrics by feature set, year, and tenor
#   - comparison versus current production forecast_variance baseline
#
# This cell does NOT replace the production forecast yet.

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib


# -----------------------------
# 1. Load model panel / feature sets if needed
# -----------------------------

try:
    model_panel
except NameError:
    latest_model_panel_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_model_feature_panel_v1_*.parquet")
    )
    if not latest_model_panel_files:
        raise FileNotFoundError("No Corsi model feature panel found.")

    print(f"Loading Corsi model feature panel: {latest_model_panel_files[-1]}")
    model_panel = pd.read_parquet(latest_model_panel_files[-1])

try:
    feature_sets
except NameError:
    latest_feature_set_files = sorted(
        CORSI_AUDIT_DIR.glob("corsi_cell5_feature_sets_*.json")
    )
    if not latest_feature_set_files:
        raise FileNotFoundError("No Cell 5 feature_sets JSON found.")

    print(f"Loading feature sets: {latest_feature_set_files[-1]}")
    with open(latest_feature_set_files[-1], "r") as f:
        feature_sets = json.load(f)

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"


# -----------------------------
# 2. Configuration
# -----------------------------

TARGET_LOG_COL = "log_forward_realized_variance_corsi"
TARGET_VAR_COL = "forward_realized_variance_corsi"
TARGET_VOL_COL = "forward_realized_vol_corsi"

TEST_YEARS = list(range(2020, 2027))

EMBARGO_DAYS = 33

RIDGE_ALPHAS = np.logspace(-4, 4, 41)

MIN_TRAIN_ROWS = 1000

MODEL_FEATURE_SETS_TO_RUN = [
    "realized_only",
    "realized_plus_implied",
    "realized_plus_implied_plus_state",
]

print("Cell 6 model configuration:")
print(f"Target column:          {TARGET_LOG_COL}")
print(f"Test years:             {TEST_YEARS}")
print(f"Embargo days:           {EMBARGO_DAYS}")
print(f"Ridge alphas:           {RIDGE_ALPHAS[0]} to {RIDGE_ALPHAS[-1]} ({len(RIDGE_ALPHAS)} values)")
print(f"Minimum train rows:     {MIN_TRAIN_ROWS}")


# -----------------------------
# 3. Prepare modeling table
# -----------------------------

mp = model_panel.copy()
mp["date"] = pd.to_datetime(mp["date"])
mp["trade_date"] = pd.to_numeric(mp["trade_date"], errors="coerce").astype(int)
mp["tenor"] = pd.to_numeric(mp["tenor"], errors="coerce").astype(int)

for c in [TARGET_LOG_COL, TARGET_VAR_COL, TARGET_VOL_COL]:
    mp[c] = pd.to_numeric(mp[c], errors="coerce")

# Clean infinities globally.
mp = mp.replace([np.inf, -np.inf], np.nan)

# Baseline columns.
for c in ["forecast_variance", "trailing_realized_variance", "implied_variance"]:
    if c in mp.columns:
        mp[c] = pd.to_numeric(mp[c], errors="coerce")

if "forecast_variance" in mp.columns:
    mp["baseline_current_forecast_log_var"] = np.log(
        mp["forecast_variance"].clip(lower=EPSILON_VARIANCE)
    )
    mp["baseline_current_forecast_var"] = mp["forecast_variance"]
    mp["baseline_current_forecast_vol"] = np.sqrt(mp["baseline_current_forecast_var"]) * 100

if "trailing_realized_variance" in mp.columns:
    mp["baseline_trailing_rv_log_var"] = np.log(
        mp["trailing_realized_variance"].clip(lower=EPSILON_VARIANCE)
    )
    mp["baseline_trailing_rv_var"] = mp["trailing_realized_variance"]
    mp["baseline_trailing_rv_vol"] = np.sqrt(mp["baseline_trailing_rv_var"]) * 100

if "implied_variance" in mp.columns:
    mp["baseline_implied_log_var"] = np.log(
        mp["implied_variance"].clip(lower=EPSILON_VARIANCE)
    )
    mp["baseline_implied_var"] = mp["implied_variance"]
    mp["baseline_implied_vol"] = np.sqrt(mp["baseline_implied_var"]) * 100

print()
print("Modeling panel base summary:")
base_summary = pd.DataFrame([{
    "rows": len(mp),
    "first_date": mp["date"].min(),
    "last_date": mp["date"].max(),
    "target_complete_rows": int(mp["has_complete_corsi_target"].sum())
        if "has_complete_corsi_target" in mp.columns else np.nan,
    "tenors": sorted(mp["tenor"].dropna().unique().tolist()),
}])
display(base_summary)


# -----------------------------
# 4. Metric helper
# -----------------------------

def compute_prediction_metrics(df, pred_log_col, pred_var_col, pred_vol_col, group_cols, label):
    """
    Compute forecast metrics for one prediction column set.
    """
    work = df.copy()

    valid = (
        work[TARGET_LOG_COL].notna()
        & work[TARGET_VAR_COL].notna()
        & work[TARGET_VOL_COL].notna()
        & work[pred_log_col].notna()
        & work[pred_var_col].notna()
        & work[pred_vol_col].notna()
        & (work[TARGET_VAR_COL] > 0)
        & (work[pred_var_col] > 0)
    )

    work = work.loc[valid].copy()

    if work.empty:
        return pd.DataFrame()

    work["log_error"] = work[pred_log_col] - work[TARGET_LOG_COL]
    work["abs_log_error"] = work["log_error"].abs()
    work["squared_log_error"] = work["log_error"] ** 2

    work["variance_error"] = work[pred_var_col] - work[TARGET_VAR_COL]
    work["abs_variance_error"] = work["variance_error"].abs()

    work["vol_error"] = work[pred_vol_col] - work[TARGET_VOL_COL]
    work["abs_vol_error"] = work["vol_error"].abs()

    work["pred_over_actual_variance_ratio"] = work[pred_var_col] / work[TARGET_VAR_COL]

    rows = []

    grouped = work.groupby(group_cols, dropna=False) if group_cols else [((), work)]

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "model_name": label,
            "rows": len(g),
            "mean_actual_vol": g[TARGET_VOL_COL].mean(),
            "mean_pred_vol": g[pred_vol_col].mean(),
            "median_actual_vol": g[TARGET_VOL_COL].median(),
            "median_pred_vol": g[pred_vol_col].median(),
            "mae_log_var": g["abs_log_error"].mean(),
            "rmse_log_var": np.sqrt(g["squared_log_error"].mean()),
            "bias_log_var": g["log_error"].mean(),
            "mae_variance": g["abs_variance_error"].mean(),
            "bias_variance": g["variance_error"].mean(),
            "mae_vol_points": g["abs_vol_error"].mean(),
            "bias_vol_points": g["vol_error"].mean(),
            "mean_pred_over_actual_variance_ratio": g["pred_over_actual_variance_ratio"].mean(),
            "median_pred_over_actual_variance_ratio": g["pred_over_actual_variance_ratio"].median(),
            "corr_log_var": g[pred_log_col].corr(g[TARGET_LOG_COL]) if len(g) >= 10 else np.nan,
            "corr_var": g[pred_var_col].corr(g[TARGET_VAR_COL]) if len(g) >= 10 else np.nan,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        rows.append(row)

    return pd.DataFrame(rows)


# -----------------------------
# 5. Walk-forward Ridge models
# -----------------------------

prediction_frames = []
model_fit_rows = []
coefficient_frames = []

for feature_set_name in MODEL_FEATURE_SETS_TO_RUN:
    if feature_set_name not in feature_sets:
        print(f"Skipping missing feature set: {feature_set_name}")
        continue

    feature_cols = [c for c in feature_sets[feature_set_name] if c in mp.columns]

    model_row_col = f"{feature_set_name}_model_row"

    if model_row_col not in mp.columns:
        raise ValueError(f"Missing model-row flag: {model_row_col}")

    print()
    print("=" * 100)
    print(f"Fitting feature set: {feature_set_name}")
    print(f"Feature count: {len(feature_cols)}")
    print("=" * 100)

    for test_year in TEST_YEARS:
        test_start = pd.Timestamp(f"{test_year}-01-01")
        test_end = pd.Timestamp(f"{test_year}-12-31")
        train_cutoff = test_start - pd.Timedelta(days=EMBARGO_DAYS)

        train_mask = (
            mp[model_row_col].astype(bool)
            & (mp["date"] <= train_cutoff)
        )

        test_mask = (
            mp[model_row_col].astype(bool)
            & (mp["date"] >= test_start)
            & (mp["date"] <= test_end)
        )

        train_df = mp.loc[train_mask].copy()
        test_df = mp.loc[test_mask].copy()

        if len(train_df) < MIN_TRAIN_ROWS or len(test_df) == 0:
            model_fit_rows.append({
                "feature_set": feature_set_name,
                "test_year": test_year,
                "status": "SKIPPED",
                "train_rows": len(train_df),
                "test_rows": len(test_df),
                "reason": "insufficient train rows or zero test rows",
            })
            print(
                f"Year {test_year}: SKIPPED "
                f"(train_rows={len(train_df):,}, test_rows={len(test_df):,})"
            )
            continue

        X_train = train_df[feature_cols]
        y_train = train_df[TARGET_LOG_COL]

        X_test = test_df[feature_cols]

        pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", RidgeCV(alphas=RIDGE_ALPHAS)),
        ])

        pipe.fit(X_train, y_train)

        pred_log = pipe.predict(X_test)
        pred_var = np.exp(pred_log)
        pred_vol = np.sqrt(pred_var) * 100

        pred_df = test_df[
            [
                "date",
                "trade_date",
                "tenor",
                "tenor_group",
                TARGET_LOG_COL,
                TARGET_VAR_COL,
                TARGET_VOL_COL,
                "forward_window_complete_corsi",
                "forward_num_sessions_corsi",
                "forward_review_session_count_corsi",
            ]
            + [c for c in [
                "implied_variance",
                "vix_style_vol",
                "forecast_variance",
                "forecast_vol",
                "trailing_realized_variance",
                "trailing_realized_vol",
                "vrp_log",
                "vrp_z_3m",
                "vrp_z_1y",
                "rv21d",
                "rsi14",
            ] if c in test_df.columns]
        ].copy()

        pred_df["feature_set"] = feature_set_name
        pred_df["model_name"] = f"{feature_set_name}_ridge"
        pred_df["test_year"] = test_year
        pred_df["train_start_date"] = train_df["date"].min()
        pred_df["train_end_date"] = train_df["date"].max()
        pred_df["train_cutoff_date"] = train_cutoff
        pred_df["train_rows"] = len(train_df)
        pred_df["test_rows"] = len(test_df)
        pred_df["ridge_alpha"] = pipe.named_steps["ridge"].alpha_

        pred_df["pred_log_forward_variance_corsi"] = pred_log
        pred_df["pred_forward_variance_corsi"] = pred_var
        pred_df["pred_forward_vol_corsi"] = pred_vol

        pred_df["pred_log_error"] = pred_df["pred_log_forward_variance_corsi"] - pred_df[TARGET_LOG_COL]
        pred_df["pred_abs_log_error"] = pred_df["pred_log_error"].abs()
        pred_df["pred_vol_error"] = pred_df["pred_forward_vol_corsi"] - pred_df[TARGET_VOL_COL]
        pred_df["pred_abs_vol_error"] = pred_df["pred_vol_error"].abs()

        prediction_frames.append(pred_df)

        model_fit_rows.append({
            "feature_set": feature_set_name,
            "test_year": test_year,
            "status": "FIT",
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "train_start_date": train_df["date"].min(),
            "train_end_date": train_df["date"].max(),
            "train_cutoff_date": train_cutoff,
            "test_start_date": test_df["date"].min(),
            "test_end_date": test_df["date"].max(),
            "feature_count": len(feature_cols),
            "ridge_alpha": pipe.named_steps["ridge"].alpha_,
        })

        # Store standardized Ridge coefficients for interpretability.
        ridge = pipe.named_steps["ridge"]
        coef_df = pd.DataFrame({
            "feature_set": feature_set_name,
            "test_year": test_year,
            "feature_name": feature_cols,
            "standardized_coefficient": ridge.coef_,
            "abs_standardized_coefficient": np.abs(ridge.coef_),
        }).sort_values("abs_standardized_coefficient", ascending=False)

        coefficient_frames.append(coef_df)

        print(
            f"Year {test_year}: FIT "
            f"(train_rows={len(train_df):,}, test_rows={len(test_df):,}, "
            f"alpha={pipe.named_steps['ridge'].alpha_:.6g})"
        )

model_fit_audit = pd.DataFrame(model_fit_rows)

if prediction_frames:
    corsi_model_predictions = pd.concat(prediction_frames, ignore_index=True)
else:
    raise RuntimeError("No model predictions were generated.")

if coefficient_frames:
    corsi_model_coefficients = pd.concat(coefficient_frames, ignore_index=True)
else:
    corsi_model_coefficients = pd.DataFrame()

print()
print("Model fit audit:")
display(model_fit_audit)


# -----------------------------
# 6. Add baseline predictions for same rows
# -----------------------------

pred = corsi_model_predictions.copy()

# Current production forecast baseline.
if "forecast_variance" in pred.columns:
    pred["baseline_current_forecast_log_var"] = np.log(
        pd.to_numeric(pred["forecast_variance"], errors="coerce").clip(lower=EPSILON_VARIANCE)
    )
    pred["baseline_current_forecast_var"] = pd.to_numeric(pred["forecast_variance"], errors="coerce")
    pred["baseline_current_forecast_vol"] = np.sqrt(pred["baseline_current_forecast_var"]) * 100

# Trailing RV baseline.
if "trailing_realized_variance" in pred.columns:
    pred["baseline_trailing_rv_log_var"] = np.log(
        pd.to_numeric(pred["trailing_realized_variance"], errors="coerce").clip(lower=EPSILON_VARIANCE)
    )
    pred["baseline_trailing_rv_var"] = pd.to_numeric(pred["trailing_realized_variance"], errors="coerce")
    pred["baseline_trailing_rv_vol"] = np.sqrt(pred["baseline_trailing_rv_var"]) * 100

# Implied variance as a diagnostic benchmark, not a pure forecast denominator.
if "implied_variance" in pred.columns:
    pred["baseline_implied_log_var"] = np.log(
        pd.to_numeric(pred["implied_variance"], errors="coerce").clip(lower=EPSILON_VARIANCE)
    )
    pred["baseline_implied_var"] = pd.to_numeric(pred["implied_variance"], errors="coerce")
    pred["baseline_implied_vol"] = np.sqrt(pred["baseline_implied_var"]) * 100

corsi_model_predictions = pred.copy()


# -----------------------------
# 7. Metrics
# -----------------------------

metric_frames = []

# Model metrics.
metric_frames.append(
    compute_prediction_metrics(
        corsi_model_predictions,
        "pred_log_forward_variance_corsi",
        "pred_forward_variance_corsi",
        "pred_forward_vol_corsi",
        ["model_name"],
        "ridge_model",
    )
)

metric_frames.append(
    compute_prediction_metrics(
        corsi_model_predictions,
        "pred_log_forward_variance_corsi",
        "pred_forward_variance_corsi",
        "pred_forward_vol_corsi",
        ["model_name", "test_year"],
        "ridge_model",
    )
)

metric_frames.append(
    compute_prediction_metrics(
        corsi_model_predictions,
        "pred_log_forward_variance_corsi",
        "pred_forward_variance_corsi",
        "pred_forward_vol_corsi",
        ["model_name", "tenor"],
        "ridge_model",
    )
)

metric_frames.append(
    compute_prediction_metrics(
        corsi_model_predictions,
        "pred_log_forward_variance_corsi",
        "pred_forward_variance_corsi",
        "pred_forward_vol_corsi",
        ["model_name", "test_year", "tenor"],
        "ridge_model",
    )
)

# Baseline metrics by feature-set rows.
# These will duplicate across feature-set prediction rows, so later we dedupe to avoid overweighting.
baseline_eval_base = (
    corsi_model_predictions
    .drop_duplicates(subset=["date", "trade_date", "tenor"])
    .copy()
)

baseline_metric_frames = []

if "baseline_current_forecast_var" in baseline_eval_base.columns:
    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_current_forecast_log_var",
            "baseline_current_forecast_var",
            "baseline_current_forecast_vol",
            [],
            "baseline_current_forecast",
        )
    )

    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_current_forecast_log_var",
            "baseline_current_forecast_var",
            "baseline_current_forecast_vol",
            ["test_year"],
            "baseline_current_forecast",
        )
    )

    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_current_forecast_log_var",
            "baseline_current_forecast_var",
            "baseline_current_forecast_vol",
            ["tenor"],
            "baseline_current_forecast",
        )
    )

if "baseline_trailing_rv_var" in baseline_eval_base.columns:
    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_trailing_rv_log_var",
            "baseline_trailing_rv_var",
            "baseline_trailing_rv_vol",
            [],
            "baseline_trailing_rv",
        )
    )

    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_trailing_rv_log_var",
            "baseline_trailing_rv_var",
            "baseline_trailing_rv_vol",
            ["test_year"],
            "baseline_trailing_rv",
        )
    )

    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_trailing_rv_log_var",
            "baseline_trailing_rv_var",
            "baseline_trailing_rv_vol",
            ["tenor"],
            "baseline_trailing_rv",
        )
    )

if "baseline_implied_var" in baseline_eval_base.columns:
    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_implied_log_var",
            "baseline_implied_var",
            "baseline_implied_vol",
            [],
            "baseline_implied_variance",
        )
    )

    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_implied_log_var",
            "baseline_implied_var",
            "baseline_implied_vol",
            ["test_year"],
            "baseline_implied_variance",
        )
    )

    baseline_metric_frames.append(
        compute_prediction_metrics(
            baseline_eval_base,
            "baseline_implied_log_var",
            "baseline_implied_var",
            "baseline_implied_vol",
            ["tenor"],
            "baseline_implied_variance",
        )
    )

all_metric_frames = metric_frames + baseline_metric_frames
all_metric_frames = [m for m in all_metric_frames if not m.empty]

corsi_model_metrics = pd.concat(all_metric_frames, ignore_index=True)


# -----------------------------
# 8. Summary views
# -----------------------------

overall_model_metrics = (
    corsi_model_metrics
    .loc[
        corsi_model_metrics["test_year"].isna()
        if "test_year" in corsi_model_metrics.columns
        else slice(None)
    ]
    .copy()
)

# Cleaner overall table: group columns absent / NaN.
overall_cols = [
    "model_name",
    "rows",
    "mean_actual_vol",
    "mean_pred_vol",
    "mae_log_var",
    "rmse_log_var",
    "bias_log_var",
    "mae_vol_points",
    "bias_vol_points",
    "corr_log_var",
    "corr_var",
]

overall_cols = [c for c in overall_cols if c in corsi_model_metrics.columns]

overall_metrics_display = (
    corsi_model_metrics
    .loc[
        corsi_model_metrics.get("test_year", pd.Series(index=corsi_model_metrics.index, dtype=float)).isna()
        & corsi_model_metrics.get("tenor", pd.Series(index=corsi_model_metrics.index, dtype=float)).isna()
    ]
    .copy()
)

if overall_metrics_display.empty:
    overall_metrics_display = corsi_model_metrics.copy()

overall_metrics_display = overall_metrics_display[overall_cols].sort_values("mae_log_var")

by_tenor_metrics_display = (
    corsi_model_metrics
    .loc[
        corsi_model_metrics.get("tenor", pd.Series(index=corsi_model_metrics.index, dtype=float)).notna()
        & corsi_model_metrics.get("test_year", pd.Series(index=corsi_model_metrics.index, dtype=float)).isna()
    ]
    .copy()
)

by_year_metrics_display = (
    corsi_model_metrics
    .loc[
        corsi_model_metrics.get("test_year", pd.Series(index=corsi_model_metrics.index, dtype=float)).notna()
        & corsi_model_metrics.get("tenor", pd.Series(index=corsi_model_metrics.index, dtype=float)).isna()
    ]
    .copy()
)

print()
print("Overall forecast metrics, sorted by MAE log variance:")
display(overall_metrics_display)

print()
print("Forecast metrics by tenor:")
if by_tenor_metrics_display.empty:
    print("No by-tenor metrics available.")
else:
    display(
        by_tenor_metrics_display[
            [c for c in [
                "model_name",
                "tenor",
                "rows",
                "mae_log_var",
                "rmse_log_var",
                "bias_log_var",
                "mae_vol_points",
                "bias_vol_points",
                "corr_log_var",
            ] if c in by_tenor_metrics_display.columns]
        ].sort_values(["model_name", "tenor"])
    )

print()
print("Forecast metrics by test year:")
if by_year_metrics_display.empty:
    print("No by-year metrics available.")
else:
    display(
        by_year_metrics_display[
            [c for c in [
                "model_name",
                "test_year",
                "rows",
                "mae_log_var",
                "rmse_log_var",
                "bias_log_var",
                "mae_vol_points",
                "bias_vol_points",
                "corr_log_var",
            ] if c in by_year_metrics_display.columns]
        ].sort_values(["model_name", "test_year"])
    )

print()
print("Top standardized Ridge coefficients by feature set / test year:")
if corsi_model_coefficients.empty:
    print("No coefficients available.")
else:
    display(
        corsi_model_coefficients
        .groupby(["feature_set", "test_year"])
        .head(15)
        .reset_index(drop=True)
    )


# -----------------------------
# 9. Representative prediction diagnostics
# -----------------------------

latest_complete_predictions = (
    corsi_model_predictions
    .loc[corsi_model_predictions["forward_window_complete_corsi"].astype(bool)]
    .sort_values(["date", "tenor", "model_name"])
    .tail(60)
    .copy()
)

worst_model_errors = (
    corsi_model_predictions
    .sort_values("pred_abs_log_error", ascending=False)
    .head(50)
    .copy()
)

print()
print("Latest complete predictions:")
display(
    latest_complete_predictions[
        [c for c in [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            "model_name",
            TARGET_VOL_COL,
            "pred_forward_vol_corsi",
            "pred_vol_error",
            TARGET_VAR_COL,
            "pred_forward_variance_corsi",
            "pred_log_error",
            "test_year",
            "ridge_alpha",
        ] if c in latest_complete_predictions.columns]
    ]
)

print()
print("Worst model log-variance errors:")
display(
    worst_model_errors[
        [c for c in [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            "model_name",
            TARGET_VOL_COL,
            "pred_forward_vol_corsi",
            "pred_vol_error",
            TARGET_VAR_COL,
            "pred_forward_variance_corsi",
            "pred_log_error",
            "pred_abs_log_error",
            "test_year",
            "forward_review_session_count_corsi",
        ] if c in worst_model_errors.columns]
    ]
)


# -----------------------------
# 10. Save outputs
# -----------------------------

safe_start = corsi_model_predictions["date"].min().strftime("%Y%m%d")
safe_end = corsi_model_predictions["date"].max().strftime("%Y%m%d")

cell6_predictions_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_direct_ridge_predictions_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell6_predictions_csv_sample_output = (
    CORSI_AUDIT_DIR
    / f"corsi_direct_ridge_predictions_v1_sample_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6_direct_ridge_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6_fit_audit_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6_model_fit_audit_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6_coefficients_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6_ridge_coefficients_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6_overall_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6_overall_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6_by_tenor_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6_by_tenor_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6_by_year_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6_by_year_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6_worst_errors_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6_worst_model_errors_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

corsi_model_predictions.to_parquet(cell6_predictions_output, index=False)
corsi_model_predictions.head(500).to_csv(cell6_predictions_csv_sample_output, index=False)
corsi_model_metrics.to_csv(cell6_metrics_output, index=False)
model_fit_audit.to_csv(cell6_fit_audit_output, index=False)
corsi_model_coefficients.to_csv(cell6_coefficients_output, index=False)
overall_metrics_display.to_csv(cell6_overall_metrics_output, index=False)
by_tenor_metrics_display.to_csv(cell6_by_tenor_metrics_output, index=False)
by_year_metrics_display.to_csv(cell6_by_year_metrics_output, index=False)
worst_model_errors.to_csv(cell6_worst_errors_output, index=False)

print()
print("Cell 6 outputs saved:")
print(f"Predictions:                  {cell6_predictions_output}")
print(f"Prediction CSV sample:        {cell6_predictions_csv_sample_output}")
print(f"All metrics:                  {cell6_metrics_output}")
print(f"Fit audit:                    {cell6_fit_audit_output}")
print(f"Ridge coefficients:           {cell6_coefficients_output}")
print(f"Overall metrics:              {cell6_overall_metrics_output}")
print(f"By-tenor metrics:             {cell6_by_tenor_metrics_output}")
print(f"By-year metrics:              {cell6_by_year_metrics_output}")
print(f"Worst model errors:           {cell6_worst_errors_output}")

print()
print("Cell 6 complete.")
print("Next step will be Cell 7: inspect forecast curves and decide which model family is worth carrying forward.")

## 6B. Robust simple HAR/Corsi models

Fits the main walk-forward model candidates with smaller, more stable feature sets. This is the key forecast-comparison cell. The main candidates are `har_total_simple` and `har_implied_shrinkage`.

In [ ]:
# Cell 6B: Simple robust HAR/Corsi models
#
# Purpose:
# Cell 6 proved that the wide Ridge model is unstable and not acceptable.
#
# This cell fits simpler, more defensible HAR/Corsi models:
#   1. har_total_simple
#   2. har_total_plus_overnight_downside
#   3. har_total_plus_implied_simple
#   4. har_implied_shrinkage
#
# Key changes vs Cell 6:
#   - far fewer features
#   - stronger Ridge alpha grid
#   - no tiny alpha values
#   - prediction clipping based on training target distribution
#   - same yearly walk-forward with 33-calendar-day embargo
#
# This still does NOT replace production forecast.

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


# -----------------------------
# 1. Load model panel if needed
# -----------------------------

try:
    model_panel
except NameError:
    latest_model_panel_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_model_feature_panel_v1_*.parquet")
    )
    if not latest_model_panel_files:
        raise FileNotFoundError("No Corsi model feature panel found.")
    print(f"Loading model panel: {latest_model_panel_files[-1]}")
    model_panel = pd.read_parquet(latest_model_panel_files[-1])

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"


# -----------------------------
# 2. Configuration
# -----------------------------

TARGET_LOG_COL = "log_forward_realized_variance_corsi"
TARGET_VAR_COL = "forward_realized_variance_corsi"
TARGET_VOL_COL = "forward_realized_vol_corsi"

TEST_YEARS = list(range(2020, 2027))
EMBARGO_DAYS = 33
MIN_TRAIN_ROWS = 750

# Stronger regularization only. No alpha near zero.
ROBUST_RIDGE_ALPHAS = np.array([
    0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0, 10000.0
])

EPSILON_VARIANCE = 1e-12

print("Cell 6B robust HAR/Corsi configuration:")
print(f"Test years:            {TEST_YEARS}")
print(f"Embargo days:          {EMBARGO_DAYS}")
print(f"Minimum train rows:    {MIN_TRAIN_ROWS}")
print(f"Ridge alphas:          {ROBUST_RIDGE_ALPHAS}")


# -----------------------------
# 3. Prepare panel
# -----------------------------

mp = model_panel.copy()
mp["date"] = pd.to_datetime(mp["date"])
mp["trade_date"] = pd.to_numeric(mp["trade_date"], errors="coerce").astype(int)
mp["tenor"] = pd.to_numeric(mp["tenor"], errors="coerce").astype(int)
mp = mp.replace([np.inf, -np.inf], np.nan)

for c in [TARGET_LOG_COL, TARGET_VAR_COL, TARGET_VOL_COL]:
    mp[c] = pd.to_numeric(mp[c], errors="coerce")

mp["has_complete_target_6b"] = (
    mp["forward_window_complete_corsi"].fillna(False).astype(bool)
    & mp[TARGET_LOG_COL].notna()
    & mp[TARGET_VAR_COL].notna()
    & (mp[TARGET_VAR_COL] > 0)
)

tenor_group_cols = [c for c in mp.columns if c.startswith("tenor_group_")]

def existing(cols):
    return [c for c in cols if c in mp.columns]

base_tenor_features = existing([
    "tenor_scaled_30d",
    "log_tenor",
] + tenor_group_cols)

har_total_features = existing([
    "log_spy_total_rv_1d_ann",
    "log_spy_total_rv_mean_5d_ann",
    "log_spy_total_rv_mean_21d_ann",
    "log_spy_total_rv_mean_63d_ann",
])

har_extended_features = existing([
    "log_spy_total_rv_1d_ann",
    "log_spy_total_rv_mean_5d_ann",
    "log_spy_total_rv_mean_21d_ann",
    "log_spy_total_rv_mean_63d_ann",
    "log_spy_intraday_rv_mean_5d_ann",
    "log_spy_intraday_rv_mean_21d_ann",
    "log_spy_overnight_rv_mean_5d_ann",
    "log_spy_overnight_rv_mean_21d_ann",
    "log_spy_downside_rv_mean_5d_ann",
    "log_spy_downside_rv_mean_21d_ann",
    "spy_overnight_share_21d",
    "spy_downside_share_21d",
    "spy_return_sum_5d",
    "spy_return_sum_21d",
    "spy_drawdown_21d",
    "spy_drawdown_63d",
])

implied_simple_features = existing([
    "log_implied_variance",
    "log_iv_curve_mean",
    "log_iv_9d",
    "log_iv_21d",
    "log_iv_33d",
    "iv_slope_33d_9d",
    "iv_slope_21d_9d",
    "iv_curvature_9_21_33",
])

state_simple_features = existing([
    "rv21d",
    "rsi14",
    "spx_log_return",
])

model_specs = {
    "har_total_simple": har_total_features + base_tenor_features,

    "har_total_plus_overnight_downside": (
        har_extended_features
        + base_tenor_features
    ),

    "har_total_plus_implied_simple": (
        har_total_features
        + implied_simple_features
        + base_tenor_features
    ),

    "har_implied_shrinkage": existing([
        "log_implied_variance",
        "log_iv_curve_mean",
        "log_spy_total_rv_mean_5d_ann",
        "log_spy_total_rv_mean_21d_ann",
        "log_spy_total_rv_mean_63d_ann",
        "spy_return_sum_21d",
        "spy_drawdown_63d",
    ]) + base_tenor_features,
}

# Deduplicate feature lists.
for k, cols in model_specs.items():
    seen = set()
    deduped = []
    for c in cols:
        if c not in seen:
            deduped.append(c)
            seen.add(c)
    model_specs[k] = deduped

print()
print("Model specs:")
display(pd.DataFrame([
    {
        "model_name": k,
        "feature_count": len(v),
        "features": ", ".join(v),
    }
    for k, v in model_specs.items()
]))


# -----------------------------
# 4. Metric helper
# -----------------------------

def metrics_table(df, pred_log_col, pred_var_col, pred_vol_col, group_cols, model_name):
    work = df.copy()

    valid = (
        work[TARGET_LOG_COL].notna()
        & work[TARGET_VAR_COL].notna()
        & work[TARGET_VOL_COL].notna()
        & work[pred_log_col].notna()
        & work[pred_var_col].notna()
        & work[pred_vol_col].notna()
        & (work[TARGET_VAR_COL] > 0)
        & (work[pred_var_col] > 0)
    )

    work = work.loc[valid].copy()

    if work.empty:
        return pd.DataFrame()

    work["log_error"] = work[pred_log_col] - work[TARGET_LOG_COL]
    work["abs_log_error"] = work["log_error"].abs()
    work["squared_log_error"] = work["log_error"] ** 2
    work["vol_error"] = work[pred_vol_col] - work[TARGET_VOL_COL]
    work["abs_vol_error"] = work["vol_error"].abs()
    work["variance_error"] = work[pred_var_col] - work[TARGET_VAR_COL]
    work["abs_variance_error"] = work["variance_error"].abs()
    work["pred_over_actual_var"] = work[pred_var_col] / work[TARGET_VAR_COL]

    rows = []

    if group_cols:
        grouped = work.groupby(group_cols, dropna=False)
    else:
        grouped = [((), work)]

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "model_name": model_name,
            "rows": len(g),
            "mean_actual_vol": g[TARGET_VOL_COL].mean(),
            "mean_pred_vol": g[pred_vol_col].mean(),
            "median_actual_vol": g[TARGET_VOL_COL].median(),
            "median_pred_vol": g[pred_vol_col].median(),
            "mae_log_var": g["abs_log_error"].mean(),
            "rmse_log_var": np.sqrt(g["squared_log_error"].mean()),
            "bias_log_var": g["log_error"].mean(),
            "mae_vol_points": g["abs_vol_error"].mean(),
            "bias_vol_points": g["vol_error"].mean(),
            "mae_variance": g["abs_variance_error"].mean(),
            "bias_variance": g["variance_error"].mean(),
            "median_pred_over_actual_var": g["pred_over_actual_var"].median(),
            "mean_pred_over_actual_var": g["pred_over_actual_var"].mean(),
            "corr_log_var": g[pred_log_col].corr(g[TARGET_LOG_COL]) if len(g) >= 10 else np.nan,
            "corr_var": g[pred_var_col].corr(g[TARGET_VAR_COL]) if len(g) >= 10 else np.nan,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        rows.append(row)

    return pd.DataFrame(rows)


# -----------------------------
# 5. Walk-forward fit
# -----------------------------

pred_frames = []
fit_rows = []
coef_frames = []

for model_name, feature_cols in model_specs.items():
    print()
    print("=" * 100)
    print(f"Fitting robust model: {model_name}")
    print(f"Feature count: {len(feature_cols)}")
    print("=" * 100)

    if len(feature_cols) == 0:
        print("Skipping because no features exist.")
        continue

    feature_complete = (
        mp[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .notna()
        .all(axis=1)
    )

    row_mask_base = mp["has_complete_target_6b"] & feature_complete

    for test_year in TEST_YEARS:
        test_start = pd.Timestamp(f"{test_year}-01-01")
        test_end = pd.Timestamp(f"{test_year}-12-31")
        train_cutoff = test_start - pd.Timedelta(days=EMBARGO_DAYS)

        train_mask = row_mask_base & (mp["date"] <= train_cutoff)
        test_mask = row_mask_base & (mp["date"] >= test_start) & (mp["date"] <= test_end)

        train_df = mp.loc[train_mask].copy()
        test_df = mp.loc[test_mask].copy()

        if len(train_df) < MIN_TRAIN_ROWS or len(test_df) == 0:
            fit_rows.append({
                "model_name": model_name,
                "test_year": test_year,
                "status": "SKIPPED",
                "train_rows": len(train_df),
                "test_rows": len(test_df),
                "reason": "insufficient train rows or zero test rows",
            })
            print(
                f"{test_year}: SKIPPED "
                f"(train_rows={len(train_df):,}, test_rows={len(test_df):,})"
            )
            continue

        X_train = train_df[feature_cols]
        y_train = train_df[TARGET_LOG_COL]
        X_test = test_df[feature_cols]

        pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", RidgeCV(alphas=ROBUST_RIDGE_ALPHAS)),
        ])

        pipe.fit(X_train, y_train)

        pred_log_raw = pipe.predict(X_test)

        # Prediction clipping to avoid pathological near-zero / insane-vol forecasts.
        # Use only the training target distribution.
        train_q_low = y_train.quantile(0.005)
        train_q_high = y_train.quantile(0.995)

        pred_log_floor = train_q_low - 0.25
        pred_log_cap = train_q_high + 0.75

        pred_log_clipped = np.clip(pred_log_raw, pred_log_floor, pred_log_cap)

        pred_var = np.exp(pred_log_clipped)
        pred_vol = np.sqrt(pred_var) * 100

        out_cols = [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            TARGET_LOG_COL,
            TARGET_VAR_COL,
            TARGET_VOL_COL,
            "forward_window_complete_corsi",
            "forward_num_sessions_corsi",
            "forward_review_session_count_corsi",
        ]

        out_cols += [
            c for c in [
                "implied_variance",
                "vix_style_vol",
                "forecast_variance",
                "forecast_vol",
                "trailing_realized_variance",
                "trailing_realized_vol",
                "vrp_log",
                "vrp_z_3m",
                "vrp_z_1y",
                "rv21d",
                "rsi14",
            ]
            if c in test_df.columns
        ]

        pred_df = test_df[out_cols].copy()
        pred_df["model_name"] = model_name
        pred_df["test_year"] = test_year
        pred_df["train_start_date"] = train_df["date"].min()
        pred_df["train_end_date"] = train_df["date"].max()
        pred_df["train_cutoff_date"] = train_cutoff
        pred_df["train_rows"] = len(train_df)
        pred_df["test_rows"] = len(test_df)
        pred_df["ridge_alpha"] = pipe.named_steps["ridge"].alpha_
        pred_df["feature_count"] = len(feature_cols)

        pred_df["pred_log_raw"] = pred_log_raw
        pred_df["pred_log_floor"] = pred_log_floor
        pred_df["pred_log_cap"] = pred_log_cap
        pred_df["pred_log_clipped_flag"] = pred_log_raw != pred_log_clipped

        pred_df["pred_log_forward_variance_corsi_6b"] = pred_log_clipped
        pred_df["pred_forward_variance_corsi_6b"] = pred_var
        pred_df["pred_forward_vol_corsi_6b"] = pred_vol

        pred_df["pred_log_error_6b"] = (
            pred_df["pred_log_forward_variance_corsi_6b"]
            - pred_df[TARGET_LOG_COL]
        )

        pred_df["pred_abs_log_error_6b"] = pred_df["pred_log_error_6b"].abs()

        pred_df["pred_vol_error_6b"] = (
            pred_df["pred_forward_vol_corsi_6b"]
            - pred_df[TARGET_VOL_COL]
        )

        pred_df["pred_abs_vol_error_6b"] = pred_df["pred_vol_error_6b"].abs()

        pred_frames.append(pred_df)

        fit_rows.append({
            "model_name": model_name,
            "test_year": test_year,
            "status": "FIT",
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "train_start_date": train_df["date"].min(),
            "train_end_date": train_df["date"].max(),
            "train_cutoff_date": train_cutoff,
            "test_start_date": test_df["date"].min(),
            "test_end_date": test_df["date"].max(),
            "feature_count": len(feature_cols),
            "ridge_alpha": pipe.named_steps["ridge"].alpha_,
            "pred_log_floor": pred_log_floor,
            "pred_log_cap": pred_log_cap,
            "clipped_predictions": int((pred_log_raw != pred_log_clipped).sum()),
            "clipped_prediction_pct": float((pred_log_raw != pred_log_clipped).mean()),
        })

        coef_df = pd.DataFrame({
            "model_name": model_name,
            "test_year": test_year,
            "feature_name": feature_cols,
            "standardized_coefficient": pipe.named_steps["ridge"].coef_,
            "abs_standardized_coefficient": np.abs(pipe.named_steps["ridge"].coef_),
        }).sort_values("abs_standardized_coefficient", ascending=False)

        coef_frames.append(coef_df)

        print(
            f"{test_year}: FIT "
            f"(train={len(train_df):,}, test={len(test_df):,}, "
            f"alpha={pipe.named_steps['ridge'].alpha_}, "
            f"clipped={int((pred_log_raw != pred_log_clipped).sum()):,})"
        )

if not pred_frames:
    raise RuntimeError("No robust HAR/Corsi predictions were generated.")

corsi_6b_predictions = pd.concat(pred_frames, ignore_index=True)
corsi_6b_fit_audit = pd.DataFrame(fit_rows)
corsi_6b_coefficients = (
    pd.concat(coef_frames, ignore_index=True)
    if coef_frames
    else pd.DataFrame()
)

print()
print("Cell 6B fit audit:")
display(corsi_6b_fit_audit)


# -----------------------------
# 6. Baseline predictions on same rows
# -----------------------------

pred = corsi_6b_predictions.copy()

if "forecast_variance" in pred.columns:
    pred["baseline_current_forecast_var"] = pd.to_numeric(pred["forecast_variance"], errors="coerce")
    pred["baseline_current_forecast_log_var"] = np.log(
        pred["baseline_current_forecast_var"].clip(lower=EPSILON_VARIANCE)
    )
    pred["baseline_current_forecast_vol"] = np.sqrt(pred["baseline_current_forecast_var"]) * 100

if "trailing_realized_variance" in pred.columns:
    pred["baseline_trailing_rv_var"] = pd.to_numeric(pred["trailing_realized_variance"], errors="coerce")
    pred["baseline_trailing_rv_log_var"] = np.log(
        pred["baseline_trailing_rv_var"].clip(lower=EPSILON_VARIANCE)
    )
    pred["baseline_trailing_rv_vol"] = np.sqrt(pred["baseline_trailing_rv_var"]) * 100

if "implied_variance" in pred.columns:
    pred["baseline_implied_var"] = pd.to_numeric(pred["implied_variance"], errors="coerce")
    pred["baseline_implied_log_var"] = np.log(
        pred["baseline_implied_var"].clip(lower=EPSILON_VARIANCE)
    )
    pred["baseline_implied_vol"] = np.sqrt(pred["baseline_implied_var"]) * 100

corsi_6b_predictions = pred.copy()


# -----------------------------
# 7. Metrics
# -----------------------------

metric_frames = []

# Robust model metrics.
for group_cols in [
    [],
    ["model_name"],
    ["model_name", "tenor"],
    ["model_name", "test_year"],
    ["model_name", "test_year", "tenor"],
]:
    metric_frames.append(
        metrics_table(
            corsi_6b_predictions,
            "pred_log_forward_variance_corsi_6b",
            "pred_forward_variance_corsi_6b",
            "pred_forward_vol_corsi_6b",
            group_cols,
            "robust_har_corsi",
        )
    )

# Baselines, deduped by date x tenor.
baseline_eval = (
    corsi_6b_predictions
    .drop_duplicates(subset=["date", "trade_date", "tenor"])
    .copy()
)

if "baseline_current_forecast_var" in baseline_eval.columns:
    for group_cols in [[], ["tenor"], ["test_year"]]:
        metric_frames.append(
            metrics_table(
                baseline_eval,
                "baseline_current_forecast_log_var",
                "baseline_current_forecast_var",
                "baseline_current_forecast_vol",
                group_cols,
                "baseline_current_forecast",
            )
        )

if "baseline_trailing_rv_var" in baseline_eval.columns:
    for group_cols in [[], ["tenor"], ["test_year"]]:
        metric_frames.append(
            metrics_table(
                baseline_eval,
                "baseline_trailing_rv_log_var",
                "baseline_trailing_rv_var",
                "baseline_trailing_rv_vol",
                group_cols,
                "baseline_trailing_rv",
            )
        )

if "baseline_implied_var" in baseline_eval.columns:
    for group_cols in [[], ["tenor"], ["test_year"]]:
        metric_frames.append(
            metrics_table(
                baseline_eval,
                "baseline_implied_log_var",
                "baseline_implied_var",
                "baseline_implied_vol",
                group_cols,
                "baseline_implied_variance",
            )
        )

metric_frames = [m for m in metric_frames if not m.empty]
corsi_6b_metrics = pd.concat(metric_frames, ignore_index=True)

overall_metrics_6b = (
    corsi_6b_metrics
    .loc[
        corsi_6b_metrics.get("tenor", pd.Series(index=corsi_6b_metrics.index, dtype=float)).isna()
        & corsi_6b_metrics.get("test_year", pd.Series(index=corsi_6b_metrics.index, dtype=float)).isna()
    ]
    .copy()
    .sort_values("mae_log_var")
)

by_tenor_metrics_6b = (
    corsi_6b_metrics
    .loc[
        corsi_6b_metrics.get("tenor", pd.Series(index=corsi_6b_metrics.index, dtype=float)).notna()
        & corsi_6b_metrics.get("test_year", pd.Series(index=corsi_6b_metrics.index, dtype=float)).isna()
    ]
    .copy()
    .sort_values(["model_name", "tenor"])
)

by_year_metrics_6b = (
    corsi_6b_metrics
    .loc[
        corsi_6b_metrics.get("test_year", pd.Series(index=corsi_6b_metrics.index, dtype=float)).notna()
        & corsi_6b_metrics.get("tenor", pd.Series(index=corsi_6b_metrics.index, dtype=float)).isna()
    ]
    .copy()
    .sort_values(["model_name", "test_year"])
)

print()
print("Cell 6B overall metrics, sorted by MAE log variance:")
display(
    overall_metrics_6b[
        [c for c in [
            "model_name",
            "rows",
            "mean_actual_vol",
            "mean_pred_vol",
            "median_actual_vol",
            "median_pred_vol",
            "mae_log_var",
            "rmse_log_var",
            "bias_log_var",
            "mae_vol_points",
            "bias_vol_points",
            "corr_log_var",
            "corr_var",
        ] if c in overall_metrics_6b.columns]
    ]
)

print()
print("Cell 6B metrics by tenor:")
display(
    by_tenor_metrics_6b[
        [c for c in [
            "model_name",
            "tenor",
            "rows",
            "mae_log_var",
            "rmse_log_var",
            "bias_log_var",
            "mae_vol_points",
            "bias_vol_points",
            "corr_log_var",
        ] if c in by_tenor_metrics_6b.columns]
    ]
)

print()
print("Cell 6B metrics by test year:")
display(
    by_year_metrics_6b[
        [c for c in [
            "model_name",
            "test_year",
            "rows",
            "mae_log_var",
            "rmse_log_var",
            "bias_log_var",
            "mae_vol_points",
            "bias_vol_points",
            "corr_log_var",
        ] if c in by_year_metrics_6b.columns]
    ]
)

print()
print("Top coefficients for robust HAR/Corsi models:")
if corsi_6b_coefficients.empty:
    print("No coefficient table.")
else:
    display(
        corsi_6b_coefficients
        .groupby(["model_name", "test_year"])
        .head(10)
        .reset_index(drop=True)
    )


# -----------------------------
# 8. Error diagnostics
# -----------------------------

worst_errors_6b = (
    corsi_6b_predictions
    .sort_values("pred_abs_log_error_6b", ascending=False)
    .head(75)
    .copy()
)

latest_predictions_6b = (
    corsi_6b_predictions
    .sort_values(["date", "tenor", "model_name"])
    .tail(80)
    .copy()
)

print()
print("Cell 6B worst log-variance errors:")
display(
    worst_errors_6b[
        [c for c in [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            "model_name",
            TARGET_VOL_COL,
            "pred_forward_vol_corsi_6b",
            "pred_vol_error_6b",
            TARGET_VAR_COL,
            "pred_forward_variance_corsi_6b",
            "pred_log_error_6b",
            "pred_abs_log_error_6b",
            "test_year",
            "ridge_alpha",
            "pred_log_clipped_flag",
            "forward_review_session_count_corsi",
        ] if c in worst_errors_6b.columns]
    ]
)

print()
print("Cell 6B latest predictions:")
display(
    latest_predictions_6b[
        [c for c in [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            "model_name",
            TARGET_VOL_COL,
            "pred_forward_vol_corsi_6b",
            "pred_vol_error_6b",
            "test_year",
            "ridge_alpha",
            "pred_log_clipped_flag",
        ] if c in latest_predictions_6b.columns]
    ]
)


# -----------------------------
# 9. Save outputs
# -----------------------------

safe_start = corsi_6b_predictions["date"].min().strftime("%Y%m%d")
safe_end = corsi_6b_predictions["date"].max().strftime("%Y%m%d")

cell6b_predictions_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_simple_har_predictions_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell6b_predictions_sample_output = (
    CORSI_AUDIT_DIR
    / f"corsi_simple_har_predictions_v1_sample_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6b_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6b_simple_har_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6b_overall_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6b_overall_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6b_by_tenor_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6b_by_tenor_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6b_by_year_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6b_by_year_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6b_fit_audit_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6b_fit_audit_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6b_coefficients_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6b_coefficients_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell6b_worst_errors_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell6b_worst_errors_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

corsi_6b_predictions.to_parquet(cell6b_predictions_output, index=False)
corsi_6b_predictions.head(750).to_csv(cell6b_predictions_sample_output, index=False)
corsi_6b_metrics.to_csv(cell6b_metrics_output, index=False)
overall_metrics_6b.to_csv(cell6b_overall_metrics_output, index=False)
by_tenor_metrics_6b.to_csv(cell6b_by_tenor_metrics_output, index=False)
by_year_metrics_6b.to_csv(cell6b_by_year_metrics_output, index=False)
corsi_6b_fit_audit.to_csv(cell6b_fit_audit_output, index=False)
corsi_6b_coefficients.to_csv(cell6b_coefficients_output, index=False)
worst_errors_6b.to_csv(cell6b_worst_errors_output, index=False)

print()
print("Cell 6B outputs saved:")
print(f"Simple HAR predictions:      {cell6b_predictions_output}")
print(f"Prediction sample:           {cell6b_predictions_sample_output}")
print(f"All metrics:                 {cell6b_metrics_output}")
print(f"Overall metrics:             {cell6b_overall_metrics_output}")
print(f"By-tenor metrics:            {cell6b_by_tenor_metrics_output}")
print(f"By-year metrics:             {cell6b_by_year_metrics_output}")
print(f"Fit audit:                   {cell6b_fit_audit_output}")
print(f"Coefficients:                {cell6b_coefficients_output}")
print(f"Worst errors:                {cell6b_worst_errors_output}")

print()
print("Cell 6B complete.")
print("Next step: compare Cell 6B against baseline. Only continue to forecast-curve charts if one of these simple models is competitive.")

## 7. Forecast-curve inspection

Inspects the shape and smoothness of candidate forecast curves across representative dates. This checks whether forecast term structures are economically reasonable.

In [ ]:
# Cell 7: Inspect Corsi / HAR forecast curves
#
# Purpose:
# Inspect the forecast term structures from the simple robust HAR/Corsi models.
#
# Focus candidates:
#   1. har_total_simple
#   2. har_implied_shrinkage
#
# Compare against:
#   - current production forecast
#   - implied vol
#   - actual forward realized vol, where available
#
# This cell is diagnostic only. It does NOT update production forecast.

import matplotlib.pyplot as plt


# -----------------------------
# 1. Load predictions if needed
# -----------------------------

try:
    corsi_6b_predictions
except NameError:
    latest_6b_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_simple_har_predictions_v1_*.parquet")
    )
    if not latest_6b_files:
        raise FileNotFoundError("No Cell 6B simple HAR prediction file found.")

    print(f"Loading Cell 6B predictions: {latest_6b_files[-1]}")
    corsi_6b_predictions = pd.read_parquet(latest_6b_files[-1])

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"


# -----------------------------
# 2. Configuration
# -----------------------------

CANDIDATE_MODELS = [
    "har_total_simple",
    "har_implied_shrinkage",
]

REPRESENTATIVE_DATES = [
    "2020-03-16",
    "2020-03-24",
    "2021-02-08",
    "2022-11-10",
    "2024-08-05",
    "2025-04-07",
    "2026-06-23",
]

CELL7_CHART_DIR = (
    CORSI_AUDIT_DIR
    / "charts"
    / "cell7_corsi_forecast_curves"
)

CELL7_CHART_DIR.mkdir(parents=True, exist_ok=True)

print("Cell 7 configuration:")
print(f"Candidate models:     {CANDIDATE_MODELS}")
print(f"Chart dir:            {CELL7_CHART_DIR}")


# -----------------------------
# 3. Prepare curve panel
# -----------------------------

pred = corsi_6b_predictions.copy()
pred["date"] = pd.to_datetime(pred["date"])
pred["tenor"] = pd.to_numeric(pred["tenor"], errors="coerce").astype(int)

pred = pred.replace([np.inf, -np.inf], np.nan)

curve_panel = pred.loc[pred["model_name"].isin(CANDIDATE_MODELS)].copy()

curve_panel["pred_vol"] = curve_panel["pred_forward_vol_corsi_6b"]
curve_panel["pred_var"] = curve_panel["pred_forward_variance_corsi_6b"]

if "forecast_variance" in curve_panel.columns:
    curve_panel["current_forecast_vol"] = np.sqrt(
        pd.to_numeric(curve_panel["forecast_variance"], errors="coerce")
    ) * 100

if "implied_variance" in curve_panel.columns:
    curve_panel["implied_vol"] = np.sqrt(
        pd.to_numeric(curve_panel["implied_variance"], errors="coerce")
    ) * 100

curve_panel["actual_forward_vol"] = curve_panel["forward_realized_vol_corsi"]


# -----------------------------
# 4. Latest complete dates by model
# -----------------------------

latest_complete_by_model = (
    curve_panel
    .loc[curve_panel["forward_window_complete_corsi"].astype(bool)]
    .groupby("model_name")["date"]
    .max()
    .reset_index()
)

print()
print("Latest complete target date by model:")
display(latest_complete_by_model)


# -----------------------------
# 5. Curve smoothness / shape diagnostics
# -----------------------------

def curve_shape_stats(g, value_col):
    g = g.sort_values("tenor").copy()

    vals = g[value_col].to_numpy(dtype=float)
    tenors = g["tenor"].to_numpy(dtype=float)

    if len(vals) < 2 or np.any(~np.isfinite(vals)):
        return pd.Series({
            "curve_rows": len(g),
            "curve_slope_33_9": np.nan,
            "mean_abs_adjacent_change": np.nan,
            "max_abs_adjacent_change": np.nan,
            "negative_adjacent_steps": np.nan,
        })

    diffs = np.diff(vals)

    return pd.Series({
        "curve_rows": len(g),
        "curve_slope_33_9": vals[-1] - vals[0],
        "mean_abs_adjacent_change": np.mean(np.abs(diffs)),
        "max_abs_adjacent_change": np.max(np.abs(diffs)),
        "negative_adjacent_steps": int((diffs < 0).sum()),
    })


shape_rows = []

for model_name, g_model in curve_panel.groupby("model_name"):
    for date, g_date in g_model.groupby("date"):
        pred_stats = curve_shape_stats(g_date, "pred_vol")
        pred_stats["model_name"] = model_name
        pred_stats["date"] = date
        pred_stats["curve_type"] = "corsi_pred_vol"
        shape_rows.append(pred_stats)

curve_shape_daily = pd.DataFrame(shape_rows)

curve_shape_summary = (
    curve_shape_daily
    .groupby(["model_name", "curve_type"])
    .agg(
        dates=("date", "nunique"),
        mean_slope_33_9=("curve_slope_33_9", "mean"),
        median_slope_33_9=("curve_slope_33_9", "median"),
        mean_abs_adjacent_change=("mean_abs_adjacent_change", "mean"),
        p95_max_abs_adjacent_change=("max_abs_adjacent_change", lambda x: x.quantile(0.95)),
        mean_negative_adjacent_steps=("negative_adjacent_steps", "mean"),
    )
    .reset_index()
)

print()
print("Forecast curve shape / smoothness summary:")
display(curve_shape_summary)


# -----------------------------
# 6. Daily prediction error by date/model
# -----------------------------

date_model_error = (
    curve_panel
    .loc[curve_panel["forward_window_complete_corsi"].astype(bool)]
    .copy()
)

date_model_error["abs_log_error"] = (
    np.log(date_model_error["pred_var"].clip(lower=EPSILON_VARIANCE))
    - date_model_error["log_forward_realized_variance_corsi"]
).abs()

date_model_error["abs_vol_error"] = (
    date_model_error["pred_vol"]
    - date_model_error["actual_forward_vol"]
).abs()

date_model_error_summary = (
    date_model_error
    .groupby(["model_name", "date"])
    .agg(
        mean_abs_log_error=("abs_log_error", "mean"),
        mean_abs_vol_error=("abs_vol_error", "mean"),
        max_abs_vol_error=("abs_vol_error", "max"),
        mean_actual_forward_vol=("actual_forward_vol", "mean"),
        mean_pred_forward_vol=("pred_vol", "mean"),
        mean_implied_vol=("implied_vol", "mean") if "implied_vol" in date_model_error.columns else ("pred_vol", "mean"),
    )
    .reset_index()
)

worst_curve_dates = (
    date_model_error_summary
    .sort_values("mean_abs_log_error", ascending=False)
    .head(40)
)

print()
print("Worst curve dates by average log error:")
display(worst_curve_dates)


# -----------------------------
# 7. Representative curve plots
# -----------------------------

available_dates = sorted(curve_panel["date"].dropna().unique())
available_dates = [pd.Timestamp(d) for d in available_dates]

rep_dates_resolved = []

for d in REPRESENTATIVE_DATES:
    target = pd.Timestamp(d)

    # Use exact date if available; otherwise use nearest prior date.
    candidates = [x for x in available_dates if x <= target]
    if candidates:
        resolved = max(candidates)
        rep_dates_resolved.append(resolved)

rep_dates_resolved = sorted(set(rep_dates_resolved))

print()
print("Representative dates resolved:")
display(pd.DataFrame({"date": rep_dates_resolved}))


def plot_forecast_curve_for_date(curve_date):
    d = pd.Timestamp(curve_date)

    fig, ax = plt.subplots(figsize=(12, 6))

    date_rows_all = curve_panel.loc[curve_panel["date"] == d].copy()

    if date_rows_all.empty:
        print(f"No rows for {d.date()}")
        return None

    # Plot implied / current / actual once using first model rows.
    base = (
        date_rows_all
        .drop_duplicates(subset=["date", "tenor"])
        .sort_values("tenor")
        .copy()
    )

    if "implied_vol" in base.columns:
        ax.plot(
            base["tenor"],
            base["implied_vol"],
            marker="o",
            label="Implied vol"
        )

    if "current_forecast_vol" in base.columns:
        ax.plot(
            base["tenor"],
            base["current_forecast_vol"],
            marker="o",
            label="Current forecast vol"
        )

    if base["forward_window_complete_corsi"].astype(bool).all():
        ax.plot(
            base["tenor"],
            base["actual_forward_vol"],
            marker="o",
            label="Actual forward realized vol"
        )

    for model_name in CANDIDATE_MODELS:
        g = (
            date_rows_all
            .loc[date_rows_all["model_name"] == model_name]
            .sort_values("tenor")
        )

        if not g.empty:
            ax.plot(
                g["tenor"],
                g["pred_vol"],
                marker="o",
                label=f"{model_name} forecast"
            )

    ax.set_title(f"Corsi forecast curve comparison — {d.date()}")
    ax.set_xlabel("Target tenor calendar days")
    ax.set_ylabel("Volatility, annualized %")
    ax.legend()
    ax.grid(True, alpha=0.3)

    out_path = CELL7_CHART_DIR / f"corsi_forecast_curve_{d.strftime('%Y%m%d')}.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()

    print(f"Saved: {out_path}")
    return out_path


chart_paths = []

for d in rep_dates_resolved:
    p = plot_forecast_curve_for_date(d)
    if p is not None:
        chart_paths.append(p)


# -----------------------------
# 8. Latest live-style forecast curve
# -----------------------------

latest_date = curve_panel["date"].max()
latest_rows = curve_panel.loc[curve_panel["date"] == latest_date].copy()

latest_curve_table = (
    latest_rows
    .pivot_table(
        index=["date", "trade_date", "tenor", "tenor_group"],
        columns="model_name",
        values="pred_vol",
        aggfunc="first",
    )
    .reset_index()
)

base_latest = (
    latest_rows
    .drop_duplicates(subset=["date", "trade_date", "tenor"])
    [[
        c for c in [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            "implied_vol",
            "current_forecast_vol",
            "actual_forward_vol",
            "forward_window_complete_corsi",
        ]
        if c in latest_rows.columns
    ]]
)

latest_curve_table = base_latest.merge(
    latest_curve_table,
    on=["date", "trade_date", "tenor", "tenor_group"],
    how="left",
    validate="one_to_one",
)

print()
print("Latest forecast curve table:")
display(latest_curve_table.sort_values("tenor"))


# -----------------------------
# 9. Save Cell 7 outputs
# -----------------------------

safe_start = curve_panel["date"].min().strftime("%Y%m%d")
safe_end = curve_panel["date"].max().strftime("%Y%m%d")

cell7_curve_panel_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_candidate_curve_panel_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell7_curve_shape_daily_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell7_curve_shape_daily_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell7_curve_shape_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell7_curve_shape_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell7_worst_curve_dates_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell7_worst_curve_dates_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell7_latest_curve_table_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell7_latest_curve_table_{latest_date.strftime('%Y%m%d')}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

curve_panel.to_parquet(cell7_curve_panel_output, index=False)
curve_shape_daily.to_csv(cell7_curve_shape_daily_output, index=False)
curve_shape_summary.to_csv(cell7_curve_shape_summary_output, index=False)
worst_curve_dates.to_csv(cell7_worst_curve_dates_output, index=False)
latest_curve_table.to_csv(cell7_latest_curve_table_output, index=False)

print()
print("Cell 7 outputs saved:")
print(f"Candidate curve panel:        {cell7_curve_panel_output}")
print(f"Curve shape daily:            {cell7_curve_shape_daily_output}")
print(f"Curve shape summary:          {cell7_curve_shape_summary_output}")
print(f"Worst curve dates:            {cell7_worst_curve_dates_output}")
print(f"Latest curve table:           {cell7_latest_curve_table_output}")
print(f"Chart directory:              {CELL7_CHART_DIR}")

print()
print("Cell 7 complete.")
print("Next step: decide whether to build VRP using har_total_simple, har_implied_shrinkage, or both as parallel forecast denominators.")

## 8. Realized/implied ratio models - rejected

Tests ratio-style alternatives that forecast realized variance relative to implied variance. These did not improve on the robust HAR candidates and are retained only as diagnostics.

In [ ]:
# Cell 8: Realized / implied ratio Corsi model
#
# Purpose:
# Improve on har_implied_shrinkage by forecasting:
#
#   log(forward realized variance / current implied variance)
#
# instead of forecasting log(forward realized variance) directly.
#
# Final forecast:
#
#   forecast_log_realized_variance =
#       log_implied_variance + predicted_log_realized_to_implied_ratio
#
# Candidate models:
#   1. ratio_base
#      Uses realized-vol state, return/drawdown, IV curve shape, tenor.
#
#   2. ratio_with_iv_level
#      Same as ratio_base, plus absolute implied-vol level.
#
#   3. ratio_with_iv_level_by_tenor_group
#      Fits separate models for front / middle / back.
#
# Comparison baselines:
#   - current/trailing forecast
#   - implied variance
#   - har_total_simple from Cell 6B, if available
#   - har_implied_shrinkage from Cell 6B, if available

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


# -----------------------------
# 1. Load model panel if needed
# -----------------------------

try:
    model_panel
except NameError:
    latest_model_panel_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_model_feature_panel_v1_*.parquet")
    )
    if not latest_model_panel_files:
        raise FileNotFoundError("No Corsi model feature panel found.")

    print(f"Loading model panel: {latest_model_panel_files[-1]}")
    model_panel = pd.read_parquet(latest_model_panel_files[-1])

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"


# -----------------------------
# 2. Configuration
# -----------------------------

TARGET_LOG_COL = "log_forward_realized_variance_corsi"
TARGET_VAR_COL = "forward_realized_variance_corsi"
TARGET_VOL_COL = "forward_realized_vol_corsi"

TEST_YEARS = list(range(2020, 2027))
EMBARGO_DAYS = 33

EPSILON_VARIANCE = 1e-12

RATIO_RIDGE_ALPHAS = np.array([
    0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0, 10000.0
])

MIN_TRAIN_ROWS_POOLED = 750
MIN_TRAIN_ROWS_BUCKET = 200

print("Cell 8 ratio-model configuration:")
print(f"Test years:               {TEST_YEARS}")
print(f"Embargo days:             {EMBARGO_DAYS}")
print(f"Ridge alphas:             {RATIO_RIDGE_ALPHAS}")
print(f"Min train rows pooled:    {MIN_TRAIN_ROWS_POOLED}")
print(f"Min train rows bucket:    {MIN_TRAIN_ROWS_BUCKET}")


# -----------------------------
# 3. Prepare modeling panel
# -----------------------------

mp = model_panel.copy()
mp["date"] = pd.to_datetime(mp["date"])
mp["trade_date"] = pd.to_numeric(mp["trade_date"], errors="coerce").astype(int)
mp["tenor"] = pd.to_numeric(mp["tenor"], errors="coerce").astype(int)

mp = mp.replace([np.inf, -np.inf], np.nan)

for c in [
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    TARGET_VOL_COL,
    "implied_variance",
    "forecast_variance",
    "trailing_realized_variance",
]:
    if c in mp.columns:
        mp[c] = pd.to_numeric(mp[c], errors="coerce")

if "implied_variance" not in mp.columns:
    raise ValueError("model_panel must contain implied_variance for the ratio model.")

mp["log_implied_variance"] = np.log(
    mp["implied_variance"].clip(lower=EPSILON_VARIANCE)
)

mp["target_log_realized_to_implied_ratio"] = (
    mp[TARGET_LOG_COL] - mp["log_implied_variance"]
)

mp["target_realized_to_implied_ratio"] = (
    mp[TARGET_VAR_COL] / mp["implied_variance"]
)

mp["has_complete_ratio_target"] = (
    mp["forward_window_complete_corsi"].fillna(False).astype(bool)
    & mp[TARGET_LOG_COL].notna()
    & mp[TARGET_VAR_COL].notna()
    & mp["implied_variance"].notna()
    & (mp[TARGET_VAR_COL] > 0)
    & (mp["implied_variance"] > 0)
    & mp["target_log_realized_to_implied_ratio"].notna()
)

print()
print("Ratio target summary:")
ratio_target_summary = pd.DataFrame([{
    "rows": len(mp),
    "complete_ratio_target_rows": int(mp["has_complete_ratio_target"].sum()),
    "mean_realized_to_implied_ratio": mp.loc[mp["has_complete_ratio_target"], "target_realized_to_implied_ratio"].mean(),
    "median_realized_to_implied_ratio": mp.loc[mp["has_complete_ratio_target"], "target_realized_to_implied_ratio"].median(),
    "p10_realized_to_implied_ratio": mp.loc[mp["has_complete_ratio_target"], "target_realized_to_implied_ratio"].quantile(0.10),
    "p90_realized_to_implied_ratio": mp.loc[mp["has_complete_ratio_target"], "target_realized_to_implied_ratio"].quantile(0.90),
}])
display(ratio_target_summary)


# -----------------------------
# 4. Define ratio model features
# -----------------------------

def existing(cols):
    return [c for c in cols if c in mp.columns]

tenor_group_cols = [c for c in mp.columns if c.startswith("tenor_group_")]

base_tenor_features = existing([
    "tenor_scaled_30d",
    "log_tenor",
] + tenor_group_cols)

base_tenor_features_no_group_dummies = existing([
    "tenor_scaled_30d",
    "log_tenor",
])

realized_state_features = existing([
    "log_spy_total_rv_mean_5d_ann",
    "log_spy_total_rv_mean_21d_ann",
    "log_spy_total_rv_mean_63d_ann",
    "spy_return_sum_5d",
    "spy_return_sum_21d",
    "spy_drawdown_21d",
    "spy_drawdown_63d",
])

curve_shape_features = existing([
    "iv_slope_33d_9d",
    "iv_slope_21d_9d",
    "log_iv_ratio_33d_9d",
    "log_iv_ratio_21d_9d",
    "iv_curvature_9_21_33",
])

iv_level_features = existing([
    "log_implied_variance",
    "log_iv_curve_mean",
    "log_iv_9d",
    "log_iv_21d",
    "log_iv_33d",
])

ratio_model_specs = {
    "ratio_base": {
        "fit_scope": "pooled",
        "features": (
            realized_state_features
            + curve_shape_features
            + base_tenor_features
        ),
    },

    "ratio_with_iv_level": {
        "fit_scope": "pooled",
        "features": (
            realized_state_features
            + curve_shape_features
            + iv_level_features
            + base_tenor_features
        ),
    },

    "ratio_with_iv_level_by_tenor_group": {
        "fit_scope": "by_tenor_group",
        "features": (
            realized_state_features
            + curve_shape_features
            + iv_level_features
            + base_tenor_features_no_group_dummies
        ),
    },
}

# Deduplicate features.
for model_name, spec in ratio_model_specs.items():
    seen = set()
    deduped = []

    for c in spec["features"]:
        if c not in seen:
            deduped.append(c)
            seen.add(c)

    ratio_model_specs[model_name]["features"] = deduped

print()
print("Ratio model specs:")
display(pd.DataFrame([
    {
        "model_name": model_name,
        "fit_scope": spec["fit_scope"],
        "feature_count": len(spec["features"]),
        "features": ", ".join(spec["features"]),
    }
    for model_name, spec in ratio_model_specs.items()
]))


# -----------------------------
# 5. Metric helper
# -----------------------------

def ratio_metrics_table(df, pred_log_col, pred_var_col, pred_vol_col, group_cols, model_name):
    work = df.copy()

    valid = (
        work[TARGET_LOG_COL].notna()
        & work[TARGET_VAR_COL].notna()
        & work[TARGET_VOL_COL].notna()
        & work[pred_log_col].notna()
        & work[pred_var_col].notna()
        & work[pred_vol_col].notna()
        & (work[TARGET_VAR_COL] > 0)
        & (work[pred_var_col] > 0)
    )

    work = work.loc[valid].copy()

    if work.empty:
        return pd.DataFrame()

    work["log_error"] = work[pred_log_col] - work[TARGET_LOG_COL]
    work["abs_log_error"] = work["log_error"].abs()
    work["squared_log_error"] = work["log_error"] ** 2

    work["vol_error"] = work[pred_vol_col] - work[TARGET_VOL_COL]
    work["abs_vol_error"] = work["vol_error"].abs()

    work["variance_error"] = work[pred_var_col] - work[TARGET_VAR_COL]
    work["abs_variance_error"] = work["variance_error"].abs()

    work["pred_over_actual_var"] = work[pred_var_col] / work[TARGET_VAR_COL]

    if group_cols:
        grouped = work.groupby(group_cols, dropna=False)
    else:
        grouped = [((), work)]

    rows = []

    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "model_name": model_name,
            "rows": len(g),
            "mean_actual_vol": g[TARGET_VOL_COL].mean(),
            "mean_pred_vol": g[pred_vol_col].mean(),
            "median_actual_vol": g[TARGET_VOL_COL].median(),
            "median_pred_vol": g[pred_vol_col].median(),
            "mae_log_var": g["abs_log_error"].mean(),
            "rmse_log_var": np.sqrt(g["squared_log_error"].mean()),
            "bias_log_var": g["log_error"].mean(),
            "mae_vol_points": g["abs_vol_error"].mean(),
            "bias_vol_points": g["vol_error"].mean(),
            "mae_variance": g["abs_variance_error"].mean(),
            "bias_variance": g["variance_error"].mean(),
            "median_pred_over_actual_var": g["pred_over_actual_var"].median(),
            "mean_pred_over_actual_var": g["pred_over_actual_var"].mean(),
            "corr_log_var": g[pred_log_col].corr(g[TARGET_LOG_COL]) if len(g) >= 10 else np.nan,
            "corr_var": g[pred_var_col].corr(g[TARGET_VAR_COL]) if len(g) >= 10 else np.nan,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        rows.append(row)

    return pd.DataFrame(rows)


# -----------------------------
# 6. Walk-forward ratio model fit
# -----------------------------

ratio_pred_frames = []
ratio_fit_rows = []
ratio_coef_frames = []

for model_name, spec in ratio_model_specs.items():
    feature_cols = spec["features"]
    fit_scope = spec["fit_scope"]

    print()
    print("=" * 100)
    print(f"Fitting ratio model: {model_name}")
    print(f"Fit scope: {fit_scope}")
    print(f"Feature count: {len(feature_cols)}")
    print("=" * 100)

    if len(feature_cols) == 0:
        print(f"Skipping {model_name}: no features found.")
        continue

    feature_complete = (
        mp[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .notna()
        .all(axis=1)
    )

    base_row_mask = mp["has_complete_ratio_target"] & feature_complete

    for test_year in TEST_YEARS:
        test_start = pd.Timestamp(f"{test_year}-01-01")
        test_end = pd.Timestamp(f"{test_year}-12-31")
        train_cutoff = test_start - pd.Timedelta(days=EMBARGO_DAYS)

        if fit_scope == "pooled":
            scope_values = ["ALL"]
        elif fit_scope == "by_tenor_group":
            scope_values = sorted(mp["tenor_group"].dropna().astype(str).unique().tolist())
        else:
            raise ValueError(f"Unknown fit_scope: {fit_scope}")

        for scope_value in scope_values:
            if fit_scope == "pooled":
                scope_mask = pd.Series(True, index=mp.index)
                min_train_rows = MIN_TRAIN_ROWS_POOLED
            else:
                scope_mask = mp["tenor_group"].astype(str).eq(scope_value)
                min_train_rows = MIN_TRAIN_ROWS_BUCKET

            train_mask = (
                base_row_mask
                & scope_mask
                & (mp["date"] <= train_cutoff)
            )

            test_mask = (
                base_row_mask
                & scope_mask
                & (mp["date"] >= test_start)
                & (mp["date"] <= test_end)
            )

            train_df = mp.loc[train_mask].copy()
            test_df = mp.loc[test_mask].copy()

            if len(train_df) < min_train_rows or len(test_df) == 0:
                ratio_fit_rows.append({
                    "model_name": model_name,
                    "fit_scope": fit_scope,
                    "scope_value": scope_value,
                    "test_year": test_year,
                    "status": "SKIPPED",
                    "train_rows": len(train_df),
                    "test_rows": len(test_df),
                    "reason": "insufficient train rows or zero test rows",
                })

                print(
                    f"{test_year} / {scope_value}: SKIPPED "
                    f"(train={len(train_df):,}, test={len(test_df):,})"
                )

                continue

            X_train = train_df[feature_cols]
            y_train = train_df["target_log_realized_to_implied_ratio"]
            X_test = test_df[feature_cols]

            pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("ridge", RidgeCV(alphas=RATIO_RIDGE_ALPHAS)),
            ])

            pipe.fit(X_train, y_train)

            pred_ratio_log_raw = pipe.predict(X_test)

            # Clip residual/ratio forecast, not final realized variance.
            # Use train target distribution only.
            train_q_low = y_train.quantile(0.005)
            train_q_high = y_train.quantile(0.995)

            pred_ratio_log_floor = train_q_low - 0.25
            pred_ratio_log_cap = train_q_high + 0.50

            pred_ratio_log = np.clip(
                pred_ratio_log_raw,
                pred_ratio_log_floor,
                pred_ratio_log_cap,
            )

            pred_log_var = test_df["log_implied_variance"].to_numpy() + pred_ratio_log
            pred_var = np.exp(pred_log_var)
            pred_vol = np.sqrt(pred_var) * 100

            out_cols = [
                "date",
                "trade_date",
                "tenor",
                "tenor_group",
                TARGET_LOG_COL,
                TARGET_VAR_COL,
                TARGET_VOL_COL,
                "target_log_realized_to_implied_ratio",
                "target_realized_to_implied_ratio",
                "implied_variance",
                "log_implied_variance",
                "vix_style_vol",
                "forecast_variance",
                "forecast_vol",
                "trailing_realized_variance",
                "trailing_realized_vol",
                "forward_window_complete_corsi",
                "forward_num_sessions_corsi",
                "forward_review_session_count_corsi",
                "vrp_log",
                "vrp_z_3m",
                "vrp_z_1y",
            ]

            out_cols = [c for c in out_cols if c in test_df.columns]

            pred_df = test_df[out_cols].copy()
            pred_df["model_name"] = model_name
            pred_df["fit_scope"] = fit_scope
            pred_df["scope_value"] = scope_value
            pred_df["test_year"] = test_year
            pred_df["train_rows"] = len(train_df)
            pred_df["test_rows"] = len(test_df)
            pred_df["train_start_date"] = train_df["date"].min()
            pred_df["train_end_date"] = train_df["date"].max()
            pred_df["train_cutoff_date"] = train_cutoff
            pred_df["ridge_alpha"] = pipe.named_steps["ridge"].alpha_
            pred_df["feature_count"] = len(feature_cols)

            pred_df["pred_ratio_log_raw"] = pred_ratio_log_raw
            pred_df["pred_ratio_log_floor"] = pred_ratio_log_floor
            pred_df["pred_ratio_log_cap"] = pred_ratio_log_cap
            pred_df["pred_ratio_log_clipped_flag"] = pred_ratio_log_raw != pred_ratio_log

            pred_df["pred_log_realized_to_implied_ratio"] = pred_ratio_log
            pred_df["pred_realized_to_implied_ratio"] = np.exp(pred_ratio_log)

            pred_df["ratio_model_pred_log_var"] = pred_log_var
            pred_df["ratio_model_pred_var"] = pred_var
            pred_df["ratio_model_pred_vol"] = pred_vol

            pred_df["ratio_model_log_error"] = (
                pred_df["ratio_model_pred_log_var"]
                - pred_df[TARGET_LOG_COL]
            )

            pred_df["ratio_model_abs_log_error"] = pred_df["ratio_model_log_error"].abs()

            pred_df["ratio_model_vol_error"] = (
                pred_df["ratio_model_pred_vol"]
                - pred_df[TARGET_VOL_COL]
            )

            pred_df["ratio_model_abs_vol_error"] = pred_df["ratio_model_vol_error"].abs()

            ratio_pred_frames.append(pred_df)

            ratio_fit_rows.append({
                "model_name": model_name,
                "fit_scope": fit_scope,
                "scope_value": scope_value,
                "test_year": test_year,
                "status": "FIT",
                "train_rows": len(train_df),
                "test_rows": len(test_df),
                "train_start_date": train_df["date"].min(),
                "train_end_date": train_df["date"].max(),
                "train_cutoff_date": train_cutoff,
                "test_start_date": test_df["date"].min(),
                "test_end_date": test_df["date"].max(),
                "feature_count": len(feature_cols),
                "ridge_alpha": pipe.named_steps["ridge"].alpha_,
                "pred_ratio_log_floor": pred_ratio_log_floor,
                "pred_ratio_log_cap": pred_ratio_log_cap,
                "clipped_predictions": int((pred_ratio_log_raw != pred_ratio_log).sum()),
                "clipped_prediction_pct": float((pred_ratio_log_raw != pred_ratio_log).mean()),
            })

            coef_df = pd.DataFrame({
                "model_name": model_name,
                "fit_scope": fit_scope,
                "scope_value": scope_value,
                "test_year": test_year,
                "feature_name": feature_cols,
                "standardized_coefficient": pipe.named_steps["ridge"].coef_,
                "abs_standardized_coefficient": np.abs(pipe.named_steps["ridge"].coef_),
            }).sort_values("abs_standardized_coefficient", ascending=False)

            ratio_coef_frames.append(coef_df)

            print(
                f"{test_year} / {scope_value}: FIT "
                f"(train={len(train_df):,}, test={len(test_df):,}, "
                f"alpha={pipe.named_steps['ridge'].alpha_}, "
                f"clipped={int((pred_ratio_log_raw != pred_ratio_log).sum()):,})"
            )

if not ratio_pred_frames:
    raise RuntimeError("No ratio-model predictions were generated.")

ratio_model_predictions = pd.concat(ratio_pred_frames, ignore_index=True)
ratio_model_fit_audit = pd.DataFrame(ratio_fit_rows)

ratio_model_coefficients = (
    pd.concat(ratio_coef_frames, ignore_index=True)
    if ratio_coef_frames
    else pd.DataFrame()
)

print()
print("Ratio model fit audit:")
display(ratio_model_fit_audit)


# -----------------------------
# 7. Add baselines
# -----------------------------

pred = ratio_model_predictions.copy()

if "forecast_variance" in pred.columns:
    pred["baseline_current_forecast_var"] = pd.to_numeric(pred["forecast_variance"], errors="coerce")
    pred["baseline_current_forecast_log_var"] = np.log(
        pred["baseline_current_forecast_var"].clip(lower=EPSILON_VARIANCE)
    )
    pred["baseline_current_forecast_vol"] = np.sqrt(pred["baseline_current_forecast_var"]) * 100

if "trailing_realized_variance" in pred.columns:
    pred["baseline_trailing_rv_var"] = pd.to_numeric(pred["trailing_realized_variance"], errors="coerce")
    pred["baseline_trailing_rv_log_var"] = np.log(
        pred["baseline_trailing_rv_var"].clip(lower=EPSILON_VARIANCE)
    )
    pred["baseline_trailing_rv_vol"] = np.sqrt(pred["baseline_trailing_rv_var"]) * 100

pred["baseline_implied_var"] = pd.to_numeric(pred["implied_variance"], errors="coerce")
pred["baseline_implied_log_var"] = np.log(
    pred["baseline_implied_var"].clip(lower=EPSILON_VARIANCE)
)
pred["baseline_implied_vol"] = np.sqrt(pred["baseline_implied_var"]) * 100

ratio_model_predictions = pred.copy()


# -----------------------------
# 8. Optional join to Cell 6B HAR predictions
# -----------------------------

baseline_6b_joined = False

try:
    latest_6b_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_simple_har_predictions_v1_*.parquet")
    )

    if latest_6b_files:
        print()
        print(f"Loading 6B predictions for comparison: {latest_6b_files[-1]}")

        pred_6b = pd.read_parquet(latest_6b_files[-1])
        pred_6b["date"] = pd.to_datetime(pred_6b["date"])
        pred_6b["trade_date"] = pd.to_numeric(pred_6b["trade_date"], errors="coerce").astype(int)
        pred_6b["tenor"] = pd.to_numeric(pred_6b["tenor"], errors="coerce").astype(int)

        pred_6b_small = pred_6b.loc[
            pred_6b["model_name"].isin(["har_total_simple", "har_implied_shrinkage"])
        ][
            [
                "date",
                "trade_date",
                "tenor",
                "model_name",
                "pred_forward_variance_corsi_6b",
                "pred_forward_vol_corsi_6b",
            ]
        ].copy()

        pred_6b_wide_var = (
            pred_6b_small
            .pivot_table(
                index=["date", "trade_date", "tenor"],
                columns="model_name",
                values="pred_forward_variance_corsi_6b",
                aggfunc="first",
            )
            .reset_index()
        )

        pred_6b_wide_vol = (
            pred_6b_small
            .pivot_table(
                index=["date", "trade_date", "tenor"],
                columns="model_name",
                values="pred_forward_vol_corsi_6b",
                aggfunc="first",
            )
            .reset_index()
        )

        pred_6b_wide_var = pred_6b_wide_var.rename(columns={
            "har_total_simple": "baseline_har_total_simple_var",
            "har_implied_shrinkage": "baseline_har_implied_shrinkage_var",
        })

        pred_6b_wide_vol = pred_6b_wide_vol.rename(columns={
            "har_total_simple": "baseline_har_total_simple_vol",
            "har_implied_shrinkage": "baseline_har_implied_shrinkage_vol",
        })

        pred_6b_wide = pred_6b_wide_var.merge(
            pred_6b_wide_vol,
            on=["date", "trade_date", "tenor"],
            how="outer",
            validate="one_to_one",
        )

        ratio_model_predictions = ratio_model_predictions.merge(
            pred_6b_wide,
            on=["date", "trade_date", "tenor"],
            how="left",
            validate="many_to_one",
        )

        for base in ["baseline_har_total_simple", "baseline_har_implied_shrinkage"]:
            var_col = f"{base}_var"
            log_col = f"{base}_log_var"

            if var_col in ratio_model_predictions.columns:
                ratio_model_predictions[log_col] = np.log(
                    ratio_model_predictions[var_col].clip(lower=EPSILON_VARIANCE)
                )

        baseline_6b_joined = True

except Exception as e:
    print()
    print(f"Could not join 6B HAR predictions. Continuing without them. Error: {e}")


# -----------------------------
# 9. Metrics
# -----------------------------

metric_frames = []

# Ratio model metrics.
for group_cols in [
    [],
    ["model_name"],
    ["model_name", "tenor"],
    ["model_name", "test_year"],
    ["model_name", "test_year", "tenor"],
]:
    metric_frames.append(
        ratio_metrics_table(
            ratio_model_predictions,
            "ratio_model_pred_log_var",
            "ratio_model_pred_var",
            "ratio_model_pred_vol",
            group_cols,
            "ratio_model",
        )
    )

# Dedup baselines by date x tenor.
baseline_eval = (
    ratio_model_predictions
    .drop_duplicates(subset=["date", "trade_date", "tenor"])
    .copy()
)

baseline_specs = [
    (
        "baseline_current_forecast",
        "baseline_current_forecast_log_var",
        "baseline_current_forecast_var",
        "baseline_current_forecast_vol",
    ),
    (
        "baseline_trailing_rv",
        "baseline_trailing_rv_log_var",
        "baseline_trailing_rv_var",
        "baseline_trailing_rv_vol",
    ),
    (
        "baseline_implied_variance",
        "baseline_implied_log_var",
        "baseline_implied_var",
        "baseline_implied_vol",
    ),
]

if baseline_6b_joined:
    baseline_specs += [
        (
            "baseline_har_total_simple_6b",
            "baseline_har_total_simple_log_var",
            "baseline_har_total_simple_var",
            "baseline_har_total_simple_vol",
        ),
        (
            "baseline_har_implied_shrinkage_6b",
            "baseline_har_implied_shrinkage_log_var",
            "baseline_har_implied_shrinkage_var",
            "baseline_har_implied_shrinkage_vol",
        ),
    ]

for label, log_col, var_col, vol_col in baseline_specs:
    if log_col in baseline_eval.columns and var_col in baseline_eval.columns and vol_col in baseline_eval.columns:
        for group_cols in [[], ["tenor"], ["test_year"]]:
            metric_frames.append(
                ratio_metrics_table(
                    baseline_eval,
                    log_col,
                    var_col,
                    vol_col,
                    group_cols,
                    label,
                )
            )

metric_frames = [m for m in metric_frames if not m.empty]
ratio_model_metrics = pd.concat(metric_frames, ignore_index=True)

overall_ratio_metrics = (
    ratio_model_metrics
    .loc[
        ratio_model_metrics.get("tenor", pd.Series(index=ratio_model_metrics.index, dtype=float)).isna()
        & ratio_model_metrics.get("test_year", pd.Series(index=ratio_model_metrics.index, dtype=float)).isna()
    ]
    .copy()
    .sort_values("mae_log_var")
)

by_tenor_ratio_metrics = (
    ratio_model_metrics
    .loc[
        ratio_model_metrics.get("tenor", pd.Series(index=ratio_model_metrics.index, dtype=float)).notna()
        & ratio_model_metrics.get("test_year", pd.Series(index=ratio_model_metrics.index, dtype=float)).isna()
    ]
    .copy()
    .sort_values(["model_name", "tenor"])
)

by_year_ratio_metrics = (
    ratio_model_metrics
    .loc[
        ratio_model_metrics.get("test_year", pd.Series(index=ratio_model_metrics.index, dtype=float)).notna()
        & ratio_model_metrics.get("tenor", pd.Series(index=ratio_model_metrics.index, dtype=float)).isna()
    ]
    .copy()
    .sort_values(["model_name", "test_year"])
)

print()
print("Cell 8 overall metrics, sorted by MAE log variance:")
display(
    overall_ratio_metrics[
        [c for c in [
            "model_name",
            "rows",
            "mean_actual_vol",
            "mean_pred_vol",
            "median_actual_vol",
            "median_pred_vol",
            "mae_log_var",
            "rmse_log_var",
            "bias_log_var",
            "mae_vol_points",
            "bias_vol_points",
            "corr_log_var",
            "corr_var",
        ] if c in overall_ratio_metrics.columns]
    ]
)

print()
print("Cell 8 metrics by tenor:")
display(
    by_tenor_ratio_metrics[
        [c for c in [
            "model_name",
            "tenor",
            "rows",
            "mae_log_var",
            "rmse_log_var",
            "bias_log_var",
            "mae_vol_points",
            "bias_vol_points",
            "corr_log_var",
        ] if c in by_tenor_ratio_metrics.columns]
    ]
)

print()
print("Cell 8 metrics by test year:")
display(
    by_year_ratio_metrics[
        [c for c in [
            "model_name",
            "test_year",
            "rows",
            "mae_log_var",
            "rmse_log_var",
            "bias_log_var",
            "mae_vol_points",
            "bias_vol_points",
            "corr_log_var",
        ] if c in by_year_ratio_metrics.columns]
    ]
)


# -----------------------------
# 10. Diagnostics
# -----------------------------

worst_ratio_errors = (
    ratio_model_predictions
    .sort_values("ratio_model_abs_log_error", ascending=False)
    .head(75)
    .copy()
)

latest_ratio_predictions = (
    ratio_model_predictions
    .sort_values(["date", "tenor", "model_name"])
    .tail(90)
    .copy()
)

print()
print("Cell 8 worst ratio-model log-variance errors:")
display(
    worst_ratio_errors[
        [c for c in [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            "model_name",
            TARGET_VOL_COL,
            "ratio_model_pred_vol",
            "ratio_model_vol_error",
            TARGET_VAR_COL,
            "ratio_model_pred_var",
            "target_realized_to_implied_ratio",
            "pred_realized_to_implied_ratio",
            "ratio_model_log_error",
            "ratio_model_abs_log_error",
            "test_year",
            "ridge_alpha",
            "pred_ratio_log_clipped_flag",
            "forward_review_session_count_corsi",
        ] if c in worst_ratio_errors.columns]
    ]
)

print()
print("Cell 8 latest ratio-model predictions:")
display(
    latest_ratio_predictions[
        [c for c in [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            "model_name",
            TARGET_VOL_COL,
            "ratio_model_pred_vol",
            "ratio_model_vol_error",
            "target_realized_to_implied_ratio",
            "pred_realized_to_implied_ratio",
            "test_year",
            "ridge_alpha",
            "pred_ratio_log_clipped_flag",
        ] if c in latest_ratio_predictions.columns]
    ]
)

print()
print("Top coefficients for ratio models:")
if ratio_model_coefficients.empty:
    print("No coefficient table.")
else:
    display(
        ratio_model_coefficients
        .groupby(["model_name", "scope_value", "test_year"])
        .head(10)
        .reset_index(drop=True)
    )


# -----------------------------
# 11. Save outputs
# -----------------------------

safe_start = ratio_model_predictions["date"].min().strftime("%Y%m%d")
safe_end = ratio_model_predictions["date"].max().strftime("%Y%m%d")

cell8_predictions_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_ratio_model_predictions_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell8_predictions_sample_output = (
    CORSI_AUDIT_DIR
    / f"corsi_ratio_model_predictions_v1_sample_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell8_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell8_ratio_model_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell8_overall_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell8_overall_ratio_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell8_by_tenor_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell8_by_tenor_ratio_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell8_by_year_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell8_by_year_ratio_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell8_fit_audit_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell8_ratio_model_fit_audit_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell8_coefficients_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell8_ratio_model_coefficients_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell8_worst_errors_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell8_worst_ratio_model_errors_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

ratio_model_predictions.to_parquet(cell8_predictions_output, index=False)
ratio_model_predictions.head(1000).to_csv(cell8_predictions_sample_output, index=False)

ratio_model_metrics.to_csv(cell8_metrics_output, index=False)
overall_ratio_metrics.to_csv(cell8_overall_metrics_output, index=False)
by_tenor_ratio_metrics.to_csv(cell8_by_tenor_metrics_output, index=False)
by_year_ratio_metrics.to_csv(cell8_by_year_metrics_output, index=False)

ratio_model_fit_audit.to_csv(cell8_fit_audit_output, index=False)
ratio_model_coefficients.to_csv(cell8_coefficients_output, index=False)
worst_ratio_errors.to_csv(cell8_worst_errors_output, index=False)

print()
print("Cell 8 outputs saved:")
print(f"Ratio model predictions:       {cell8_predictions_output}")
print(f"Prediction sample:             {cell8_predictions_sample_output}")
print(f"All metrics:                   {cell8_metrics_output}")
print(f"Overall metrics:               {cell8_overall_metrics_output}")
print(f"By-tenor metrics:              {cell8_by_tenor_metrics_output}")
print(f"By-year metrics:               {cell8_by_year_metrics_output}")
print(f"Fit audit:                     {cell8_fit_audit_output}")
print(f"Coefficients:                  {cell8_coefficients_output}")
print(f"Worst errors:                  {cell8_worst_errors_output}")

print()
print("Cell 8 complete.")
print("Next step: compare ratio models against har_implied_shrinkage and decide whether the ratio formulation is better.")

## 9. Final fitted model scores and VRP panel

Fits final models on all complete history and scores the full feature-complete panel, including latest live-style dates. This is appropriate for current/live scoring diagnostics, but **not** for historical model selection because it uses all available history.

In [ ]:
# Cell 9: Final Corsi forecast panel + VRP measures
#
# Primary model:
#   har_implied_shrinkage
#
# Control model:
#   har_total_simple
#
# Purpose:
#   1. Fit final models on all complete forward-target history.
#   2. Score every feature-complete date x tenor row, including latest live-style rows.
#   3. Build forecast variance / forecast vol by candidate model.
#   4. Build VRP measures:
#        vrp_log_model = log(implied_variance / forecast_variance_model)
#   5. Build rolling 3m and 1y z-scores by tenor.
#
# This cell does NOT overwrite production files.

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt


# -----------------------------
# 1. Load model panel if needed
# -----------------------------

try:
    model_panel
except NameError:
    latest_model_panel_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_model_feature_panel_v1_*.parquet")
    )
    if not latest_model_panel_files:
        raise FileNotFoundError("No Corsi model feature panel found.")
    print(f"Loading model panel: {latest_model_panel_files[-1]}")
    model_panel = pd.read_parquet(latest_model_panel_files[-1])

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"


# -----------------------------
# 2. Configuration
# -----------------------------

TARGET_LOG_COL = "log_forward_realized_variance_corsi"
TARGET_VAR_COL = "forward_realized_variance_corsi"
TARGET_VOL_COL = "forward_realized_vol_corsi"

EPSILON_VARIANCE = 1e-12

FINAL_RIDGE_ALPHAS = np.array([
    0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0, 10000.0
])

ROLLING_Z_WINDOWS = {
    "3m": {
        "window": 63,
        "min_periods": 30,
    },
    "1y": {
        "window": 252,
        "min_periods": 126,
    },
}

PRIMARY_MODEL = "har_implied_shrinkage"
CONTROL_MODEL = "har_total_simple"

FINAL_MODELS = [
    PRIMARY_MODEL,
    CONTROL_MODEL,
]

CELL9_CHART_DIR = (
    CORSI_AUDIT_DIR
    / "charts"
    / "cell9_final_corsi_vrp"
)

CELL9_CHART_DIR.mkdir(parents=True, exist_ok=True)

print("Cell 9 configuration:")
print(f"Primary model:       {PRIMARY_MODEL}")
print(f"Control model:       {CONTROL_MODEL}")
print(f"Ridge alphas:        {FINAL_RIDGE_ALPHAS}")
print(f"Chart dir:           {CELL9_CHART_DIR}")


# -----------------------------
# 3. Prepare panel
# -----------------------------

mp = model_panel.copy()
mp["date"] = pd.to_datetime(mp["date"])
mp["trade_date"] = pd.to_numeric(mp["trade_date"], errors="coerce").astype(int)
mp["tenor"] = pd.to_numeric(mp["tenor"], errors="coerce").astype(int)

mp = mp.replace([np.inf, -np.inf], np.nan)

for c in [
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    TARGET_VOL_COL,
    "implied_variance",
    "forecast_variance",
    "trailing_realized_variance",
]:
    if c in mp.columns:
        mp[c] = pd.to_numeric(mp[c], errors="coerce")

mp["log_implied_variance"] = np.log(
    mp["implied_variance"].clip(lower=EPSILON_VARIANCE)
)

mp["has_complete_target_final"] = (
    mp["forward_window_complete_corsi"].fillna(False).astype(bool)
    & mp[TARGET_LOG_COL].notna()
    & mp[TARGET_VAR_COL].notna()
    & (mp[TARGET_VAR_COL] > 0)
)


# -----------------------------
# 4. Define final model features
# -----------------------------

def existing(cols):
    return [c for c in cols if c in mp.columns]

tenor_group_cols = [c for c in mp.columns if c.startswith("tenor_group_")]

base_tenor_features = existing([
    "tenor_scaled_30d",
    "log_tenor",
] + tenor_group_cols)

har_total_simple_features = existing([
    "log_spy_total_rv_1d_ann",
    "log_spy_total_rv_mean_5d_ann",
    "log_spy_total_rv_mean_21d_ann",
    "log_spy_total_rv_mean_63d_ann",
]) + base_tenor_features

har_implied_shrinkage_features = existing([
    "log_implied_variance",
    "log_iv_curve_mean",
    "log_spy_total_rv_mean_5d_ann",
    "log_spy_total_rv_mean_21d_ann",
    "log_spy_total_rv_mean_63d_ann",
    "spy_return_sum_21d",
    "spy_drawdown_63d",
]) + base_tenor_features

final_model_specs = {
    "har_implied_shrinkage": har_implied_shrinkage_features,
    "har_total_simple": har_total_simple_features,
}

# Deduplicate feature lists.
for model_name, cols in final_model_specs.items():
    seen = set()
    deduped = []
    for c in cols:
        if c not in seen:
            deduped.append(c)
            seen.add(c)
    final_model_specs[model_name] = deduped

print()
print("Final model specs:")
display(pd.DataFrame([
    {
        "model_name": model_name,
        "feature_count": len(cols),
        "features": ", ".join(cols),
    }
    for model_name, cols in final_model_specs.items()
]))


# -----------------------------
# 5. Fit final models and score all available rows
# -----------------------------

score_frames = []
fit_rows = []
coef_frames = []

for model_name in FINAL_MODELS:
    feature_cols = final_model_specs[model_name]

    feature_complete = (
        mp[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .notna()
        .all(axis=1)
    )

    train_mask = mp["has_complete_target_final"] & feature_complete
    score_mask = feature_complete

    train_df = mp.loc[train_mask].copy()
    score_df = mp.loc[score_mask].copy()

    if train_df.empty:
        raise RuntimeError(f"No training rows for {model_name}")

    if score_df.empty:
        raise RuntimeError(f"No scoring rows for {model_name}")

    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", RidgeCV(alphas=FINAL_RIDGE_ALPHAS)),
    ])

    X_train = train_df[feature_cols]
    y_train = train_df[TARGET_LOG_COL]
    X_score = score_df[feature_cols]

    pipe.fit(X_train, y_train)

    pred_log_raw = pipe.predict(X_score)

    # Clip based only on the training target distribution.
    train_q_low = y_train.quantile(0.005)
    train_q_high = y_train.quantile(0.995)

    pred_log_floor = train_q_low - 0.25
    pred_log_cap = train_q_high + 0.75

    pred_log = np.clip(pred_log_raw, pred_log_floor, pred_log_cap)
    pred_var = np.exp(pred_log)
    pred_vol = np.sqrt(pred_var) * 100

    out_cols = [
        "date",
        "trade_date",
        "tenor",
        "tenor_group",
        "implied_variance",
        "vix_style_vol",
        "forecast_variance",
        "forecast_vol",
        "trailing_realized_variance",
        "trailing_realized_vol",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "forward_window_complete_corsi",
        TARGET_LOG_COL,
        TARGET_VAR_COL,
        TARGET_VOL_COL,
    ]

    out_cols = [c for c in out_cols if c in score_df.columns]

    scored = score_df[out_cols].copy()
    scored["model_name"] = model_name
    scored["feature_count"] = len(feature_cols)
    scored["train_rows"] = len(train_df)
    scored["train_start_date"] = train_df["date"].min()
    scored["train_end_date"] = train_df["date"].max()
    scored["ridge_alpha"] = pipe.named_steps["ridge"].alpha_
    scored["pred_log_raw"] = pred_log_raw
    scored["pred_log_floor"] = pred_log_floor
    scored["pred_log_cap"] = pred_log_cap
    scored["pred_log_clipped_flag"] = pred_log_raw != pred_log
    scored["corsi_forecast_log_variance"] = pred_log
    scored["corsi_forecast_variance"] = pred_var
    scored["corsi_forecast_vol"] = pred_vol

    score_frames.append(scored)

    fit_rows.append({
        "model_name": model_name,
        "feature_count": len(feature_cols),
        "train_rows": len(train_df),
        "score_rows": len(score_df),
        "train_start_date": train_df["date"].min(),
        "train_end_date": train_df["date"].max(),
        "score_start_date": score_df["date"].min(),
        "score_end_date": score_df["date"].max(),
        "ridge_alpha": pipe.named_steps["ridge"].alpha_,
        "pred_log_floor": pred_log_floor,
        "pred_log_cap": pred_log_cap,
        "clipped_predictions": int((pred_log_raw != pred_log).sum()),
        "clipped_prediction_pct": float((pred_log_raw != pred_log).mean()),
    })

    coef_df = pd.DataFrame({
        "model_name": model_name,
        "feature_name": feature_cols,
        "standardized_coefficient": pipe.named_steps["ridge"].coef_,
        "abs_standardized_coefficient": np.abs(pipe.named_steps["ridge"].coef_),
    }).sort_values("abs_standardized_coefficient", ascending=False)

    coef_frames.append(coef_df)

final_scored_long = pd.concat(score_frames, ignore_index=True)
final_fit_audit = pd.DataFrame(fit_rows)
final_coefficients = pd.concat(coef_frames, ignore_index=True)

print()
print("Final fit audit:")
display(final_fit_audit)

print()
print("Final model coefficients:")
display(
    final_coefficients
    .groupby("model_name")
    .head(20)
    .reset_index(drop=True)
)


# -----------------------------
# 6. Convert long scored panel to wide forecast panel
# -----------------------------

base_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_group",
    "implied_variance",
    "vix_style_vol",
    "forecast_variance",
    "forecast_vol",
    "trailing_realized_variance",
    "trailing_realized_vol",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "forward_window_complete_corsi",
    TARGET_LOG_COL,
    TARGET_VAR_COL,
    TARGET_VOL_COL,
]

base_cols = [c for c in base_cols if c in final_scored_long.columns]

final_forecast_panel = (
    final_scored_long
    .drop_duplicates(subset=["date", "trade_date", "tenor"])[base_cols]
    .copy()
)

forecast_wide = (
    final_scored_long
    .pivot_table(
        index=["date", "trade_date", "tenor"],
        columns="model_name",
        values="corsi_forecast_variance",
        aggfunc="first",
    )
    .reset_index()
)

forecast_vol_wide = (
    final_scored_long
    .pivot_table(
        index=["date", "trade_date", "tenor"],
        columns="model_name",
        values="corsi_forecast_vol",
        aggfunc="first",
    )
    .reset_index()
)

forecast_wide = forecast_wide.rename(columns={
    "har_implied_shrinkage": "forecast_variance_har_implied_shrinkage",
    "har_total_simple": "forecast_variance_har_total_simple",
})

forecast_vol_wide = forecast_vol_wide.rename(columns={
    "har_implied_shrinkage": "forecast_vol_har_implied_shrinkage",
    "har_total_simple": "forecast_vol_har_total_simple",
})

final_forecast_panel = final_forecast_panel.merge(
    forecast_wide,
    on=["date", "trade_date", "tenor"],
    how="left",
    validate="one_to_one",
)

final_forecast_panel = final_forecast_panel.merge(
    forecast_vol_wide,
    on=["date", "trade_date", "tenor"],
    how="left",
    validate="one_to_one",
)


# -----------------------------
# 7. Build VRP measures and rolling z-scores
# -----------------------------

candidate_forecast_cols = [
    "forecast_variance_har_implied_shrinkage",
    "forecast_variance_har_total_simple",
]

for fcst_col in candidate_forecast_cols:
    suffix = fcst_col.replace("forecast_variance_", "")

    final_forecast_panel[f"vrp_log_{suffix}"] = np.log(
        final_forecast_panel["implied_variance"].clip(lower=EPSILON_VARIANCE)
        / final_forecast_panel[fcst_col].clip(lower=EPSILON_VARIANCE)
    )

    final_forecast_panel[f"vrp_variance_spread_{suffix}"] = (
        final_forecast_panel["implied_variance"]
        - final_forecast_panel[fcst_col]
    )

    final_forecast_panel[f"vrp_ratio_{suffix}"] = (
        final_forecast_panel["implied_variance"]
        / final_forecast_panel[fcst_col]
    )

final_forecast_panel = final_forecast_panel.sort_values(["tenor", "date"]).reset_index(drop=True)

for fcst_col in candidate_forecast_cols:
    suffix = fcst_col.replace("forecast_variance_", "")
    vrp_col = f"vrp_log_{suffix}"

    for z_name, z_cfg in ROLLING_Z_WINDOWS.items():
        window = z_cfg["window"]
        min_periods = z_cfg["min_periods"]

        rolling_mean = (
            final_forecast_panel
            .groupby("tenor")[vrp_col]
            .transform(lambda s: s.rolling(window, min_periods=min_periods).mean())
        )

        rolling_std = (
            final_forecast_panel
            .groupby("tenor")[vrp_col]
            .transform(lambda s: s.rolling(window, min_periods=min_periods).std())
        )

        final_forecast_panel[f"{vrp_col}_mean_{z_name}"] = rolling_mean
        final_forecast_panel[f"{vrp_col}_std_{z_name}"] = rolling_std

        final_forecast_panel[f"vrp_z_{z_name}_{suffix}"] = (
            (final_forecast_panel[vrp_col] - rolling_mean)
            / rolling_std.replace(0, np.nan)
        )

final_forecast_panel = final_forecast_panel.sort_values(["date", "tenor"]).reset_index(drop=True)


# -----------------------------
# 8. Diagnostics
# -----------------------------

latest_date = final_forecast_panel["date"].max()

latest_table_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_group",
    "vix_style_vol",
    "forecast_vol",
    "forecast_vol_har_implied_shrinkage",
    "forecast_vol_har_total_simple",
    "vrp_log",
    "vrp_log_har_implied_shrinkage",
    "vrp_z_3m_har_implied_shrinkage",
    "vrp_z_1y_har_implied_shrinkage",
    "vrp_log_har_total_simple",
    "vrp_z_3m_har_total_simple",
    "vrp_z_1y_har_total_simple",
]

latest_table_cols = [c for c in latest_table_cols if c in final_forecast_panel.columns]

latest_forecast_vrp_table = (
    final_forecast_panel
    .loc[final_forecast_panel["date"] == latest_date, latest_table_cols]
    .sort_values("tenor")
    .copy()
)

zscore_coverage_rows = []

for suffix in ["har_implied_shrinkage", "har_total_simple"]:
    for z_name in ROLLING_Z_WINDOWS.keys():
        z_col = f"vrp_z_{z_name}_{suffix}"

        zscore_coverage_rows.append({
            "model_suffix": suffix,
            "z_window": z_name,
            "z_col": z_col,
            "non_null_rows": int(final_forecast_panel[z_col].notna().sum()),
            "first_non_null_date": final_forecast_panel.loc[
                final_forecast_panel[z_col].notna(), "date"
            ].min(),
            "latest_non_null_date": final_forecast_panel.loc[
                final_forecast_panel[z_col].notna(), "date"
            ].max(),
        })

zscore_coverage = pd.DataFrame(zscore_coverage_rows)

print()
print(f"Latest forecast + VRP table: {latest_date.date()}")
display(latest_forecast_vrp_table)

print()
print("Z-score coverage:")
display(zscore_coverage)

print()
print("Final forecast panel summary:")
display(pd.DataFrame([{
    "rows": len(final_forecast_panel),
    "first_date": final_forecast_panel["date"].min(),
    "last_date": final_forecast_panel["date"].max(),
    "unique_dates": final_forecast_panel["date"].nunique(),
    "unique_tenors": final_forecast_panel["tenor"].nunique(),
    "latest_date_rows": len(latest_forecast_vrp_table),
}]))


# -----------------------------
# 9. Plot latest term structures
# -----------------------------

fig, ax = plt.subplots(figsize=(12, 6))

plot_df = latest_forecast_vrp_table.sort_values("tenor").copy()

if "vix_style_vol" in plot_df.columns:
    ax.plot(plot_df["tenor"], plot_df["vix_style_vol"], marker="o", label="Implied vol")

if "forecast_vol" in plot_df.columns:
    ax.plot(plot_df["tenor"], plot_df["forecast_vol"], marker="o", label="Current forecast vol")

ax.plot(
    plot_df["tenor"],
    plot_df["forecast_vol_har_implied_shrinkage"],
    marker="o",
    label="HAR implied shrinkage forecast vol",
)

ax.plot(
    plot_df["tenor"],
    plot_df["forecast_vol_har_total_simple"],
    marker="o",
    label="HAR total simple forecast vol",
)

ax.set_title(f"Latest forecast-vol term structure — {latest_date.date()}")
ax.set_xlabel("Tenor, calendar days")
ax.set_ylabel("Annualized volatility, %")
ax.grid(True, alpha=0.3)
ax.legend()

latest_forecast_chart = CELL9_CHART_DIR / f"latest_forecast_vol_term_structure_{latest_date.strftime('%Y%m%d')}.png"

plt.tight_layout()
plt.savefig(latest_forecast_chart, dpi=150)
plt.show()

print(f"Saved chart: {latest_forecast_chart}")


fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    plot_df["tenor"],
    plot_df["vrp_z_1y_har_implied_shrinkage"],
    marker="o",
    label="1Y VRP z-score — HAR implied shrinkage",
)

ax.plot(
    plot_df["tenor"],
    plot_df["vrp_z_1y_har_total_simple"],
    marker="o",
    label="1Y VRP z-score — HAR total simple",
)

if "vrp_z_1y" in plot_df.columns:
    ax.plot(
        plot_df["tenor"],
        plot_df["vrp_z_1y"],
        marker="o",
        label="1Y VRP z-score — current",
    )

ax.axhline(0, linewidth=1)
ax.set_title(f"Latest 1Y VRP z-score term structure — {latest_date.date()}")
ax.set_xlabel("Tenor, calendar days")
ax.set_ylabel("Z-score")
ax.grid(True, alpha=0.3)
ax.legend()

latest_zscore_chart = CELL9_CHART_DIR / f"latest_vrp_zscore_term_structure_{latest_date.strftime('%Y%m%d')}.png"

plt.tight_layout()
plt.savefig(latest_zscore_chart, dpi=150)
plt.show()

print(f"Saved chart: {latest_zscore_chart}")


# -----------------------------
# 10. Save outputs
# -----------------------------

safe_start = final_forecast_panel["date"].min().strftime("%Y%m%d")
safe_end = final_forecast_panel["date"].max().strftime("%Y%m%d")

cell9_long_scores_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_final_model_scores_long_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell9_final_forecast_panel_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_final_forecast_vrp_panel_v1_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell9_latest_table_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell9_latest_forecast_vrp_table_{latest_date.strftime('%Y%m%d')}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell9_fit_audit_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell9_final_fit_audit_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell9_coefficients_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell9_final_coefficients_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell9_zscore_coverage_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell9_zscore_coverage_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

final_scored_long.to_parquet(cell9_long_scores_output, index=False)
final_forecast_panel.to_parquet(cell9_final_forecast_panel_output, index=False)
latest_forecast_vrp_table.to_csv(cell9_latest_table_output, index=False)
final_fit_audit.to_csv(cell9_fit_audit_output, index=False)
final_coefficients.to_csv(cell9_coefficients_output, index=False)
zscore_coverage.to_csv(cell9_zscore_coverage_output, index=False)

print()
print("Cell 9 outputs saved:")
print(f"Long model scores:          {cell9_long_scores_output}")
print(f"Final forecast VRP panel:   {cell9_final_forecast_panel_output}")
print(f"Latest forecast VRP table:  {cell9_latest_table_output}")
print(f"Fit audit:                  {cell9_fit_audit_output}")
print(f"Coefficients:               {cell9_coefficients_output}")
print(f"Z-score coverage:           {cell9_zscore_coverage_output}")
print(f"Chart directory:            {CELL9_CHART_DIR}")

print()
print("Cell 9 complete.")
print("Next step: compare trade selection using current VRP vs har_implied_shrinkage VRP.")

## 10. Initial trade-selection comparison - superseded

This diagnostic compared signal rankings but mixed score definitions. It is retained for audit only and is superseded by Cell 10B.

In [ ]:
# Cell 10: Compare trade selection using current VRP vs HAR-implied-shrinkage VRP
#
# Purpose:
# Compare current production VRP ranking / trade selection against:
#   1. har_implied_shrinkage VRP
#   2. har_total_simple VRP control
#
# This cell does not run a P&L backtest yet.
# It answers:
#   - How often does the top tenor change?
#   - Does the model shift selection front/middle/back?
#   - How similar are daily tenor rankings?
#   - What are the calibrated top-N trade candidates under each method?
#
# Next step after this is a true trade backtest using the new selected trades.

from pathlib import Path
import numpy as np
import pandas as pd


# -----------------------------
# 1. Load final forecast VRP panel if needed
# -----------------------------

try:
    final_forecast_panel
except NameError:
    latest_cell9_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_final_forecast_vrp_panel_v1_*.parquet")
    )
    if not latest_cell9_files:
        raise FileNotFoundError("No Cell 9 final forecast VRP panel found.")
    print(f"Loading Cell 9 final forecast VRP panel: {latest_cell9_files[-1]}")
    final_forecast_panel = pd.read_parquet(latest_cell9_files[-1])

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "PRODUCTION_FEATURE_PANEL_PATH" not in globals():
    PRODUCTION_FEATURE_PANEL_PATH = (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "production_feature_panel_v0_1.parquet"
    )


# -----------------------------
# 2. Prepare comparison panel
# -----------------------------

panel = final_forecast_panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

# Pull extra production signal columns if available.
if PRODUCTION_FEATURE_PANEL_PATH.exists():
    prod = pd.read_parquet(PRODUCTION_FEATURE_PANEL_PATH)
    prod["date"] = pd.to_datetime(prod["date"])
    prod["trade_date"] = pd.to_numeric(prod["trade_date"], errors="coerce").astype(int)
    prod["tenor"] = pd.to_numeric(prod["tenor"], errors="coerce").astype(int)

    signal_like_cols = [
        c for c in prod.columns
        if any(
            token in c.lower()
            for token in [
                "signal",
                "selected",
                "trade",
                "rank",
                "tier",
                "eligible",
                "core",
                "secondary",
            ]
        )
    ]

    extra_cols = [
        c for c in signal_like_cols
        if c not in panel.columns
    ]

    merge_cols = ["date", "trade_date", "tenor"] + extra_cols

    prod_extra = (
        prod[merge_cols]
        .drop_duplicates(subset=["date", "trade_date", "tenor"])
        .copy()
    )

    panel = panel.merge(
        prod_extra,
        on=["date", "trade_date", "tenor"],
        how="left",
        validate="one_to_one",
    )

    print("Loaded production feature panel extra signal-like columns:")
    display(pd.DataFrame({"column": extra_cols}))
else:
    print(f"Production feature panel not found: {PRODUCTION_FEATURE_PANEL_PATH}")


# -----------------------------
# 3. Infer current signal formula if primary_vrp_signal exists
# -----------------------------

def build_candidate_signal_formulas(df, z3_col, z1_col):
    out = pd.DataFrame(index=df.index)
    out["z3"] = df[z3_col]
    out["z1"] = df[z1_col]
    out["avg_z3_z1"] = 0.5 * (df[z3_col] + df[z1_col])
    out["min_z3_z1"] = np.minimum(df[z3_col], df[z1_col])
    out["max_z3_z1"] = np.maximum(df[z3_col], df[z1_col])
    out["weighted_25_75"] = 0.25 * df[z3_col] + 0.75 * df[z1_col]
    out["weighted_75_25"] = 0.75 * df[z3_col] + 0.25 * df[z1_col]
    return out


formula_fit_summary = pd.DataFrame()
selected_current_formula = None

if "primary_vrp_signal" in panel.columns:
    candidates = build_candidate_signal_formulas(panel, "vrp_z_3m", "vrp_z_1y")
    work = pd.concat(
        [
            panel[["primary_vrp_signal"]],
            candidates,
        ],
        axis=1,
    ).replace([np.inf, -np.inf], np.nan).dropna()

    rows = []

    for c in candidates.columns:
        diff = work[c] - work["primary_vrp_signal"]

        rows.append({
            "formula": c,
            "rows": len(work),
            "mae_vs_primary_vrp_signal": diff.abs().mean(),
            "rmse_vs_primary_vrp_signal": np.sqrt((diff ** 2).mean()),
            "max_abs_diff": diff.abs().max(),
            "corr_vs_primary_vrp_signal": work[c].corr(work["primary_vrp_signal"]),
        })

    formula_fit_summary = pd.DataFrame(rows).sort_values("rmse_vs_primary_vrp_signal")
    selected_current_formula = formula_fit_summary.iloc[0]["formula"]

    print()
    print("Formula inference vs production primary_vrp_signal:")
    display(formula_fit_summary)

    print(f"Selected formula for new-model signal comparison: {selected_current_formula}")
else:
    selected_current_formula = "avg_z3_z1"
    print()
    print("primary_vrp_signal not found. Falling back to avg_z3_z1.")


def apply_selected_formula(df, z3_col, z1_col, formula):
    formulas = build_candidate_signal_formulas(df, z3_col, z1_col)

    if formula not in formulas.columns:
        raise ValueError(f"Unknown formula: {formula}")

    return formulas[formula]


# Current score.
if "primary_vrp_signal" in panel.columns:
    panel["score_current"] = pd.to_numeric(panel["primary_vrp_signal"], errors="coerce")
else:
    panel["score_current"] = apply_selected_formula(
        panel,
        "vrp_z_3m",
        "vrp_z_1y",
        selected_current_formula,
    )

# New model scores using same signal formula.
panel["score_har_implied_shrinkage"] = apply_selected_formula(
    panel,
    "vrp_z_3m_har_implied_shrinkage",
    "vrp_z_1y_har_implied_shrinkage",
    selected_current_formula,
)

panel["score_har_total_simple"] = apply_selected_formula(
    panel,
    "vrp_z_3m_har_total_simple",
    "vrp_z_1y_har_total_simple",
    selected_current_formula,
)


# -----------------------------
# 4. Daily top-tenor selection comparison
# -----------------------------

def daily_top_selection(df, score_col, model_label):
    work = df.dropna(subset=[score_col]).copy()

    work = work.sort_values(
        ["date", score_col, "tenor"],
        ascending=[True, False, True],
    )

    top = (
        work
        .groupby("date", as_index=False)
        .head(1)
        .copy()
    )

    keep_cols = [
        "date",
        "trade_date",
        "tenor",
        "tenor_group",
        score_col,
        "vix_style_vol",
        "forecast_vol",
        "forecast_vol_har_implied_shrinkage",
        "forecast_vol_har_total_simple",
        "vrp_log",
        "vrp_log_har_implied_shrinkage",
        "vrp_log_har_total_simple",
        "vrp_z_3m",
        "vrp_z_1y",
        "vrp_z_3m_har_implied_shrinkage",
        "vrp_z_1y_har_implied_shrinkage",
        "vrp_z_3m_har_total_simple",
        "vrp_z_1y_har_total_simple",
    ]

    keep_cols = [c for c in keep_cols if c in top.columns]

    top = top[keep_cols].copy()
    top = top.rename(columns={
        "tenor": f"tenor_{model_label}",
        "tenor_group": f"tenor_group_{model_label}",
        score_col: f"score_{model_label}",
    })

    return top


top_current = daily_top_selection(panel, "score_current", "current")
top_har_implied = daily_top_selection(panel, "score_har_implied_shrinkage", "har_implied")
top_har_total = daily_top_selection(panel, "score_har_total_simple", "har_total")

top_compare = top_current.merge(
    top_har_implied[
        [
            "date",
            "trade_date",
            "tenor_har_implied",
            "tenor_group_har_implied",
            "score_har_implied",
        ]
    ],
    on=["date", "trade_date"],
    how="inner",
    validate="one_to_one",
).merge(
    top_har_total[
        [
            "date",
            "trade_date",
            "tenor_har_total",
            "tenor_group_har_total",
            "score_har_total",
        ]
    ],
    on=["date", "trade_date"],
    how="inner",
    validate="one_to_one",
)

top_compare["same_tenor_current_vs_har_implied"] = (
    top_compare["tenor_current"] == top_compare["tenor_har_implied"]
)

top_compare["same_group_current_vs_har_implied"] = (
    top_compare["tenor_group_current"] == top_compare["tenor_group_har_implied"]
)

top_compare["same_tenor_har_implied_vs_har_total"] = (
    top_compare["tenor_har_implied"] == top_compare["tenor_har_total"]
)

top_compare["same_group_har_implied_vs_har_total"] = (
    top_compare["tenor_group_har_implied"] == top_compare["tenor_group_har_total"]
)

top_selection_summary = pd.DataFrame([{
    "rows": len(top_compare),
    "first_date": top_compare["date"].min(),
    "last_date": top_compare["date"].max(),
    "same_tenor_current_vs_har_implied_pct": top_compare["same_tenor_current_vs_har_implied"].mean(),
    "same_group_current_vs_har_implied_pct": top_compare["same_group_current_vs_har_implied"].mean(),
    "same_tenor_har_implied_vs_har_total_pct": top_compare["same_tenor_har_implied_vs_har_total"].mean(),
    "same_group_har_implied_vs_har_total_pct": top_compare["same_group_har_implied_vs_har_total"].mean(),
}])

print()
print("Daily top-tenor selection summary:")
display(top_selection_summary)

print()
print("Top-tenor distribution by method:")
dist_rows = []

for label, tenor_col, group_col in [
    ("current", "tenor_current", "tenor_group_current"),
    ("har_implied", "tenor_har_implied", "tenor_group_har_implied"),
    ("har_total", "tenor_har_total", "tenor_group_har_total"),
]:
    tenor_dist = (
        top_compare[tenor_col]
        .value_counts()
        .rename_axis("tenor")
        .reset_index(name="count")
    )
    tenor_dist["method"] = label
    tenor_dist["pct"] = tenor_dist["count"] / len(top_compare)

    group_dist = (
        top_compare[group_col]
        .value_counts()
        .rename_axis("tenor_group")
        .reset_index(name="count")
    )
    group_dist["method"] = label
    group_dist["pct"] = group_dist["count"] / len(top_compare)

    dist_rows.append(("tenor", tenor_dist))
    dist_rows.append(("group", group_dist))

top_tenor_distribution = pd.concat(
    [x[1].assign(distribution_type=x[0]) for x in dist_rows],
    ignore_index=True,
    sort=False,
)

display(top_tenor_distribution.sort_values(["distribution_type", "method", "tenor"]))


# -----------------------------
# 5. Daily cross-tenor rank correlation
# -----------------------------

rank_corr_rows = []

for d, g in panel.groupby("date"):
    g = g.dropna(
        subset=[
            "score_current",
            "score_har_implied_shrinkage",
            "score_har_total_simple",
        ]
    ).copy()

    if len(g) < 5:
        continue

    rank_corr_rows.append({
        "date": d,
        "trade_date": int(g["trade_date"].iloc[0]),
        "rank_corr_current_vs_har_implied": g["score_current"].corr(
            g["score_har_implied_shrinkage"],
            method="spearman",
        ),
        "rank_corr_current_vs_har_total": g["score_current"].corr(
            g["score_har_total_simple"],
            method="spearman",
        ),
        "rank_corr_har_implied_vs_har_total": g["score_har_implied_shrinkage"].corr(
            g["score_har_total_simple"],
            method="spearman",
        ),
    })

rank_corr_daily = pd.DataFrame(rank_corr_rows)

rank_corr_summary = rank_corr_daily.describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])

print()
print("Daily cross-tenor rank-correlation summary:")
display(rank_corr_summary)


# -----------------------------
# 6. Calibrated top-N candidate comparison
# -----------------------------

# Try to infer existing selected trade count from production flags.
selected_flag_candidates = [
    c for c in panel.columns
    if c.lower() in [
        "selected",
        "is_selected",
        "selected_trade",
        "trade_selected",
        "selected_trade_flag",
        "is_trade",
        "is_selected_trade",
    ]
]

inferred_trade_count = None
selected_flag_used = None

for c in selected_flag_candidates:
    s = panel[c]

    if s.dropna().empty:
        continue

    vals = set(pd.Series(s.dropna().unique()).astype(str).str.lower().tolist())

    if vals.issubset({"0", "1", "false", "true"}):
        numeric = (
            s.astype(str)
            .str.lower()
            .map({"true": 1, "false": 0, "1": 1, "0": 0})
        )

        count = int(numeric.fillna(0).sum())

        if 50 <= count <= panel["date"].nunique():
            inferred_trade_count = count
            selected_flag_used = c
            break

# Fallback to locked known selected-trade count from current project.
if inferred_trade_count is None:
    calibrated_trade_count = 577
    selected_count_source = "fallback_locked_backtest_count_577"
else:
    calibrated_trade_count = inferred_trade_count
    selected_count_source = f"production_flag_{selected_flag_used}"

print()
print("Calibrated trade count:")
display(pd.DataFrame([{
    "calibrated_trade_count": calibrated_trade_count,
    "selected_count_source": selected_count_source,
}]))


def top_n_daily_candidates(top_df, model_label, n):
    score_col = f"score_{model_label}"
    tenor_col = f"tenor_{model_label}"
    group_col = f"tenor_group_{model_label}"

    work = top_df.copy()

    keep_cols = [
        "date",
        "trade_date",
        tenor_col,
        group_col,
        score_col,
    ]

    work = work[keep_cols].copy()
    work = work.rename(columns={
        tenor_col: "selected_tenor",
        group_col: "selected_tenor_group",
        score_col: "selected_score",
    })

    work["method"] = model_label

    work = work.sort_values(
        ["selected_score", "date"],
        ascending=[False, True],
    ).head(n)

    return work


topn_current = top_n_daily_candidates(top_current, "current", calibrated_trade_count)
topn_har_implied = top_n_daily_candidates(top_har_implied, "har_implied", calibrated_trade_count)
topn_har_total = top_n_daily_candidates(top_har_total, "har_total", calibrated_trade_count)

topn_all = pd.concat(
    [topn_current, topn_har_implied, topn_har_total],
    ignore_index=True,
)

def topn_pair_overlap(a, b, label_a, label_b):
    a_keys_date = set(a["date"])
    b_keys_date = set(b["date"])

    a_keys_exact = set(zip(a["date"], a["selected_tenor"]))
    b_keys_exact = set(zip(b["date"], b["selected_tenor"]))

    return {
        "comparison": f"{label_a}_vs_{label_b}",
        "a_rows": len(a),
        "b_rows": len(b),
        "date_overlap_count": len(a_keys_date & b_keys_date),
        "date_overlap_pct_of_a": len(a_keys_date & b_keys_date) / len(a) if len(a) else np.nan,
        "exact_date_tenor_overlap_count": len(a_keys_exact & b_keys_exact),
        "exact_date_tenor_overlap_pct_of_a": len(a_keys_exact & b_keys_exact) / len(a) if len(a) else np.nan,
        "a_only_dates": len(a_keys_date - b_keys_date),
        "b_only_dates": len(b_keys_date - a_keys_date),
    }

topn_overlap_summary = pd.DataFrame([
    topn_pair_overlap(topn_current, topn_har_implied, "current", "har_implied"),
    topn_pair_overlap(topn_current, topn_har_total, "current", "har_total"),
    topn_pair_overlap(topn_har_implied, topn_har_total, "har_implied", "har_total"),
])

print()
print("Calibrated top-N overlap summary:")
display(topn_overlap_summary)

print()
print("Calibrated top-N tenor group distribution:")
topn_group_distribution = (
    topn_all
    .groupby(["method", "selected_tenor_group"])
    .agg(count=("date", "size"))
    .reset_index()
)

topn_group_distribution["pct"] = (
    topn_group_distribution["count"]
    / topn_group_distribution.groupby("method")["count"].transform("sum")
)

display(topn_group_distribution)

print()
print("Calibrated top-N tenor distribution:")
topn_tenor_distribution = (
    topn_all
    .groupby(["method", "selected_tenor"])
    .agg(count=("date", "size"))
    .reset_index()
)

topn_tenor_distribution["pct"] = (
    topn_tenor_distribution["count"]
    / topn_tenor_distribution.groupby("method")["count"].transform("sum")
)

display(topn_tenor_distribution)


# -----------------------------
# 7. Latest signal table
# -----------------------------

latest_date = panel["date"].max()

latest_signal_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_group",
    "vix_style_vol",
    "forecast_vol",
    "forecast_vol_har_implied_shrinkage",
    "forecast_vol_har_total_simple",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "score_current",
    "vrp_log_har_implied_shrinkage",
    "vrp_z_3m_har_implied_shrinkage",
    "vrp_z_1y_har_implied_shrinkage",
    "score_har_implied_shrinkage",
    "vrp_log_har_total_simple",
    "vrp_z_3m_har_total_simple",
    "vrp_z_1y_har_total_simple",
    "score_har_total_simple",
]

latest_signal_cols = [c for c in latest_signal_cols if c in panel.columns]

latest_signal_table = (
    panel
    .loc[panel["date"] == latest_date, latest_signal_cols]
    .sort_values("score_har_implied_shrinkage", ascending=False)
    .copy()
)

print()
print(f"Latest signal comparison table: {latest_date.date()}")
display(latest_signal_table)


# -----------------------------
# 8. Save outputs
# -----------------------------

safe_start = panel["date"].min().strftime("%Y%m%d")
safe_end = panel["date"].max().strftime("%Y%m%d")

cell10_panel_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_cell10_signal_comparison_panel_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell10_formula_fit_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10_formula_fit_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10_top_compare_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10_daily_top_selection_compare_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10_top_selection_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10_top_selection_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10_top_distribution_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10_top_tenor_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10_rank_corr_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10_daily_rank_correlations_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10_topn_all_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10_calibrated_topn_candidates_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10_topn_overlap_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10_calibrated_topn_overlap_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10_latest_signal_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10_latest_signal_comparison_{latest_date.strftime('%Y%m%d')}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

panel.to_parquet(cell10_panel_output, index=False)
formula_fit_summary.to_csv(cell10_formula_fit_output, index=False)
top_compare.to_csv(cell10_top_compare_output, index=False)
top_selection_summary.to_csv(cell10_top_selection_summary_output, index=False)
top_tenor_distribution.to_csv(cell10_top_distribution_output, index=False)
rank_corr_daily.to_csv(cell10_rank_corr_output, index=False)
topn_all.to_csv(cell10_topn_all_output, index=False)
topn_overlap_summary.to_csv(cell10_topn_overlap_output, index=False)
latest_signal_table.to_csv(cell10_latest_signal_output, index=False)

print()
print("Cell 10 outputs saved:")
print(f"Signal comparison panel:      {cell10_panel_output}")
print(f"Formula fit summary:          {cell10_formula_fit_output}")
print(f"Daily top selection compare:  {cell10_top_compare_output}")
print(f"Top selection summary:        {cell10_top_selection_summary_output}")
print(f"Top tenor distribution:       {cell10_top_distribution_output}")
print(f"Daily rank correlations:      {cell10_rank_corr_output}")
print(f"Calibrated top-N candidates:  {cell10_topn_all_output}")
print(f"Calibrated top-N overlap:     {cell10_topn_overlap_output}")
print(f"Latest signal comparison:     {cell10_latest_signal_output}")

print()
print("Cell 10 complete.")
print("Next step: backtest selected trades under har_implied_shrinkage vs current production selection.")

## 10B. Corrected apples-to-apples signal comparison

Rebuilds current, `har_implied`, and `har_total` scores using the same score definitions. Confirms that the production `primary_vrp_signal` is raw `vrp_log`, not a z-score blend.

In [ ]:
# Cell 10B: Corrected apples-to-apples signal comparison
#
# Purpose:
# Cell 10 mixed score definitions:
#   - current score used primary_vrp_signal, which appears equal to raw vrp_log
#   - new model scores used an inferred z-score formula
#
# This cell compares current vs HAR-implied vs HAR-total under the SAME score definitions:
#   1. raw_vrp_log
#   2. z_3m
#   3. z_1y
#   4. max_z_3m_1y
#   5. avg_z_3m_1y
#   6. min_z_3m_1y
#
# It does not backtest yet.

from itertools import combinations


# -----------------------------
# 1. Load panel if needed
# -----------------------------

try:
    panel
except NameError:
    latest_cell10_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_cell10_signal_comparison_panel_*.parquet")
    )
    latest_cell9_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_final_forecast_vrp_panel_v1_*.parquet")
    )

    if latest_cell10_files:
        print(f"Loading Cell 10 signal comparison panel: {latest_cell10_files[-1]}")
        panel = pd.read_parquet(latest_cell10_files[-1])
    elif latest_cell9_files:
        print(f"Loading Cell 9 final forecast VRP panel: {latest_cell9_files[-1]}")
        panel = pd.read_parquet(latest_cell9_files[-1])
    else:
        raise FileNotFoundError("No Cell 10 or Cell 9 panel found.")

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"

panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)

panel = panel.replace([np.inf, -np.inf], np.nan)


# -----------------------------
# 2. Check what primary_vrp_signal actually is
# -----------------------------

identity_summary = pd.DataFrame()

if "primary_vrp_signal" in panel.columns:
    identity_candidates = {}

    candidate_source_cols = [
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
    ]

    for c in candidate_source_cols:
        if c in panel.columns:
            identity_candidates[c] = panel[c]

    if all(c in panel.columns for c in ["vrp_z_3m", "vrp_z_1y"]):
        identity_candidates["max_z_3m_1y"] = np.maximum(panel["vrp_z_3m"], panel["vrp_z_1y"])
        identity_candidates["avg_z_3m_1y"] = 0.5 * (panel["vrp_z_3m"] + panel["vrp_z_1y"])
        identity_candidates["min_z_3m_1y"] = np.minimum(panel["vrp_z_3m"], panel["vrp_z_1y"])

    rows = []

    for name, series in identity_candidates.items():
        work = pd.DataFrame({
            "primary_vrp_signal": panel["primary_vrp_signal"],
            "candidate": series,
        }).replace([np.inf, -np.inf], np.nan).dropna()

        if work.empty:
            continue

        diff = work["candidate"] - work["primary_vrp_signal"]

        rows.append({
            "candidate": name,
            "rows": len(work),
            "mae_vs_primary_vrp_signal": diff.abs().mean(),
            "rmse_vs_primary_vrp_signal": np.sqrt((diff ** 2).mean()),
            "max_abs_diff": diff.abs().max(),
            "corr_vs_primary_vrp_signal": work["candidate"].corr(work["primary_vrp_signal"]),
        })

    identity_summary = pd.DataFrame(rows).sort_values("rmse_vs_primary_vrp_signal")

print()
print("Identity check: what is primary_vrp_signal closest to?")
if identity_summary.empty:
    print("primary_vrp_signal not available.")
else:
    display(identity_summary)


# -----------------------------
# 3. Build apples-to-apples score columns
# -----------------------------

required_cols = [
    "vrp_log",
    "vrp_log_har_implied_shrinkage",
    "vrp_log_har_total_simple",
    "vrp_z_3m",
    "vrp_z_3m_har_implied_shrinkage",
    "vrp_z_3m_har_total_simple",
    "vrp_z_1y",
    "vrp_z_1y_har_implied_shrinkage",
    "vrp_z_1y_har_total_simple",
]

missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise ValueError(f"Missing required score columns: {missing_required}")

model_labels = {
    "current": {
        "raw": "vrp_log",
        "z3": "vrp_z_3m",
        "z1": "vrp_z_1y",
    },
    "har_implied": {
        "raw": "vrp_log_har_implied_shrinkage",
        "z3": "vrp_z_3m_har_implied_shrinkage",
        "z1": "vrp_z_1y_har_implied_shrinkage",
    },
    "har_total": {
        "raw": "vrp_log_har_total_simple",
        "z3": "vrp_z_3m_har_total_simple",
        "z1": "vrp_z_1y_har_total_simple",
    },
}

score_methods = [
    "raw_vrp_log",
    "z_3m",
    "z_1y",
    "max_z_3m_1y",
    "avg_z_3m_1y",
    "min_z_3m_1y",
]

for model_label, cols in model_labels.items():
    panel[f"score_{model_label}_raw_vrp_log"] = panel[cols["raw"]]
    panel[f"score_{model_label}_z_3m"] = panel[cols["z3"]]
    panel[f"score_{model_label}_z_1y"] = panel[cols["z1"]]

    panel[f"score_{model_label}_max_z_3m_1y"] = np.maximum(
        panel[cols["z3"]],
        panel[cols["z1"]],
    )

    panel[f"score_{model_label}_avg_z_3m_1y"] = 0.5 * (
        panel[cols["z3"]]
        + panel[cols["z1"]]
    )

    panel[f"score_{model_label}_min_z_3m_1y"] = np.minimum(
        panel[cols["z3"]],
        panel[cols["z1"]],
    )


# -----------------------------
# 4. Daily top-tenor selections
# -----------------------------

top_rows = []

for score_method in score_methods:
    for model_label in model_labels.keys():
        score_col = f"score_{model_label}_{score_method}"

        work = panel.dropna(subset=[score_col]).copy()

        work = work.sort_values(
            ["date", score_col, "tenor"],
            ascending=[True, False, True],
        )

        top = work.groupby("date", as_index=False).head(1).copy()

        top_rows.append(
            top[["date", "trade_date", "tenor", "tenor_group", score_col]]
            .rename(columns={
                "tenor": "selected_tenor",
                "tenor_group": "selected_tenor_group",
                score_col: "selected_score",
            })
            .assign(
                score_method=score_method,
                model_label=model_label,
            )
        )

top_daily_long = pd.concat(top_rows, ignore_index=True)


# -----------------------------
# 5. Top-selection overlap by score method
# -----------------------------

top_compare_rows = []

for score_method in score_methods:
    wide = (
        top_daily_long
        .loc[top_daily_long["score_method"] == score_method]
        .pivot_table(
            index=["date", "trade_date"],
            columns="model_label",
            values=["selected_tenor", "selected_tenor_group", "selected_score"],
            aggfunc="first",
        )
    )

    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    for a, b in [
        ("current", "har_implied"),
        ("current", "har_total"),
        ("har_implied", "har_total"),
    ]:
        top_compare_rows.append({
            "score_method": score_method,
            "comparison": f"{a}_vs_{b}",
            "rows": len(wide),
            "same_tenor_pct": (
                wide[f"selected_tenor_{a}"] == wide[f"selected_tenor_{b}"]
            ).mean(),
            "same_group_pct": (
                wide[f"selected_tenor_group_{a}"] == wide[f"selected_tenor_group_{b}"]
            ).mean(),
            "mean_score_a": wide[f"selected_score_{a}"].mean(),
            "mean_score_b": wide[f"selected_score_{b}"].mean(),
        })

top_selection_overlap_summary = pd.DataFrame(top_compare_rows)

print()
print("Corrected daily top-selection overlap summary:")
display(top_selection_overlap_summary)


# -----------------------------
# 6. Top-tenor and top-group distributions
# -----------------------------

top_tenor_distribution = (
    top_daily_long
    .groupby(["score_method", "model_label", "selected_tenor"])
    .agg(count=("date", "size"))
    .reset_index()
)

top_tenor_distribution["pct"] = (
    top_tenor_distribution["count"]
    / top_tenor_distribution.groupby(["score_method", "model_label"])["count"].transform("sum")
)

top_group_distribution = (
    top_daily_long
    .groupby(["score_method", "model_label", "selected_tenor_group"])
    .agg(count=("date", "size"))
    .reset_index()
)

top_group_distribution["pct"] = (
    top_group_distribution["count"]
    / top_group_distribution.groupby(["score_method", "model_label"])["count"].transform("sum")
)

print()
print("Corrected top-group distribution:")
display(top_group_distribution.sort_values(["score_method", "model_label", "selected_tenor_group"]))

print()
print("Corrected top-tenor distribution:")
display(top_tenor_distribution.sort_values(["score_method", "model_label", "selected_tenor"]))


# -----------------------------
# 7. Daily cross-tenor rank correlations by score method
# -----------------------------

rank_corr_rows = []

for score_method in score_methods:
    cols = {
        model_label: f"score_{model_label}_{score_method}"
        for model_label in model_labels.keys()
    }

    for d, g in panel.groupby("date"):
        needed = list(cols.values())

        gg = g.dropna(subset=needed).copy()

        if len(gg) < 5:
            continue

        rank_corr_rows.append({
            "score_method": score_method,
            "date": d,
            "trade_date": int(gg["trade_date"].iloc[0]),
            "rank_corr_current_vs_har_implied": gg[cols["current"]].corr(
                gg[cols["har_implied"]],
                method="spearman",
            ),
            "rank_corr_current_vs_har_total": gg[cols["current"]].corr(
                gg[cols["har_total"]],
                method="spearman",
            ),
            "rank_corr_har_implied_vs_har_total": gg[cols["har_implied"]].corr(
                gg[cols["har_total"]],
                method="spearman",
            ),
        })

rank_corr_daily_corrected = pd.DataFrame(rank_corr_rows)

rank_corr_summary_corrected = (
    rank_corr_daily_corrected
    .groupby("score_method")
    .agg(
        rows=("date", "size"),
        mean_current_vs_har_implied=("rank_corr_current_vs_har_implied", "mean"),
        median_current_vs_har_implied=("rank_corr_current_vs_har_implied", "median"),
        p05_current_vs_har_implied=("rank_corr_current_vs_har_implied", lambda x: x.quantile(0.05)),
        p95_current_vs_har_implied=("rank_corr_current_vs_har_implied", lambda x: x.quantile(0.95)),
        mean_har_implied_vs_har_total=("rank_corr_har_implied_vs_har_total", "mean"),
        median_har_implied_vs_har_total=("rank_corr_har_implied_vs_har_total", "median"),
    )
    .reset_index()
)

print()
print("Corrected daily rank-correlation summary:")
display(rank_corr_summary_corrected)


# -----------------------------
# 8. Calibrated top-N comparison under each score method
# -----------------------------

CALIBRATED_TRADE_COUNT = 577

topn_rows = []

for score_method in score_methods:
    for model_label in model_labels.keys():
        work = top_daily_long.loc[
            (top_daily_long["score_method"] == score_method)
            & (top_daily_long["model_label"] == model_label)
        ].copy()

        work = work.sort_values(
            ["selected_score", "date"],
            ascending=[False, True],
        ).head(CALIBRATED_TRADE_COUNT)

        topn_rows.append(work)

topn_corrected = pd.concat(topn_rows, ignore_index=True)

topn_overlap_rows = []

for score_method in score_methods:
    method_rows = topn_corrected.loc[topn_corrected["score_method"] == score_method].copy()

    for a, b in [
        ("current", "har_implied"),
        ("current", "har_total"),
        ("har_implied", "har_total"),
    ]:
        aa = method_rows.loc[method_rows["model_label"] == a]
        bb = method_rows.loc[method_rows["model_label"] == b]

        a_dates = set(aa["date"])
        b_dates = set(bb["date"])

        a_exact = set(zip(aa["date"], aa["selected_tenor"]))
        b_exact = set(zip(bb["date"], bb["selected_tenor"]))

        topn_overlap_rows.append({
            "score_method": score_method,
            "comparison": f"{a}_vs_{b}",
            "a_rows": len(aa),
            "b_rows": len(bb),
            "date_overlap_count": len(a_dates & b_dates),
            "date_overlap_pct": len(a_dates & b_dates) / len(aa) if len(aa) else np.nan,
            "exact_date_tenor_overlap_count": len(a_exact & b_exact),
            "exact_date_tenor_overlap_pct": len(a_exact & b_exact) / len(aa) if len(aa) else np.nan,
        })

topn_overlap_corrected = pd.DataFrame(topn_overlap_rows)

topn_group_distribution_corrected = (
    topn_corrected
    .groupby(["score_method", "model_label", "selected_tenor_group"])
    .agg(count=("date", "size"))
    .reset_index()
)

topn_group_distribution_corrected["pct"] = (
    topn_group_distribution_corrected["count"]
    / topn_group_distribution_corrected.groupby(["score_method", "model_label"])["count"].transform("sum")
)

topn_tenor_distribution_corrected = (
    topn_corrected
    .groupby(["score_method", "model_label", "selected_tenor"])
    .agg(count=("date", "size"))
    .reset_index()
)

topn_tenor_distribution_corrected["pct"] = (
    topn_tenor_distribution_corrected["count"]
    / topn_tenor_distribution_corrected.groupby(["score_method", "model_label"])["count"].transform("sum")
)

print()
print("Corrected calibrated top-N overlap:")
display(topn_overlap_corrected)

print()
print("Corrected calibrated top-N group distribution:")
display(topn_group_distribution_corrected)

print()
print("Corrected calibrated top-N tenor distribution:")
display(topn_tenor_distribution_corrected)


# -----------------------------
# 9. Latest signal table
# -----------------------------

latest_date = panel["date"].max()

latest_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_group",
    "vix_style_vol",
    "forecast_vol",
    "forecast_vol_har_implied_shrinkage",
    "forecast_vol_har_total_simple",

    "vrp_log",
    "vrp_log_har_implied_shrinkage",
    "vrp_log_har_total_simple",

    "vrp_z_3m",
    "vrp_z_3m_har_implied_shrinkage",
    "vrp_z_3m_har_total_simple",

    "vrp_z_1y",
    "vrp_z_1y_har_implied_shrinkage",
    "vrp_z_1y_har_total_simple",

    "score_current_raw_vrp_log",
    "score_har_implied_raw_vrp_log",
    "score_har_total_raw_vrp_log",

    "score_current_max_z_3m_1y",
    "score_har_implied_max_z_3m_1y",
    "score_har_total_max_z_3m_1y",
]

latest_cols = [c for c in latest_cols if c in panel.columns]

latest_corrected_signal_table = (
    panel
    .loc[panel["date"] == latest_date, latest_cols]
    .sort_values("score_har_implied_max_z_3m_1y", ascending=False)
    .copy()
)

print()
print(f"Corrected latest signal table: {latest_date.date()}")
display(latest_corrected_signal_table)


# -----------------------------
# 10. Save outputs
# -----------------------------

safe_start = panel["date"].min().strftime("%Y%m%d")
safe_end = panel["date"].max().strftime("%Y%m%d")

cell10b_panel_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_cell10b_corrected_signal_panel_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell10b_identity_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_primary_signal_identity_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_top_daily_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_top_daily_long_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_overlap_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_top_selection_overlap_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_top_group_dist_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_top_group_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_top_tenor_dist_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_top_tenor_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_rank_corr_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_rank_correlation_daily_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_rank_corr_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_rank_correlation_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_topn_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_calibrated_topn_candidates_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_topn_overlap_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_calibrated_topn_overlap_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_topn_group_dist_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_calibrated_topn_group_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_topn_tenor_dist_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_calibrated_topn_tenor_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell10b_latest_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell10b_latest_corrected_signal_table_{latest_date.strftime('%Y%m%d')}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

panel.to_parquet(cell10b_panel_output, index=False)
identity_summary.to_csv(cell10b_identity_output, index=False)
top_daily_long.to_csv(cell10b_top_daily_output, index=False)
top_selection_overlap_summary.to_csv(cell10b_overlap_output, index=False)
top_group_distribution.to_csv(cell10b_top_group_dist_output, index=False)
top_tenor_distribution.to_csv(cell10b_top_tenor_dist_output, index=False)
rank_corr_daily_corrected.to_csv(cell10b_rank_corr_output, index=False)
rank_corr_summary_corrected.to_csv(cell10b_rank_corr_summary_output, index=False)
topn_corrected.to_csv(cell10b_topn_output, index=False)
topn_overlap_corrected.to_csv(cell10b_topn_overlap_output, index=False)
topn_group_distribution_corrected.to_csv(cell10b_topn_group_dist_output, index=False)
topn_tenor_distribution_corrected.to_csv(cell10b_topn_tenor_dist_output, index=False)
latest_corrected_signal_table.to_csv(cell10b_latest_output, index=False)

print()
print("Cell 10B outputs saved:")
print(f"Corrected signal panel:          {cell10b_panel_output}")
print(f"Primary signal identity:         {cell10b_identity_output}")
print(f"Top daily long:                  {cell10b_top_daily_output}")
print(f"Top selection overlap summary:   {cell10b_overlap_output}")
print(f"Top group distribution:          {cell10b_top_group_dist_output}")
print(f"Top tenor distribution:          {cell10b_top_tenor_dist_output}")
print(f"Rank corr daily:                 {cell10b_rank_corr_output}")
print(f"Rank corr summary:               {cell10b_rank_corr_summary_output}")
print(f"Calibrated top-N candidates:     {cell10b_topn_output}")
print(f"Calibrated top-N overlap:        {cell10b_topn_overlap_output}")
print(f"Calibrated top-N group dist:     {cell10b_topn_group_dist_output}")
print(f"Calibrated top-N tenor dist:     {cell10b_topn_tenor_dist_output}")
print(f"Latest corrected signal table:   {cell10b_latest_output}")

print()
print("Cell 10B complete.")
print("Next step: choose which score definition to backtest first.")

## 11. In-sample final-panel trade-selection diagnostic

Joins final scored signals to normalized P&L outcomes. This is useful as a diagnostic, but the model-selection decision should be based on the out-of-sample version in Cell 13.

In [ ]:
# Cell 11: Backtest current vs HAR-implied-shrinkage signal variants
#
# Corrected full replacement:
#   - safer trade-outcome file discovery
#   - requires an actual P&L / return outcome column
#   - avoids selecting high-coverage signal-selection files with no outcome column
#   - robust date parsing
#   - robust tenor parsing
#   - robust large-file handling
#
# Purpose:
#   1. Load corrected Cell 10B signal panel.
#   2. Discover available trade-outcome / P&L files.
#   3. Join selected date x tenor candidates to trade outcomes.
#   4. Compare:
#        - current
#        - har_implied
#        - har_total
#      under:
#        - raw_vrp_log
#        - max_z_3m_1y
#        - avg_z_3m_1y
#
# Important:
#   This requires an all-candidate trade outcome file with at least:
#        date / trade_date, tenor, and a P&L or return column.
#
# If auto-discovery still picks the wrong file, set:
#   TRADE_OUTCOME_FILE_OVERRIDE
#   OUTCOME_VALUE_COL_OVERRIDE
#   OUTCOME_VALUE_KIND_OVERRIDE

from pathlib import Path
import numpy as np
import pandas as pd


# -----------------------------
# 1. Configuration / overrides
# -----------------------------

CALIBRATED_TRADE_COUNT = 577

SCORE_METHODS_TO_BACKTEST = [
    "raw_vrp_log",
    "max_z_3m_1y",
    "avg_z_3m_1y",
]

MODEL_LABELS_TO_BACKTEST = [
    "current",
    "har_implied",
    "har_total",
]

# Optional manual overrides.
# Leave as None unless auto-discovery picks the wrong file/column.
TRADE_OUTCOME_FILE_OVERRIDE = None
OUTCOME_VALUE_COL_OVERRIDE = None
OUTCOME_VALUE_KIND_OVERRIDE = None   # None, "pnl", or "return"

# Discovery controls.
DISCOVERY_SAMPLE_ROWS = 25000
MIN_DISCOVERY_SCORE = 45


# -----------------------------
# 2. Resolve project paths
# -----------------------------

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "CORSI_PROCESSED_DIR" not in globals():
    CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "forecast_model_corsi_v1"

if "CORSI_AUDIT_DIR" not in globals():
    CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"


# -----------------------------
# 3. Helper functions
# -----------------------------

def first_series(obj):
    """
    Return a Series from either a Series or a duplicate-column DataFrame.
    """
    if isinstance(obj, pd.DataFrame):
        return obj.iloc[:, 0]
    return obj


def robust_parse_date_series(s):
    """
    Parse dates robustly.

    Handles:
      - datetime64
      - YYYY-MM-DD strings
      - YYYYMMDD integers
      - YYYYMMDD strings
    """
    s = first_series(s)

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s, errors="coerce")

    s_str = s.astype(str).str.strip()

    # Prefer YYYYMMDD when present.
    ymd = s_str.str.extract(r"(\d{8})", expand=False)
    ymd_dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")

    # Use direct parsing only for non-YYYYMMDD values.
    direct_dt = pd.to_datetime(s_str.where(ymd_dt.isna()), errors="coerce")

    out = ymd_dt.copy()
    out.loc[out.isna()] = direct_dt.loc[out.isna()]
    return out


def robust_parse_trade_date_series(s):
    """
    Return integer YYYYMMDD trade_date where possible.
    """
    dt = robust_parse_date_series(s)
    out = pd.to_numeric(dt.dt.strftime("%Y%m%d"), errors="coerce")
    return out.astype("Int64")


def robust_parse_tenor_series(s, valid_target_tenors=None):
    """
    Parse tenor robustly and only keep integer-like target tenors.

    Handles:
      - int / float tenors
      - numeric strings
      - duplicate-column DataFrame inputs
      - rejects nonnumeric buckets like 'front', 'middle', 'back'
    """
    s = first_series(s)
    tenor_num = pd.to_numeric(s, errors="coerce")

    out = pd.Series(pd.NA, index=s.index, dtype="Int64")

    valid = tenor_num.notna() & np.isfinite(tenor_num)
    integer_like = (tenor_num - tenor_num.round()).abs() < 1e-8
    valid = valid & integer_like.fillna(False)

    out.loc[valid] = tenor_num.loc[valid].round().astype("Int64")

    if valid_target_tenors is not None:
        valid_target_tenors = set(int(x) for x in valid_target_tenors)
        out.loc[~out.isin(valid_target_tenors)] = pd.NA

    return out


def max_drawdown(cum_series):
    running_max = cum_series.cummax()
    dd = cum_series - running_max
    return dd.min()


def infer_col(cols, exact_candidates, contains_candidates=None):
    lower_map = {str(c).lower(): c for c in cols}

    for c in exact_candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    if contains_candidates:
        for token in contains_candidates:
            matches = [c for c in cols if token.lower() in str(c).lower()]
            if matches:
                return matches[0]

    return None


def safe_read_sample(path, nrows=25000):
    try:
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
            full_rows = len(df)
            return df.head(nrows).copy(), full_rows, None

        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path, nrows=nrows)
            try:
                full_rows = sum(
                    1 for _ in open(path, "r", encoding="utf-8", errors="ignore")
                ) - 1
            except Exception:
                full_rows = np.nan
            return df, full_rows, None

        return None, 0, "unsupported extension"

    except Exception as e:
        return None, 0, str(e)


def choose_outcome_column(cols):
    """
    Select a true outcome column.

    Priority:
      1. user override
      2. sized / normalized / dollar P&L
      3. generic P&L
      4. points P&L
      5. return-on-risk style fields

    Avoids selecting diagnostic count columns like num_returns.
    """
    cols = list(cols)
    lower_map = {str(c).lower(): c for c in cols}

    if OUTCOME_VALUE_COL_OVERRIDE is not None:
        if OUTCOME_VALUE_COL_OVERRIDE in cols:
            return OUTCOME_VALUE_COL_OVERRIDE, OUTCOME_VALUE_KIND_OVERRIDE or "pnl"
        raise ValueError(f"OUTCOME_VALUE_COL_OVERRIDE not found: {OUTCOME_VALUE_COL_OVERRIDE}")

    pnl_priority = [
        "sized_pnl",
        "weighted_pnl",
        "normalized_pnl_dollars",
        "normalized_pnl_dollar",
        "normalized_pnl_usd",
        "normalized_pnl",
        "trade_pnl",
        "realized_pnl",
        "pnl_dollars",
        "pnl_dollar",
        "pnl_usd",
        "short_option_pnl_dollars",
        "short_option_pnl_dollar",
        "short_option_pnl_points",
        "net_pnl",
        "total_pnl",
        "pnl",
    ]

    return_priority = [
        "return_on_risk",
        "return_vs_max_loss",
        "return_on_max_loss",
        "trade_return",
        "realized_return",
        "pnl_pct",
        "return_pct",
    ]

    # Exact P&L matches.
    for c in pnl_priority:
        if c in lower_map:
            return lower_map[c], "pnl"

    # Exact return matches.
    for c in return_priority:
        if c in lower_map:
            return lower_map[c], "return"

    # Contains-based P&L matches.
    bad_tokens = [
        "num_returns",
        "number_returns",
        "forward_num_returns",
        "trailing_num_returns",
        "count",
        "rank",
        "score",
        "signal",
        "selected",
        "tenor",
        "date",
        "dte",
    ]

    pnl_contains = []
    return_contains = []

    for c in cols:
        cl = str(c).lower()

        if any(bad in cl for bad in bad_tokens):
            continue

        if "pnl" in cl or "p&l" in cl:
            pnl_contains.append(c)

        elif "return" in cl or cl in ["ret"]:
            return_contains.append(c)

    if pnl_contains:
        # Prefer dollar / normalized / sized fields over points.
        def pnl_sort_key(c):
            cl = str(c).lower()
            score = 0
            if "sized" in cl:
                score -= 100
            if "normalized" in cl:
                score -= 80
            if "dollar" in cl or "usd" in cl:
                score -= 60
            if "point" in cl:
                score += 20
            return score

        pnl_contains = sorted(pnl_contains, key=pnl_sort_key)
        return pnl_contains[0], "pnl"

    if return_contains:
        return return_contains[0], "return"

    return None, None


# -----------------------------
# 4. Load corrected signal panel
# -----------------------------

try:
    panel
except NameError:
    latest_cell10b_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_cell10b_corrected_signal_panel_*.parquet")
    )

    if not latest_cell10b_files:
        raise FileNotFoundError("No Cell 10B corrected signal panel found.")

    print(f"Loading corrected signal panel: {latest_cell10b_files[-1]}")
    panel = pd.read_parquet(latest_cell10b_files[-1])

panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
panel["trade_date"] = pd.to_numeric(panel["trade_date"], errors="coerce").astype(int)
panel["tenor"] = pd.to_numeric(panel["tenor"], errors="coerce").astype(int)
panel = panel.replace([np.inf, -np.inf], np.nan)

VALID_TARGET_TENORS = sorted(panel["tenor"].dropna().astype(int).unique())

print("Signal panel summary:")
display(pd.DataFrame([{
    "rows": len(panel),
    "first_date": panel["date"].min(),
    "last_date": panel["date"].max(),
    "unique_dates": panel["date"].nunique(),
    "unique_tenors": panel["tenor"].nunique(),
    "valid_target_tenors": VALID_TARGET_TENORS,
}]))


# -----------------------------
# 5. Build selected trade candidates
# -----------------------------

missing_score_cols = []

for score_method in SCORE_METHODS_TO_BACKTEST:
    for model_label in MODEL_LABELS_TO_BACKTEST:
        score_col = f"score_{model_label}_{score_method}"
        if score_col not in panel.columns:
            missing_score_cols.append(score_col)

if missing_score_cols:
    raise ValueError(f"Missing score columns from Cell 10B panel: {missing_score_cols}")

selected_rows = []

for score_method in SCORE_METHODS_TO_BACKTEST:
    for model_label in MODEL_LABELS_TO_BACKTEST:
        score_col = f"score_{model_label}_{score_method}"

        work = panel.dropna(subset=[score_col]).copy()

        # One best tenor per date.
        daily_top = (
            work
            .sort_values(["date", score_col, "tenor"], ascending=[True, False, True])
            .groupby("date", as_index=False)
            .head(1)
            .copy()
        )

        # Calibrate to same number of trades as locked production backtest.
        topn = (
            daily_top
            .sort_values([score_col, "date"], ascending=[False, True])
            .head(CALIBRATED_TRADE_COUNT)
            .copy()
        )

        keep_cols = [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            score_col,
            "vix_style_vol",
            "forecast_vol",
            "forecast_vol_har_implied_shrinkage",
            "forecast_vol_har_total_simple",
            "vrp_log",
            "vrp_log_har_implied_shrinkage",
            "vrp_log_har_total_simple",
            "vrp_z_3m",
            "vrp_z_1y",
            "vrp_z_3m_har_implied_shrinkage",
            "vrp_z_1y_har_implied_shrinkage",
            "vrp_z_3m_har_total_simple",
            "vrp_z_1y_har_total_simple",
        ]

        keep_cols = [c for c in keep_cols if c in topn.columns]

        topn = topn[keep_cols].copy()
        topn = topn.rename(columns={score_col: "selected_score"})
        topn["score_method"] = score_method
        topn["model_label"] = model_label
        topn["selection_rank"] = np.arange(1, len(topn) + 1)

        selected_rows.append(topn)

selected_candidates = pd.concat(selected_rows, ignore_index=True)

print()
print("Selected candidate summary before outcome join:")
display(
    selected_candidates
    .groupby(["score_method", "model_label"])
    .agg(
        trades=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        mean_score=("selected_score", "mean"),
        min_score=("selected_score", "min"),
        max_score=("selected_score", "max"),
    )
    .reset_index()
)


# -----------------------------
# 6. Discover trade-outcome files
# -----------------------------

file_keywords = [
    "trade",
    "backtest",
    "pnl",
    "selected",
    "outcome",
    "naked",
    "put",
    "candidate",
    "result",
]

search_roots = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "audit",
]

candidate_files = []

if TRADE_OUTCOME_FILE_OVERRIDE is not None:
    candidate_files = [Path(TRADE_OUTCOME_FILE_OVERRIDE)]
else:
    for root in search_roots:
        if not root.exists():
            continue

        for ext in ["*.parquet", "*.csv"]:
            for p in root.rglob(ext):
                name = p.name.lower()
                if any(k in name for k in file_keywords):
                    candidate_files.append(p)

candidate_files = sorted(set(candidate_files))

selected_keys = set(zip(selected_candidates["date"], selected_candidates["tenor"]))

discovery_rows = []

for p in candidate_files:
    sample, full_rows, err = safe_read_sample(p, nrows=DISCOVERY_SAMPLE_ROWS)

    if sample is None:
        discovery_rows.append({
            "path": str(p),
            "rows_est": full_rows,
            "columns": np.nan,
            "date_col": None,
            "trade_date_col": None,
            "tenor_col": None,
            "pnl_cols": "",
            "return_cols": "",
            "chosen_outcome_col": None,
            "chosen_outcome_kind": None,
            "has_outcome_col": False,
            "coverage_pct_sample": np.nan,
            "score": -999,
            "eligible_for_backtest": False,
            "read_error": err,
            "parse_error": None,
        })
        continue

    cols = list(sample.columns)

    date_col = infer_col(
        cols,
        ["date", "signal_date", "entry_date", "trade_date_dt"],
        ["date"],
    )

    trade_date_col = infer_col(
        cols,
        ["trade_date", "yyyymmdd"],
    )

    tenor_col = infer_col(
        cols,
        ["tenor", "target_days", "target_dte", "dte", "days_to_expiry", "selected_tenor"],
        ["tenor", "dte"],
    )

    chosen_outcome_col, chosen_outcome_kind = choose_outcome_column(cols)

    has_outcome_col = chosen_outcome_col is not None

    pnl_cols = [
        c for c in cols
        if "pnl" in str(c).lower()
        or "p&l" in str(c).lower()
    ]

    return_cols = [
        c for c in cols
        if "return" in str(c).lower()
        or str(c).lower() == "ret"
    ]

    coverage_pct_sample = np.nan
    parse_error = None

    if date_col is not None and tenor_col is not None:
        try:
            tmp = pd.DataFrame(index=sample.index)
            tmp["_date"] = robust_parse_date_series(sample[date_col])
            tmp["_tenor"] = robust_parse_tenor_series(
                sample[tenor_col],
                valid_target_tenors=VALID_TARGET_TENORS,
            )

            tmp = tmp.loc[
                tmp["_date"].notna()
                & tmp["_tenor"].notna()
            ].copy()

            tmp["_tenor"] = tmp["_tenor"].astype(int)

            sample_keys = set(zip(tmp["_date"], tmp["_tenor"]))

            if selected_keys:
                coverage_pct_sample = len(sample_keys & selected_keys) / len(selected_keys)

        except Exception as e:
            parse_error = str(e)
            coverage_pct_sample = np.nan

    eligible_for_backtest = (
        date_col is not None
        and tenor_col is not None
        and has_outcome_col
    )

    score = 0

    if eligible_for_backtest:
        score += 40
        score += 10 if trade_date_col is not None else 0
        score += min(float(coverage_pct_sample) if pd.notna(coverage_pct_sample) else 0.0, 1.0) * 35
        score += min(float(full_rows) / 10000, 1.0) * 5 if pd.notna(full_rows) else 0

        # Prefer processed files over audit files.
        p_str = str(p).lower()
        if "\\data\\processed\\" in p_str or "/data/processed/" in p_str:
            score += 8

        # Prefer all-candidate style files over selected-only files.
        name = p.name.lower()
        if "candidate" in name or "all" in name or "grid" in name:
            score += 6
        if "selected" in name and "candidate" not in name:
            score -= 8

        # Prefer dollar / normalized / sized outcomes.
        c_low = str(chosen_outcome_col).lower()
        if "sized" in c_low:
            score += 12
        if "normalized" in c_low:
            score += 10
        if "dollar" in c_low or "usd" in c_low:
            score += 8
        if "points" in c_low:
            score += 2

    else:
        # Keep in discovery report, but make impossible to select.
        score = -100

    discovery_rows.append({
        "path": str(p),
        "rows_est": full_rows,
        "columns": len(cols),
        "date_col": date_col,
        "trade_date_col": trade_date_col,
        "tenor_col": tenor_col,
        "pnl_cols": ", ".join([str(x) for x in pnl_cols[:12]]),
        "return_cols": ", ".join([str(x) for x in return_cols[:12]]),
        "chosen_outcome_col": chosen_outcome_col,
        "chosen_outcome_kind": chosen_outcome_kind,
        "has_outcome_col": has_outcome_col,
        "coverage_pct_sample": coverage_pct_sample,
        "score": score,
        "eligible_for_backtest": eligible_for_backtest,
        "read_error": err,
        "parse_error": parse_error,
    })

discovery = pd.DataFrame(discovery_rows).sort_values("score", ascending=False)

eligible_discovery = discovery.loc[discovery["eligible_for_backtest"] == True].copy()

print()
print("Trade outcome file discovery:")
display(discovery.head(30))

print()
print("Eligible trade outcome files:")
display(eligible_discovery.head(20))

if eligible_discovery.empty or eligible_discovery.iloc[0]["score"] < MIN_DISCOVERY_SCORE:
    print()
    print("No suitable all-candidate trade outcome file was found.")
    print("Review the discovery tables above.")
    print("Set TRADE_OUTCOME_FILE_OVERRIDE and OUTCOME_VALUE_COL_OVERRIDE, then rerun Cell 11.")
    raise RuntimeError("Stopping before backtest because no suitable trade outcome file was found.")

chosen_file = Path(eligible_discovery.iloc[0]["path"])
chosen_date_col = eligible_discovery.iloc[0]["date_col"]
chosen_trade_date_col = eligible_discovery.iloc[0]["trade_date_col"]
chosen_tenor_col = eligible_discovery.iloc[0]["tenor_col"]
chosen_outcome_col = eligible_discovery.iloc[0]["chosen_outcome_col"]
chosen_outcome_kind = eligible_discovery.iloc[0]["chosen_outcome_kind"]

if chosen_outcome_col is None:
    raise RuntimeError("Internal error: chosen_outcome_col is None after eligibility filtering.")

print()
print("Chosen trade outcome file:")
display(pd.DataFrame([{
    "chosen_file": str(chosen_file),
    "date_col": chosen_date_col,
    "trade_date_col": chosen_trade_date_col,
    "tenor_col": chosen_tenor_col,
    "outcome_col": chosen_outcome_col,
    "outcome_kind": chosen_outcome_kind,
}]))


# -----------------------------
# 7. Load and normalize chosen trade outcomes
# -----------------------------

if chosen_file.suffix.lower() == ".parquet":
    outcome_raw = pd.read_parquet(chosen_file)
else:
    outcome_raw = pd.read_csv(chosen_file)

outcome = outcome_raw.copy()

outcome["date"] = robust_parse_date_series(outcome[chosen_date_col])

if chosen_trade_date_col is not None and chosen_trade_date_col in outcome.columns:
    outcome["trade_date"] = robust_parse_trade_date_series(outcome[chosen_trade_date_col])
else:
    outcome["trade_date"] = pd.to_numeric(
        outcome["date"].dt.strftime("%Y%m%d"),
        errors="coerce",
    ).astype("Int64")

outcome["tenor"] = robust_parse_tenor_series(
    outcome[chosen_tenor_col],
    valid_target_tenors=VALID_TARGET_TENORS,
)

outcome["outcome_value"] = pd.to_numeric(
    first_series(outcome[chosen_outcome_col]),
    errors="coerce",
)

outcome = outcome.dropna(
    subset=["date", "trade_date", "tenor", "outcome_value"]
).copy()

outcome["trade_date"] = outcome["trade_date"].astype(int)
outcome["tenor"] = outcome["tenor"].astype(int)

dup_count = outcome.duplicated(subset=["date", "trade_date", "tenor"]).sum()

if dup_count > 0:
    print()
    print(f"Warning: outcome file has {dup_count:,} duplicate date x tenor rows.")
    print("Aggregating outcome_value by mean.")
    outcome_keyed = (
        outcome
        .groupby(["date", "trade_date", "tenor"], as_index=False)
        .agg(outcome_value=("outcome_value", "mean"))
    )
else:
    outcome_keyed = outcome[["date", "trade_date", "tenor", "outcome_value"]].copy()

print()
print("Outcome table summary:")
display(pd.DataFrame([{
    "rows": len(outcome_keyed),
    "first_date": outcome_keyed["date"].min(),
    "last_date": outcome_keyed["date"].max(),
    "unique_dates": outcome_keyed["date"].nunique(),
    "unique_tenors": outcome_keyed["tenor"].nunique(),
    "outcome_col": chosen_outcome_col,
    "outcome_kind": chosen_outcome_kind,
    "duplicate_rows_before_keying": int(dup_count),
}]))


# -----------------------------
# 8. Join selected candidates to outcomes
# -----------------------------

bt = selected_candidates.merge(
    outcome_keyed,
    on=["date", "trade_date", "tenor"],
    how="left",
    validate="many_to_one",
)

bt["has_outcome"] = bt["outcome_value"].notna()

coverage = (
    bt
    .groupby(["score_method", "model_label"])
    .agg(
        selected_trades=("date", "size"),
        matched_outcomes=("has_outcome", "sum"),
        outcome_coverage_pct=("has_outcome", "mean"),
    )
    .reset_index()
)

print()
print("Outcome coverage by strategy:")
display(coverage)

if coverage["outcome_coverage_pct"].min() < 0.95:
    print()
    print("Warning: at least one strategy has less than 95% outcome coverage.")
    print("This usually means the chosen file is not the right all-candidate outcome file.")
    print("Backtest metrics below will still be shown, but should not be trusted until coverage is fixed.")


# -----------------------------
# 9. Backtest metrics
# -----------------------------

metrics_rows = []
equity_rows = []
year_rows = []

for (score_method, model_label), g in bt.groupby(["score_method", "model_label"]):
    gg = g.dropna(subset=["outcome_value"]).copy()
    gg = gg.sort_values(["date", "tenor"]).reset_index(drop=True)

    if gg.empty:
        continue

    gg["trade_number"] = np.arange(1, len(gg) + 1)
    gg["cum_outcome"] = gg["outcome_value"].cumsum()
    gg["drawdown"] = gg["cum_outcome"] - gg["cum_outcome"].cummax()
    gg["rolling_20_trade_sum"] = gg["outcome_value"].rolling(20, min_periods=1).sum()
    gg["year"] = gg["date"].dt.year

    wins = gg.loc[gg["outcome_value"] > 0, "outcome_value"]
    losses = gg.loc[gg["outcome_value"] < 0, "outcome_value"]

    total_gain = wins.sum()
    total_loss = losses.sum()

    metrics_rows.append({
        "score_method": score_method,
        "model_label": model_label,
        "outcome_kind": chosen_outcome_kind,
        "outcome_col": chosen_outcome_col,
        "selected_trades": len(g),
        "matched_trades": len(gg),
        "coverage_pct": len(gg) / len(g),
        "first_trade_date": gg["date"].min(),
        "last_trade_date": gg["date"].max(),
        "total_outcome": gg["outcome_value"].sum(),
        "mean_outcome": gg["outcome_value"].mean(),
        "median_outcome": gg["outcome_value"].median(),
        "win_rate": (gg["outcome_value"] > 0).mean(),
        "avg_win": wins.mean() if len(wins) else np.nan,
        "avg_loss": losses.mean() if len(losses) else np.nan,
        "profit_factor": total_gain / abs(total_loss) if total_loss < 0 else np.nan,
        "max_drawdown": max_drawdown(gg["cum_outcome"]),
        "worst_20_trade_sum": gg["rolling_20_trade_sum"].min(),
        "best_20_trade_sum": gg["rolling_20_trade_sum"].max(),
        "worst_single_trade": gg["outcome_value"].min(),
        "best_single_trade": gg["outcome_value"].max(),
        "mean_selected_score": gg["selected_score"].mean(),
    })

    equity_rows.append(
        gg[
            [
                "score_method",
                "model_label",
                "date",
                "trade_date",
                "tenor",
                "tenor_group",
                "selected_score",
                "outcome_value",
                "trade_number",
                "cum_outcome",
                "drawdown",
                "rolling_20_trade_sum",
            ]
        ].copy()
    )

    yr = (
        gg
        .groupby("year")
        .agg(
            trades=("date", "size"),
            total_outcome=("outcome_value", "sum"),
            mean_outcome=("outcome_value", "mean"),
            win_rate=("outcome_value", lambda x: (x > 0).mean()),
            worst_trade=("outcome_value", "min"),
        )
        .reset_index()
    )
    yr["score_method"] = score_method
    yr["model_label"] = model_label
    year_rows.append(yr)

bt_metrics = pd.DataFrame(metrics_rows).sort_values(
    ["score_method", "total_outcome"],
    ascending=[True, False],
)

bt_equity = pd.concat(equity_rows, ignore_index=True) if equity_rows else pd.DataFrame()
bt_by_year = pd.concat(year_rows, ignore_index=True) if year_rows else pd.DataFrame()

print()
print("Backtest metrics:")
display(bt_metrics)

print()
print("Backtest by year:")
display(bt_by_year.sort_values(["score_method", "model_label", "year"]))


# -----------------------------
# 10. Pairwise comparison to current
# -----------------------------

comparison_rows = []

for score_method in SCORE_METHODS_TO_BACKTEST:
    base = bt_metrics.loc[
        (bt_metrics["score_method"] == score_method)
        & (bt_metrics["model_label"] == "current")
    ]

    if base.empty:
        continue

    base_row = base.iloc[0]

    for model_label in ["har_implied", "har_total"]:
        alt = bt_metrics.loc[
            (bt_metrics["score_method"] == score_method)
            & (bt_metrics["model_label"] == model_label)
        ]

        if alt.empty:
            continue

        alt_row = alt.iloc[0]

        comparison_rows.append({
            "score_method": score_method,
            "comparison": f"{model_label}_minus_current",
            "current_total_outcome": base_row["total_outcome"],
            "alt_total_outcome": alt_row["total_outcome"],
            "delta_total_outcome": alt_row["total_outcome"] - base_row["total_outcome"],
            "current_max_drawdown": base_row["max_drawdown"],
            "alt_max_drawdown": alt_row["max_drawdown"],
            "delta_max_drawdown": alt_row["max_drawdown"] - base_row["max_drawdown"],
            "current_worst_20": base_row["worst_20_trade_sum"],
            "alt_worst_20": alt_row["worst_20_trade_sum"],
            "delta_worst_20": alt_row["worst_20_trade_sum"] - base_row["worst_20_trade_sum"],
            "current_win_rate": base_row["win_rate"],
            "alt_win_rate": alt_row["win_rate"],
            "delta_win_rate": alt_row["win_rate"] - base_row["win_rate"],
        })

bt_pairwise_vs_current = pd.DataFrame(comparison_rows)

print()
print("Pairwise comparison vs current:")
display(bt_pairwise_vs_current)


# -----------------------------
# 11. Save outputs
# -----------------------------

safe_start = selected_candidates["date"].min().strftime("%Y%m%d")
safe_end = selected_candidates["date"].max().strftime("%Y%m%d")

cell11_selected_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell11_selected_candidates_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell11_discovery_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell11_trade_outcome_file_discovery_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell11_joined_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_cell11_joined_backtest_trades_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell11_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell11_backtest_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell11_equity_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell11_backtest_equity_trades_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell11_by_year_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell11_backtest_by_year_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell11_pairwise_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell11_pairwise_vs_current_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

selected_candidates.to_csv(cell11_selected_output, index=False)
discovery.to_csv(cell11_discovery_output, index=False)
bt.to_parquet(cell11_joined_output, index=False)
bt_metrics.to_csv(cell11_metrics_output, index=False)
bt_equity.to_csv(cell11_equity_output, index=False)
bt_by_year.to_csv(cell11_by_year_output, index=False)
bt_pairwise_vs_current.to_csv(cell11_pairwise_output, index=False)

print()
print("Cell 11 outputs saved:")
print(f"Selected candidates:       {cell11_selected_output}")
print(f"Outcome file discovery:    {cell11_discovery_output}")
print(f"Joined backtest trades:    {cell11_joined_output}")
print(f"Backtest metrics:          {cell11_metrics_output}")
print(f"Backtest equity trades:    {cell11_equity_output}")
print(f"Backtest by year:          {cell11_by_year_output}")
print(f"Pairwise vs current:       {cell11_pairwise_output}")

print()
print("Cell 11 complete.")
print("Next step: review whether HAR-implied improves raw_vrp_log and/or max_z_3m_1y selection.")

## 12. Build out-of-sample OOS forecast/VRP panel

Uses Cell 6B walk-forward predictions to create a true out-of-sample VRP signal panel. This avoids selecting a model based on final models fit on all history.

In [ ]:
# Cell 12: Build out-of-sample Corsi forecast / VRP signal panel
#
# Purpose:
#   Use Cell 6B walk-forward predictions to create a true OOS signal panel.
#
# Why:
#   Cell 9 / Cell 11 used final models fit on all complete history.
#   That is fine for live scoring, but not enough for historical model selection.
#
# This cell creates:
#   - current baseline VRP fields
#   - OOS har_implied_shrinkage forecast variance / vol
#   - OOS har_total_simple forecast variance / vol
#   - OOS VRP log fields
#   - rolling 3m / 1y z-scores recomputed within the OOS panel
#   - score columns for:
#       raw_vrp_log
#       z_3m
#       z_1y
#       max_z_3m_1y
#       avg_z_3m_1y
#       min_z_3m_1y
#
# This cell does NOT overwrite production files.

from pathlib import Path
import numpy as np
import pandas as pd


# -----------------------------
# 1. Paths / config
# -----------------------------

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "CORSI_PROCESSED_DIR" not in globals():
    CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "forecast_model_corsi_v1"

if "CORSI_AUDIT_DIR" not in globals():
    CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"

EPSILON_VARIANCE = 1e-12

OOS_MODELS = [
    "har_implied_shrinkage",
    "har_total_simple",
]

ROLLING_Z_WINDOWS = {
    "3m": {
        "window": 63,
        "min_periods": 30,
    },
    "1y": {
        "window": 252,
        "min_periods": 126,
    },
}

print("Cell 12 configuration:")
print(f"OOS models:       {OOS_MODELS}")
print(f"Rolling windows:  {ROLLING_Z_WINDOWS}")
print(f"Processed dir:    {CORSI_PROCESSED_DIR}")
print(f"Audit dir:        {CORSI_AUDIT_DIR}")


# -----------------------------
# 2. Load Cell 6B walk-forward predictions
# -----------------------------

latest_6b_files = sorted(
    CORSI_PROCESSED_DIR.glob("corsi_simple_har_predictions_v1_*.parquet")
)

if not latest_6b_files:
    raise FileNotFoundError(
        "No Cell 6B walk-forward prediction file found: "
        "corsi_simple_har_predictions_v1_*.parquet"
    )

cell6b_path = latest_6b_files[-1]

print()
print(f"Loading Cell 6B OOS predictions: {cell6b_path}")

pred_6b = pd.read_parquet(cell6b_path)

pred_6b = pred_6b.copy()
pred_6b["date"] = pd.to_datetime(pred_6b["date"])
pred_6b["trade_date"] = pd.to_numeric(pred_6b["trade_date"], errors="coerce").astype(int)
pred_6b["tenor"] = pd.to_numeric(pred_6b["tenor"], errors="coerce").astype(int)

pred_6b = pred_6b.replace([np.inf, -np.inf], np.nan)

print()
print("Cell 6B prediction file summary:")
display(pd.DataFrame([{
    "rows": len(pred_6b),
    "first_date": pred_6b["date"].min(),
    "last_date": pred_6b["date"].max(),
    "unique_dates": pred_6b["date"].nunique(),
    "unique_tenors": pred_6b["tenor"].nunique(),
    "models": sorted(pred_6b["model_name"].dropna().unique().tolist()),
}]))


# -----------------------------
# 3. Resolve prediction columns
# -----------------------------

candidate_pred_var_cols = [
    "pred_forward_variance_corsi_6b",
    "pred_forward_variance_corsi",
    "pred_variance",
    "pred_var",
    "forecast_variance",
]

candidate_pred_vol_cols = [
    "pred_forward_vol_corsi_6b",
    "pred_forward_vol_corsi",
    "pred_vol",
    "forecast_vol",
]

pred_var_col = None
for c in candidate_pred_var_cols:
    if c in pred_6b.columns:
        pred_var_col = c
        break

pred_vol_col = None
for c in candidate_pred_vol_cols:
    if c in pred_6b.columns:
        pred_vol_col = c
        break

if pred_var_col is None:
    raise ValueError(
        "Could not find predicted variance column in Cell 6B file. "
        f"Checked: {candidate_pred_var_cols}"
    )

if pred_vol_col is None:
    pred_6b["_pred_vol_from_var"] = np.sqrt(
        pd.to_numeric(pred_6b[pred_var_col], errors="coerce").clip(lower=EPSILON_VARIANCE)
    ) * 100
    pred_vol_col = "_pred_vol_from_var"

print()
print("Resolved prediction columns:")
display(pd.DataFrame([{
    "pred_var_col": pred_var_col,
    "pred_vol_col": pred_vol_col,
}]))


# -----------------------------
# 4. Keep OOS candidate models
# -----------------------------

missing_models = sorted(set(OOS_MODELS) - set(pred_6b["model_name"].dropna().unique()))

if missing_models:
    raise ValueError(f"Missing requested OOS models in Cell 6B predictions: {missing_models}")

oos_long = pred_6b.loc[
    pred_6b["model_name"].isin(OOS_MODELS)
].copy()

oos_long[pred_var_col] = pd.to_numeric(oos_long[pred_var_col], errors="coerce")
oos_long[pred_vol_col] = pd.to_numeric(oos_long[pred_vol_col], errors="coerce")

oos_long = oos_long.loc[
    oos_long[pred_var_col].notna()
    & (oos_long[pred_var_col] > 0)
].copy()

print()
print("OOS model row coverage:")
display(
    oos_long
    .groupby("model_name")
    .agg(
        rows=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        unique_dates=("date", "nunique"),
        unique_tenors=("tenor", "nunique"),
    )
    .reset_index()
)


# -----------------------------
# 5. Build base panel from OOS predictions
# -----------------------------

base_candidate_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_group",
    "implied_variance",
    "vix_style_vol",
    "forecast_variance",
    "forecast_vol",
    "trailing_realized_variance",
    "trailing_realized_vol",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "primary_vrp_signal",
    "forward_window_complete_corsi",
    "log_forward_realized_variance_corsi",
    "forward_realized_variance_corsi",
    "forward_realized_vol_corsi",
]

base_cols = [c for c in base_candidate_cols if c in oos_long.columns]

base_panel = (
    oos_long
    .drop_duplicates(subset=["date", "trade_date", "tenor"])[base_cols]
    .copy()
)

# Ensure required current/implied columns exist.
if "implied_variance" not in base_panel.columns:
    raise ValueError("Cell 6B predictions must contain implied_variance.")

base_panel["implied_variance"] = pd.to_numeric(base_panel["implied_variance"], errors="coerce")

if "vix_style_vol" not in base_panel.columns:
    base_panel["vix_style_vol"] = np.sqrt(
        base_panel["implied_variance"].clip(lower=EPSILON_VARIANCE)
    ) * 100

if "vrp_log" not in base_panel.columns:
    if "forecast_variance" not in base_panel.columns:
        raise ValueError("Cannot build current vrp_log: missing both vrp_log and forecast_variance.")
    base_panel["vrp_log"] = np.log(
        base_panel["implied_variance"].clip(lower=EPSILON_VARIANCE)
        / pd.to_numeric(base_panel["forecast_variance"], errors="coerce").clip(lower=EPSILON_VARIANCE)
    )

base_panel = base_panel.rename(columns={
    "vrp_log": "vrp_log_current",
    "vrp_z_3m": "vrp_z_3m_current_production",
    "vrp_z_1y": "vrp_z_1y_current_production",
    "primary_vrp_signal": "primary_vrp_signal_current_production",
})

# Pivot OOS forecast variance.
forecast_var_wide = (
    oos_long
    .pivot_table(
        index=["date", "trade_date", "tenor"],
        columns="model_name",
        values=pred_var_col,
        aggfunc="first",
    )
    .reset_index()
)

forecast_vol_wide = (
    oos_long
    .pivot_table(
        index=["date", "trade_date", "tenor"],
        columns="model_name",
        values=pred_vol_col,
        aggfunc="first",
    )
    .reset_index()
)

forecast_var_wide = forecast_var_wide.rename(columns={
    "har_implied_shrinkage": "forecast_variance_har_implied_oos",
    "har_total_simple": "forecast_variance_har_total_oos",
})

forecast_vol_wide = forecast_vol_wide.rename(columns={
    "har_implied_shrinkage": "forecast_vol_har_implied_oos",
    "har_total_simple": "forecast_vol_har_total_oos",
})

oos_signal_panel = base_panel.merge(
    forecast_var_wide,
    on=["date", "trade_date", "tenor"],
    how="inner",
    validate="one_to_one",
)

oos_signal_panel = oos_signal_panel.merge(
    forecast_vol_wide,
    on=["date", "trade_date", "tenor"],
    how="inner",
    validate="one_to_one",
)

# Keep rows where both OOS candidate forecasts are present.
required_oos_forecast_cols = [
    "forecast_variance_har_implied_oos",
    "forecast_variance_har_total_oos",
]

oos_signal_panel = oos_signal_panel.loc[
    oos_signal_panel[required_oos_forecast_cols].notna().all(axis=1)
].copy()


# -----------------------------
# 6. Build OOS VRP fields
# -----------------------------

oos_signal_panel["vrp_log_har_implied_oos"] = np.log(
    oos_signal_panel["implied_variance"].clip(lower=EPSILON_VARIANCE)
    / oos_signal_panel["forecast_variance_har_implied_oos"].clip(lower=EPSILON_VARIANCE)
)

oos_signal_panel["vrp_log_har_total_oos"] = np.log(
    oos_signal_panel["implied_variance"].clip(lower=EPSILON_VARIANCE)
    / oos_signal_panel["forecast_variance_har_total_oos"].clip(lower=EPSILON_VARIANCE)
)

oos_signal_panel["vrp_ratio_current"] = np.exp(oos_signal_panel["vrp_log_current"])
oos_signal_panel["vrp_ratio_har_implied_oos"] = np.exp(oos_signal_panel["vrp_log_har_implied_oos"])
oos_signal_panel["vrp_ratio_har_total_oos"] = np.exp(oos_signal_panel["vrp_log_har_total_oos"])

oos_signal_panel["vrp_variance_spread_current"] = (
    oos_signal_panel["implied_variance"]
    - (
        oos_signal_panel["implied_variance"]
        / oos_signal_panel["vrp_ratio_current"].clip(lower=EPSILON_VARIANCE)
    )
)

oos_signal_panel["vrp_variance_spread_har_implied_oos"] = (
    oos_signal_panel["implied_variance"]
    - oos_signal_panel["forecast_variance_har_implied_oos"]
)

oos_signal_panel["vrp_variance_spread_har_total_oos"] = (
    oos_signal_panel["implied_variance"]
    - oos_signal_panel["forecast_variance_har_total_oos"]
)


# -----------------------------
# 7. Recompute rolling z-scores inside the OOS panel
# -----------------------------

# This is intentionally recomputed for current, har_implied, and har_total
# on the same OOS date window so z-score comparisons are apples-to-apples.
vrp_model_cols = {
    "current": "vrp_log_current",
    "har_implied": "vrp_log_har_implied_oos",
    "har_total": "vrp_log_har_total_oos",
}

oos_signal_panel = oos_signal_panel.sort_values(["tenor", "date"]).reset_index(drop=True)

zscore_coverage_rows = []

for model_label, vrp_col in vrp_model_cols.items():
    for z_name, z_cfg in ROLLING_Z_WINDOWS.items():
        window = z_cfg["window"]
        min_periods = z_cfg["min_periods"]

        mean_col = f"{vrp_col}_mean_{z_name}_oos_window"
        std_col = f"{vrp_col}_std_{z_name}_oos_window"
        z_col = f"vrp_z_{z_name}_{model_label}_oos_window"

        oos_signal_panel[mean_col] = (
            oos_signal_panel
            .groupby("tenor")[vrp_col]
            .transform(lambda s: s.rolling(window, min_periods=min_periods).mean())
        )

        oos_signal_panel[std_col] = (
            oos_signal_panel
            .groupby("tenor")[vrp_col]
            .transform(lambda s: s.rolling(window, min_periods=min_periods).std())
        )

        oos_signal_panel[z_col] = (
            (oos_signal_panel[vrp_col] - oos_signal_panel[mean_col])
            / oos_signal_panel[std_col].replace(0, np.nan)
        )

        zscore_coverage_rows.append({
            "model_label": model_label,
            "vrp_col": vrp_col,
            "z_window": z_name,
            "z_col": z_col,
            "non_null_rows": int(oos_signal_panel[z_col].notna().sum()),
            "first_non_null_date": oos_signal_panel.loc[
                oos_signal_panel[z_col].notna(), "date"
            ].min(),
            "latest_non_null_date": oos_signal_panel.loc[
                oos_signal_panel[z_col].notna(), "date"
            ].max(),
        })

zscore_coverage = pd.DataFrame(zscore_coverage_rows)

oos_signal_panel = oos_signal_panel.sort_values(["date", "tenor"]).reset_index(drop=True)


# -----------------------------
# 8. Build score columns
# -----------------------------

score_model_config = {
    "current": {
        "raw": "vrp_log_current",
        "z3": "vrp_z_3m_current_oos_window",
        "z1": "vrp_z_1y_current_oos_window",
    },
    "har_implied": {
        "raw": "vrp_log_har_implied_oos",
        "z3": "vrp_z_3m_har_implied_oos_window",
        "z1": "vrp_z_1y_har_implied_oos_window",
    },
    "har_total": {
        "raw": "vrp_log_har_total_oos",
        "z3": "vrp_z_3m_har_total_oos_window",
        "z1": "vrp_z_1y_har_total_oos_window",
    },
}

for model_label, cfg in score_model_config.items():
    raw_col = cfg["raw"]
    z3_col = cfg["z3"]
    z1_col = cfg["z1"]

    oos_signal_panel[f"score_{model_label}_raw_vrp_log"] = oos_signal_panel[raw_col]
    oos_signal_panel[f"score_{model_label}_z_3m"] = oos_signal_panel[z3_col]
    oos_signal_panel[f"score_{model_label}_z_1y"] = oos_signal_panel[z1_col]

    oos_signal_panel[f"score_{model_label}_max_z_3m_1y"] = np.maximum(
        oos_signal_panel[z3_col],
        oos_signal_panel[z1_col],
    )

    oos_signal_panel[f"score_{model_label}_avg_z_3m_1y"] = 0.5 * (
        oos_signal_panel[z3_col]
        + oos_signal_panel[z1_col]
    )

    oos_signal_panel[f"score_{model_label}_min_z_3m_1y"] = np.minimum(
        oos_signal_panel[z3_col],
        oos_signal_panel[z1_col],
    )


# -----------------------------
# 9. Diagnostics
# -----------------------------

latest_oos_date = oos_signal_panel["date"].max()

latest_oos_cols = [
    "date",
    "trade_date",
    "tenor",
    "tenor_group",
    "vix_style_vol",
    "forecast_vol",
    "forecast_vol_har_implied_oos",
    "forecast_vol_har_total_oos",
    "vrp_log_current",
    "vrp_log_har_implied_oos",
    "vrp_log_har_total_oos",
    "vrp_z_3m_current_oos_window",
    "vrp_z_3m_har_implied_oos_window",
    "vrp_z_3m_har_total_oos_window",
    "vrp_z_1y_current_oos_window",
    "vrp_z_1y_har_implied_oos_window",
    "vrp_z_1y_har_total_oos_window",
    "score_current_raw_vrp_log",
    "score_har_implied_raw_vrp_log",
    "score_har_total_raw_vrp_log",
    "score_current_max_z_3m_1y",
    "score_har_implied_max_z_3m_1y",
    "score_har_total_max_z_3m_1y",
]

latest_oos_cols = [c for c in latest_oos_cols if c in oos_signal_panel.columns]

latest_oos_signal_table = (
    oos_signal_panel
    .loc[oos_signal_panel["date"] == latest_oos_date, latest_oos_cols]
    .sort_values("tenor")
    .copy()
)

oos_panel_summary = pd.DataFrame([{
    "rows": len(oos_signal_panel),
    "first_date": oos_signal_panel["date"].min(),
    "last_date": oos_signal_panel["date"].max(),
    "unique_dates": oos_signal_panel["date"].nunique(),
    "unique_tenors": oos_signal_panel["tenor"].nunique(),
    "latest_date_rows": len(latest_oos_signal_table),
}])

vrp_distribution_rows = []

for model_label, vrp_col in vrp_model_cols.items():
    s = oos_signal_panel[vrp_col].replace([np.inf, -np.inf], np.nan).dropna()

    vrp_distribution_rows.append({
        "model_label": model_label,
        "vrp_col": vrp_col,
        "rows": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "p10": s.quantile(0.10),
        "p25": s.quantile(0.25),
        "p75": s.quantile(0.75),
        "p90": s.quantile(0.90),
        "min": s.min(),
        "max": s.max(),
    })

vrp_distribution = pd.DataFrame(vrp_distribution_rows)

print()
print("OOS signal panel summary:")
display(oos_panel_summary)

print()
print("OOS z-score coverage:")
display(zscore_coverage)

print()
print("OOS VRP distribution:")
display(vrp_distribution)

print()
print(f"Latest OOS signal table: {latest_oos_date.date()}")
display(latest_oos_signal_table)


# -----------------------------
# 10. Save outputs
# -----------------------------

safe_start = oos_signal_panel["date"].min().strftime("%Y%m%d")
safe_end = oos_signal_panel["date"].max().strftime("%Y%m%d")

cell12_oos_panel_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_cell12_oos_forecast_vrp_signal_panel_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell12_oos_sample_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell12_oos_forecast_vrp_signal_panel_sample_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell12_summary_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell12_oos_panel_summary_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell12_zscore_coverage_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell12_oos_zscore_coverage_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell12_vrp_distribution_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell12_oos_vrp_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell12_latest_table_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell12_latest_oos_signal_table_{latest_oos_date.strftime('%Y%m%d')}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

oos_signal_panel.to_parquet(cell12_oos_panel_output, index=False)
oos_signal_panel.head(1000).to_csv(cell12_oos_sample_output, index=False)
oos_panel_summary.to_csv(cell12_summary_output, index=False)
zscore_coverage.to_csv(cell12_zscore_coverage_output, index=False)
vrp_distribution.to_csv(cell12_vrp_distribution_output, index=False)
latest_oos_signal_table.to_csv(cell12_latest_table_output, index=False)

print()
print("Cell 12 outputs saved:")
print(f"OOS signal panel:       {cell12_oos_panel_output}")
print(f"OOS sample:             {cell12_oos_sample_output}")
print(f"OOS panel summary:      {cell12_summary_output}")
print(f"Z-score coverage:       {cell12_zscore_coverage_output}")
print(f"VRP distribution:       {cell12_vrp_distribution_output}")
print(f"Latest OOS table:       {cell12_latest_table_output}")

print()
print("Cell 12 complete.")
print("Next step: rerun Cell 11-style trade selection backtest using this OOS signal panel.")

## 13. Out-of-sample trade-selection backtest

Runs the model-selection backtest using only OOS forecasts. This is the decisive test used to carry forward `har_total_simple` as the Corsi challenger and reject `har_implied_shrinkage` for trading-signal selection.

In [ ]:
# Cell 13: OOS trade-selection backtest using Cell 12 OOS signal panel
#
# Purpose:
#   Use true walk-forward / out-of-sample forecasts from Cell 6B / Cell 12
#   to compare trade selection across:
#
#   Models:
#       current
#       har_implied
#       har_total
#
#   Score definitions:
#       raw_vrp_log
#       max_z_3m_1y
#       avg_z_3m_1y
#
# This is the model-selection backtest.
#
# Important:
#   Unlike Cell 11, this uses only OOS forecast predictions.
#   This is the backtest we should use to choose the denominator/model.

from pathlib import Path
import numpy as np
import pandas as pd


# -----------------------------
# 1. Configuration
# -----------------------------

CALIBRATED_TRADE_COUNT = 577

SCORE_METHODS_TO_BACKTEST = [
    "raw_vrp_log",
    "max_z_3m_1y",
    "avg_z_3m_1y",
]

MODEL_LABELS_TO_BACKTEST = [
    "current",
    "har_implied",
    "har_total",
]

# Optional manual overrides.
# Leave as None unless needed.
TRADE_OUTCOME_FILE_OVERRIDE = None
OUTCOME_VALUE_COL_OVERRIDE = None
OUTCOME_VALUE_KIND_OVERRIDE = None   # None, "pnl", or "return"

FILTER_OOS_TO_OUTCOME_DATE_RANGE = True


# -----------------------------
# 2. Paths
# -----------------------------

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "CORSI_PROCESSED_DIR" not in globals():
    CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "forecast_model_corsi_v1"

if "CORSI_AUDIT_DIR" not in globals():
    CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "forecast_model_corsi_v1"

if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

if "SPY_STOCK_VENUE" not in globals():
    SPY_STOCK_VENUE = "utp_cta"


# -----------------------------
# 3. Helper functions
# -----------------------------

def first_series(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.iloc[:, 0]
    return obj


def robust_parse_date_series(s):
    s = first_series(s)

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s, errors="coerce")

    s_str = s.astype(str).str.strip()

    ymd = s_str.str.extract(r"(\d{8})", expand=False)
    ymd_dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")

    direct_dt = pd.to_datetime(s_str.where(ymd_dt.isna()), errors="coerce")

    out = ymd_dt.copy()
    out.loc[out.isna()] = direct_dt.loc[out.isna()]
    return out


def robust_parse_trade_date_series(s):
    dt = robust_parse_date_series(s)
    out = pd.to_numeric(dt.dt.strftime("%Y%m%d"), errors="coerce")
    return out.astype("Int64")


def robust_parse_tenor_series(s, valid_target_tenors=None):
    s = first_series(s)
    tenor_num = pd.to_numeric(s, errors="coerce")

    out = pd.Series(pd.NA, index=s.index, dtype="Int64")

    valid = tenor_num.notna() & np.isfinite(tenor_num)
    integer_like = (tenor_num - tenor_num.round()).abs() < 1e-8
    valid = valid & integer_like.fillna(False)

    out.loc[valid] = tenor_num.loc[valid].round().astype("Int64")

    if valid_target_tenors is not None:
        valid_target_tenors = set(int(x) for x in valid_target_tenors)
        out.loc[~out.isin(valid_target_tenors)] = pd.NA

    return out


def max_drawdown_details(equity_df, value_col="cum_outcome"):
    """
    Returns max drawdown and peak/trough dates.
    """
    if equity_df.empty:
        return {
            "max_drawdown": np.nan,
            "max_drawdown_start_date": pd.NaT,
            "max_drawdown_trough_date": pd.NaT,
        }

    cum = equity_df[value_col]
    running_max = cum.cummax()
    dd = cum - running_max

    trough_idx = dd.idxmin()
    max_dd = dd.loc[trough_idx]

    peak_idx = cum.loc[:trough_idx].idxmax()

    return {
        "max_drawdown": max_dd,
        "max_drawdown_start_date": equity_df.loc[peak_idx, "date"],
        "max_drawdown_trough_date": equity_df.loc[trough_idx, "date"],
    }


def worst_rolling_window_details(equity_df, rolling_col="rolling_20_trade_sum", window=20):
    if equity_df.empty or rolling_col not in equity_df.columns:
        return {
            "worst_20_trade_sum": np.nan,
            "worst_20_start_date": pd.NaT,
            "worst_20_end_date": pd.NaT,
        }

    idx = equity_df[rolling_col].idxmin()
    end_pos = equity_df.index.get_loc(idx)
    start_pos = max(0, end_pos - window + 1)

    start_idx = equity_df.index[start_pos]
    end_idx = idx

    return {
        "worst_20_trade_sum": equity_df.loc[idx, rolling_col],
        "worst_20_start_date": equity_df.loc[start_idx, "date"],
        "worst_20_end_date": equity_df.loc[end_idx, "date"],
    }


# -----------------------------
# 4. Load OOS signal panel from Cell 12
# -----------------------------

try:
    oos_signal_panel
except NameError:
    latest_oos_files = sorted(
        CORSI_PROCESSED_DIR.glob("corsi_cell12_oos_forecast_vrp_signal_panel_*.parquet")
    )

    if not latest_oos_files:
        raise FileNotFoundError("No Cell 12 OOS signal panel found.")

    print(f"Loading Cell 12 OOS signal panel: {latest_oos_files[-1]}")
    oos_signal_panel = pd.read_parquet(latest_oos_files[-1])

oos_panel = oos_signal_panel.copy()
oos_panel["date"] = pd.to_datetime(oos_panel["date"])
oos_panel["trade_date"] = pd.to_numeric(oos_panel["trade_date"], errors="coerce").astype(int)
oos_panel["tenor"] = pd.to_numeric(oos_panel["tenor"], errors="coerce").astype(int)
oos_panel = oos_panel.replace([np.inf, -np.inf], np.nan)

VALID_TARGET_TENORS = sorted(oos_panel["tenor"].dropna().astype(int).unique())

print("OOS signal panel summary:")
display(pd.DataFrame([{
    "rows": len(oos_panel),
    "first_date": oos_panel["date"].min(),
    "last_date": oos_panel["date"].max(),
    "unique_dates": oos_panel["date"].nunique(),
    "unique_tenors": oos_panel["tenor"].nunique(),
    "valid_target_tenors": VALID_TARGET_TENORS,
}]))


# -----------------------------
# 5. Load chosen outcome file from Cell 11 discovery
# -----------------------------

if TRADE_OUTCOME_FILE_OVERRIDE is not None:
    chosen_file = Path(TRADE_OUTCOME_FILE_OVERRIDE)
    chosen_outcome_col = OUTCOME_VALUE_COL_OVERRIDE
    chosen_outcome_kind = OUTCOME_VALUE_KIND_OVERRIDE or "pnl"

    if chosen_outcome_col is None:
        raise ValueError("If TRADE_OUTCOME_FILE_OVERRIDE is set, also set OUTCOME_VALUE_COL_OVERRIDE.")

    # Defaults based on known project schema.
    chosen_date_col = "trade_date"
    chosen_trade_date_col = "trade_date"
    chosen_tenor_col = "tenor"

else:
    latest_discovery_files = sorted(
        CORSI_AUDIT_DIR.glob("corsi_cell11_trade_outcome_file_discovery_*.csv")
    )

    if not latest_discovery_files:
        raise FileNotFoundError(
            "No Cell 11 outcome discovery file found. "
            "Run Cell 11 first or set TRADE_OUTCOME_FILE_OVERRIDE."
        )

    discovery_path = latest_discovery_files[-1]
    print()
    print(f"Loading Cell 11 outcome discovery: {discovery_path}")

    discovery = pd.read_csv(discovery_path)

    # Normalize bool/string columns.
    if "eligible_for_backtest" in discovery.columns:
        eligible_mask = discovery["eligible_for_backtest"].astype(str).str.lower().isin(["true", "1"])
    else:
        eligible_mask = pd.Series(True, index=discovery.index)

    eligible = discovery.loc[
        eligible_mask
        & discovery["chosen_outcome_col"].notna()
        & discovery["chosen_outcome_col"].astype(str).ne("None")
        & discovery["path"].notna()
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            "Cell 11 discovery file exists, but no eligible outcome file was found. "
            "Set TRADE_OUTCOME_FILE_OVERRIDE manually."
        )

    eligible["score"] = pd.to_numeric(eligible["score"], errors="coerce")
    eligible = eligible.sort_values("score", ascending=False)

    chosen = eligible.iloc[0]

    chosen_file = Path(chosen["path"])
    chosen_date_col = chosen["date_col"]
    chosen_trade_date_col = chosen["trade_date_col"] if pd.notna(chosen["trade_date_col"]) else None
    chosen_tenor_col = chosen["tenor_col"]
    chosen_outcome_col = chosen["chosen_outcome_col"]
    chosen_outcome_kind = chosen["chosen_outcome_kind"]

print()
print("Chosen outcome file for OOS backtest:")
display(pd.DataFrame([{
    "chosen_file": str(chosen_file),
    "date_col": chosen_date_col,
    "trade_date_col": chosen_trade_date_col,
    "tenor_col": chosen_tenor_col,
    "outcome_col": chosen_outcome_col,
    "outcome_kind": chosen_outcome_kind,
}]))


# -----------------------------
# 6. Load and normalize trade outcomes
# -----------------------------

if chosen_file.suffix.lower() == ".parquet":
    outcome_raw = pd.read_parquet(chosen_file)
else:
    outcome_raw = pd.read_csv(chosen_file, low_memory=False)

outcome = outcome_raw.copy()

outcome["date"] = robust_parse_date_series(outcome[chosen_date_col])

if chosen_trade_date_col is not None and chosen_trade_date_col in outcome.columns:
    outcome["trade_date"] = robust_parse_trade_date_series(outcome[chosen_trade_date_col])
else:
    outcome["trade_date"] = pd.to_numeric(
        outcome["date"].dt.strftime("%Y%m%d"),
        errors="coerce",
    ).astype("Int64")

outcome["tenor"] = robust_parse_tenor_series(
    outcome[chosen_tenor_col],
    valid_target_tenors=VALID_TARGET_TENORS,
)

outcome["outcome_value"] = pd.to_numeric(
    first_series(outcome[chosen_outcome_col]),
    errors="coerce",
)

outcome = outcome.dropna(
    subset=["date", "trade_date", "tenor", "outcome_value"]
).copy()

outcome["trade_date"] = outcome["trade_date"].astype(int)
outcome["tenor"] = outcome["tenor"].astype(int)

dup_count = outcome.duplicated(subset=["date", "trade_date", "tenor"]).sum()

if dup_count > 0:
    print()
    print(f"Warning: outcome file has {dup_count:,} duplicate date x tenor rows.")
    print("Aggregating outcome_value by mean.")
    outcome_keyed = (
        outcome
        .groupby(["date", "trade_date", "tenor"], as_index=False)
        .agg(outcome_value=("outcome_value", "mean"))
    )
else:
    outcome_keyed = outcome[["date", "trade_date", "tenor", "outcome_value"]].copy()

print()
print("Outcome table summary:")
display(pd.DataFrame([{
    "rows": len(outcome_keyed),
    "first_date": outcome_keyed["date"].min(),
    "last_date": outcome_keyed["date"].max(),
    "unique_dates": outcome_keyed["date"].nunique(),
    "unique_tenors": outcome_keyed["tenor"].nunique(),
    "outcome_col": chosen_outcome_col,
    "outcome_kind": chosen_outcome_kind,
    "duplicate_rows_before_keying": int(dup_count),
}]))


# -----------------------------
# 7. Align OOS signal panel to available outcomes
# -----------------------------

oos_bt_panel = oos_panel.copy()

if FILTER_OOS_TO_OUTCOME_DATE_RANGE:
    outcome_min_date = outcome_keyed["date"].min()
    outcome_max_date = outcome_keyed["date"].max()

    before_rows = len(oos_bt_panel)

    oos_bt_panel = oos_bt_panel.loc[
        (oos_bt_panel["date"] >= outcome_min_date)
        & (oos_bt_panel["date"] <= outcome_max_date)
    ].copy()

    print()
    print("Filtered OOS panel to outcome date range:")
    display(pd.DataFrame([{
        "rows_before": before_rows,
        "rows_after": len(oos_bt_panel),
        "outcome_min_date": outcome_min_date,
        "outcome_max_date": outcome_max_date,
        "oos_min_date_after": oos_bt_panel["date"].min(),
        "oos_max_date_after": oos_bt_panel["date"].max(),
    }]))


# -----------------------------
# 8. Build selected candidates from OOS panel
# -----------------------------

missing_score_cols = []

for score_method in SCORE_METHODS_TO_BACKTEST:
    for model_label in MODEL_LABELS_TO_BACKTEST:
        score_col = f"score_{model_label}_{score_method}"
        if score_col not in oos_bt_panel.columns:
            missing_score_cols.append(score_col)

if missing_score_cols:
    raise ValueError(f"Missing score columns from OOS panel: {missing_score_cols}")

selected_rows = []

for score_method in SCORE_METHODS_TO_BACKTEST:
    for model_label in MODEL_LABELS_TO_BACKTEST:
        score_col = f"score_{model_label}_{score_method}"

        work = oos_bt_panel.dropna(subset=[score_col]).copy()

        daily_top = (
            work
            .sort_values(["date", score_col, "tenor"], ascending=[True, False, True])
            .groupby("date", as_index=False)
            .head(1)
            .copy()
        )

        topn = (
            daily_top
            .sort_values([score_col, "date"], ascending=[False, True])
            .head(CALIBRATED_TRADE_COUNT)
            .copy()
        )

        keep_cols = [
            "date",
            "trade_date",
            "tenor",
            "tenor_group",
            score_col,
            "vix_style_vol",
            "forecast_vol",
            "forecast_vol_har_implied_oos",
            "forecast_vol_har_total_oos",
            "vrp_log_current",
            "vrp_log_har_implied_oos",
            "vrp_log_har_total_oos",
            "vrp_z_3m_current_oos_window",
            "vrp_z_1y_current_oos_window",
            "vrp_z_3m_har_implied_oos_window",
            "vrp_z_1y_har_implied_oos_window",
            "vrp_z_3m_har_total_oos_window",
            "vrp_z_1y_har_total_oos_window",
        ]

        keep_cols = [c for c in keep_cols if c in topn.columns]

        topn = topn[keep_cols].copy()
        topn = topn.rename(columns={score_col: "selected_score"})
        topn["score_method"] = score_method
        topn["model_label"] = model_label
        topn["selection_rank"] = np.arange(1, len(topn) + 1)

        selected_rows.append(topn)

selected_candidates_oos = pd.concat(selected_rows, ignore_index=True)

print()
print("OOS selected candidate summary before outcome join:")
display(
    selected_candidates_oos
    .groupby(["score_method", "model_label"])
    .agg(
        trades=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        mean_score=("selected_score", "mean"),
        min_score=("selected_score", "min"),
        max_score=("selected_score", "max"),
    )
    .reset_index()
)


# -----------------------------
# 9. Join selected candidates to outcomes
# -----------------------------

bt_oos = selected_candidates_oos.merge(
    outcome_keyed,
    on=["date", "trade_date", "tenor"],
    how="left",
    validate="many_to_one",
)

bt_oos["has_outcome"] = bt_oos["outcome_value"].notna()

coverage_oos = (
    bt_oos
    .groupby(["score_method", "model_label"])
    .agg(
        selected_trades=("date", "size"),
        matched_outcomes=("has_outcome", "sum"),
        outcome_coverage_pct=("has_outcome", "mean"),
    )
    .reset_index()
)

print()
print("OOS outcome coverage by strategy:")
display(coverage_oos)

if coverage_oos["outcome_coverage_pct"].min() < 0.95:
    print()
    print("Warning: at least one strategy has less than 95% outcome coverage.")
    print("Backtest metrics below should not be trusted until coverage is fixed.")


# -----------------------------
# 10. Backtest metrics
# -----------------------------

metrics_rows = []
equity_rows = []
year_rows = []

for (score_method, model_label), g in bt_oos.groupby(["score_method", "model_label"]):
    gg = g.dropna(subset=["outcome_value"]).copy()
    gg = gg.sort_values(["date", "tenor"]).reset_index(drop=True)

    if gg.empty:
        continue

    gg["trade_number"] = np.arange(1, len(gg) + 1)
    gg["cum_outcome"] = gg["outcome_value"].cumsum()
    gg["drawdown"] = gg["cum_outcome"] - gg["cum_outcome"].cummax()
    gg["rolling_20_trade_sum"] = gg["outcome_value"].rolling(20, min_periods=1).sum()
    gg["year"] = gg["date"].dt.year

    wins = gg.loc[gg["outcome_value"] > 0, "outcome_value"]
    losses = gg.loc[gg["outcome_value"] < 0, "outcome_value"]

    total_gain = wins.sum()
    total_loss = losses.sum()

    dd = max_drawdown_details(gg)
    worst20 = worst_rolling_window_details(gg)

    metrics_rows.append({
        "score_method": score_method,
        "model_label": model_label,
        "outcome_kind": chosen_outcome_kind,
        "outcome_col": chosen_outcome_col,
        "selected_trades": len(g),
        "matched_trades": len(gg),
        "coverage_pct": len(gg) / len(g),
        "first_trade_date": gg["date"].min(),
        "last_trade_date": gg["date"].max(),
        "total_outcome": gg["outcome_value"].sum(),
        "mean_outcome": gg["outcome_value"].mean(),
        "median_outcome": gg["outcome_value"].median(),
        "win_rate": (gg["outcome_value"] > 0).mean(),
        "avg_win": wins.mean() if len(wins) else np.nan,
        "avg_loss": losses.mean() if len(losses) else np.nan,
        "profit_factor": total_gain / abs(total_loss) if total_loss < 0 else np.nan,
        "max_drawdown": dd["max_drawdown"],
        "max_drawdown_start_date": dd["max_drawdown_start_date"],
        "max_drawdown_trough_date": dd["max_drawdown_trough_date"],
        "worst_20_trade_sum": worst20["worst_20_trade_sum"],
        "worst_20_start_date": worst20["worst_20_start_date"],
        "worst_20_end_date": worst20["worst_20_end_date"],
        "best_20_trade_sum": gg["rolling_20_trade_sum"].max(),
        "worst_single_trade": gg["outcome_value"].min(),
        "best_single_trade": gg["outcome_value"].max(),
        "mean_selected_score": gg["selected_score"].mean(),
    })

    equity_rows.append(
        gg[
            [
                "score_method",
                "model_label",
                "date",
                "trade_date",
                "tenor",
                "tenor_group",
                "selected_score",
                "outcome_value",
                "trade_number",
                "cum_outcome",
                "drawdown",
                "rolling_20_trade_sum",
            ]
        ].copy()
    )

    yr = (
        gg
        .groupby("year")
        .agg(
            trades=("date", "size"),
            total_outcome=("outcome_value", "sum"),
            mean_outcome=("outcome_value", "mean"),
            win_rate=("outcome_value", lambda x: (x > 0).mean()),
            worst_trade=("outcome_value", "min"),
        )
        .reset_index()
    )
    yr["score_method"] = score_method
    yr["model_label"] = model_label
    year_rows.append(yr)

bt_oos_metrics = pd.DataFrame(metrics_rows).sort_values(
    ["score_method", "total_outcome"],
    ascending=[True, False],
)

bt_oos_equity = pd.concat(equity_rows, ignore_index=True) if equity_rows else pd.DataFrame()
bt_oos_by_year = pd.concat(year_rows, ignore_index=True) if year_rows else pd.DataFrame()

print()
print("OOS backtest metrics:")
display(bt_oos_metrics)

print()
print("OOS backtest by year:")
display(bt_oos_by_year.sort_values(["score_method", "model_label", "year"]))


# -----------------------------
# 11. Pairwise comparison to current
# -----------------------------

comparison_rows = []

for score_method in SCORE_METHODS_TO_BACKTEST:
    base = bt_oos_metrics.loc[
        (bt_oos_metrics["score_method"] == score_method)
        & (bt_oos_metrics["model_label"] == "current")
    ]

    if base.empty:
        continue

    base_row = base.iloc[0]

    for model_label in ["har_implied", "har_total"]:
        alt = bt_oos_metrics.loc[
            (bt_oos_metrics["score_method"] == score_method)
            & (bt_oos_metrics["model_label"] == model_label)
        ]

        if alt.empty:
            continue

        alt_row = alt.iloc[0]

        comparison_rows.append({
            "score_method": score_method,
            "comparison": f"{model_label}_minus_current",
            "current_total_outcome": base_row["total_outcome"],
            "alt_total_outcome": alt_row["total_outcome"],
            "delta_total_outcome": alt_row["total_outcome"] - base_row["total_outcome"],
            "current_max_drawdown": base_row["max_drawdown"],
            "alt_max_drawdown": alt_row["max_drawdown"],
            "delta_max_drawdown": alt_row["max_drawdown"] - base_row["max_drawdown"],
            "current_worst_20": base_row["worst_20_trade_sum"],
            "alt_worst_20": alt_row["worst_20_trade_sum"],
            "delta_worst_20": alt_row["worst_20_trade_sum"] - base_row["worst_20_trade_sum"],
            "current_win_rate": base_row["win_rate"],
            "alt_win_rate": alt_row["win_rate"],
            "delta_win_rate": alt_row["win_rate"] - base_row["win_rate"],
        })

bt_oos_pairwise_vs_current = pd.DataFrame(comparison_rows)

print()
print("OOS pairwise comparison vs current:")
display(bt_oos_pairwise_vs_current)


# -----------------------------
# 12. Top tenor / bucket distribution
# -----------------------------

oos_selection_distribution = (
    selected_candidates_oos
    .groupby(["score_method", "model_label", "tenor_group"])
    .agg(count=("date", "size"))
    .reset_index()
)

oos_selection_distribution["pct"] = (
    oos_selection_distribution["count"]
    / oos_selection_distribution.groupby(["score_method", "model_label"])["count"].transform("sum")
)

oos_tenor_distribution = (
    selected_candidates_oos
    .groupby(["score_method", "model_label", "tenor"])
    .agg(count=("date", "size"))
    .reset_index()
)

oos_tenor_distribution["pct"] = (
    oos_tenor_distribution["count"]
    / oos_tenor_distribution.groupby(["score_method", "model_label"])["count"].transform("sum")
)

print()
print("OOS selected tenor-group distribution:")
display(oos_selection_distribution)

print()
print("OOS selected tenor distribution:")
display(oos_tenor_distribution)


# -----------------------------
# 13. Save outputs
# -----------------------------

safe_start = selected_candidates_oos["date"].min().strftime("%Y%m%d")
safe_end = selected_candidates_oos["date"].max().strftime("%Y%m%d")

cell13_selected_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell13_oos_selected_candidates_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell13_joined_output = (
    CORSI_PROCESSED_DIR
    / f"corsi_cell13_oos_joined_backtest_trades_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.parquet"
)

cell13_metrics_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell13_oos_backtest_metrics_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell13_equity_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell13_oos_backtest_equity_trades_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell13_by_year_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell13_oos_backtest_by_year_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell13_pairwise_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell13_oos_pairwise_vs_current_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell13_group_dist_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell13_oos_tenor_group_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

cell13_tenor_dist_output = (
    CORSI_AUDIT_DIR
    / f"corsi_cell13_oos_tenor_distribution_{safe_start}_{safe_end}_{SPY_STOCK_VENUE}_{RUN_TIMESTAMP}.csv"
)

selected_candidates_oos.to_csv(cell13_selected_output, index=False)
bt_oos.to_parquet(cell13_joined_output, index=False)
bt_oos_metrics.to_csv(cell13_metrics_output, index=False)
bt_oos_equity.to_csv(cell13_equity_output, index=False)
bt_oos_by_year.to_csv(cell13_by_year_output, index=False)
bt_oos_pairwise_vs_current.to_csv(cell13_pairwise_output, index=False)
oos_selection_distribution.to_csv(cell13_group_dist_output, index=False)
oos_tenor_distribution.to_csv(cell13_tenor_dist_output, index=False)

print()
print("Cell 13 outputs saved:")
print(f"OOS selected candidates:       {cell13_selected_output}")
print(f"OOS joined backtest trades:    {cell13_joined_output}")
print(f"OOS backtest metrics:          {cell13_metrics_output}")
print(f"OOS backtest equity trades:    {cell13_equity_output}")
print(f"OOS backtest by year:          {cell13_by_year_output}")
print(f"OOS pairwise vs current:       {cell13_pairwise_output}")
print(f"OOS tenor group distribution:  {cell13_group_dist_output}")
print(f"OOS tenor distribution:        {cell13_tenor_dist_output}")

print()
print("Cell 13 complete.")
print("Next step: choose denominator/model candidate, then start a new notebook for signal-definition and parameter sweeps.")

# Research conclusion and next step

The out-of-sample trading-signal test supports the following decision:

- Keep the **current / legacy denominator** as the production baseline.
- Carry forward **`har_total_simple`** as the Corsi/HAR challenger denominator.
- Reject **`har_implied_shrinkage`** as a trading-signal denominator, despite better forecast-error statistics.

The next notebook should start fresh as a signal-definition and parameter-sweep notebook. It should compare the current baseline and `har_total_simple` across raw VRP, 3-month z-score, 1-year z-score, z-score combinations, RSI filters, thresholds, tenor buckets, Core/Secondary rules, and other trade-selection parameters.
